# Canonicalization wrapper on OpenVLA-OFT: camera and robot initial state
**GPU: A100.** Frozen OFT, no retraining. Two channels, both using only what a deployment knows:
the **state** channel drives the arm back to its nominal start pose; the **camera** channel calibrates the real camera
pose from the robot's own silhouette after a short arm sweep, then reprojects the RGB-D image into the nominal view.

1. Setup (~20 min, same install as plus_1).
2. **Validate** (~5 min): calibration error vs ground truth and reprojection quality on 3 camera tasks, plus
   side-by-side PNGs on Drive. Paste this output before running the rest.
3. **Calibration check** (~30 min, no policy): old vs new camera calibration. Paste the table.
4. **Run** (~4 h, resumable): robot_init tasks (state, full2), camera tasks (true-pose and oracle upper bounds,
   full2), light tasks (full2, no-harm control). The nominal camera is read per scene (LIBERO-10 spans several
   rooms). Rows made with the old single nominal are backed up and re-queued automatically. Paste the final table.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['MPLBACKEND'] = 'Agg'   # Colab's inline matplotlib backend crashes every venv python
os.environ['PYTHONUNBUFFERED'] = '1'   # run cells are piped into a log on Drive; without this their output arrives in bursts
os.makedirs('/content/drive/MyDrive/restoring_organs/plus/logs', exist_ok=True)
os.makedirs('/content/logs', exist_ok=True)
print('outputs ->', '/content/drive/MyDrive/restoring_organs/plus')
# Drive uploads a file only after it is CLOSED, so a log held open by a running cell never arrives. Run cells log to
# /content/logs; this thread rewrites the last 64 KB of each log on Drive once a minute as a fresh, closed file.
import builtins, glob, threading, time
def _mirror_logs():
    while True:
        for p in glob.glob('/content/logs/*.log'):
            try:
                with open(p, 'rb') as f:
                    f.seek(max(0, os.path.getsize(p) - 65536))
                    tail = f.read()
                with open('/content/drive/MyDrive/restoring_organs/plus/logs/' + os.path.basename(p)[:-4] + '.' + os.uname().nodename[-6:] + '.log', 'wb') as f:
                    f.write(tail)
            except Exception:
                pass
        time.sleep(60)
if not getattr(builtins, '_log_mirror_on', False):
    builtins._log_mirror_on = True
    threading.Thread(target=_mirror_logs, daemon=True).start()
# Hugging Face login from the Colab secret HF_TOKEN (key icon, left sidebar), if you added one; optional here
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('Hugging Face login ok')
except Exception as e:
    print('no HF_TOKEN secret, continuing without Hugging Face login:', type(e).__name__)

In [ ]:
import os; os.makedirs('/content/scripts', exist_ok=True); print('ok')

In [ ]:
%%writefile /content/scripts/plus_common.py
"""Shared by every LIBERO-Plus notebook: one deterministic task subset (so all five policy families see
exactly the same perturbed tasks), perturbation categories parsed from task names, and the summary table."""
import json
import math
import os
import random
import re
from collections import Counter, defaultdict

# LIBERO-Plus task names, checked on the real suite: token perturbations carry their own tag; camera and robot
# initial state are encoded as values, e.g. ..._view_0_0_100_0_0_initstate_0 is the unperturbed setting.
CATS = [("sensor_noise", r"_noise_\d+"), ("language", r"_language_\d+"), ("light", r"_light_\d+"),
        ("background", r"_table_\d+|_tb_\d+"), ("layout", r"_add_\d+|_level")]
DEFAULT_VIEW = ("0", "0", "100", "0", "0")
# Every run script reads the suite from the environment, so one variable switches a whole notebook:
#   LIBERO_SUITE=libero_spatial | libero_object | libero_goal | libero_10 (default; nothing changes when it is unset)
SUITE = os.environ.get("LIBERO_SUITE", "libero_10")
MAX_STEPS = {"libero_spatial": 220, "libero_object": 280, "libero_goal": 300, "libero_10": 520}     # OFT's own horizons
OFT_CKPT = {s: "moojink/openvla-7b-oft-finetuned-" + s.replace("_", "-") for s in MAX_STEPS}


def categorize(name):
    for cat, pat in CATS:
        if re.search(pat, name):
            return cat
    m = re.search(r"_view_(\d+)_(\d+)_(\d+)_(\d+)_(\d+)", name)
    if m and m.groups() != DEFAULT_VIEW:
        return "camera"
    s = re.search(r"_initstate_(\d+)", name)
    if s and s.group(1) != "0":
        return "robot_init"
    return "other"


def build_subset(suite_name="libero_10", per_cat=30, seed=0):
    from libero.libero import benchmark
    suite = benchmark.get_benchmark_dict()[suite_name]()
    names = [suite.get_task(i).name for i in range(suite.n_tasks)]
    langs = [suite.get_task(i).language for i in range(suite.n_tasks)]
    by = defaultdict(list)
    for i, n in enumerate(names):
        by[categorize(n)].append(i)
    rng = random.Random(seed)
    pick = []
    for cat in sorted(by):
        ids = sorted(by[cat])
        rng.shuffle(ids)
        pick += ids[:per_cat]
    pick.sort()
    tokens = Counter(tok for n in names for tok in re.findall(r"_([a-z]+)_\d+", n))
    return {"suite": suite_name, "n_tasks": suite.n_tasks, "per_cat": per_cat, "seed": seed,
            "counts": {c: len(v) for c, v in sorted(by.items())},
            "other_examples": [names[i] for i in by.get("other", [])[:5]],
            "name_tokens": dict(tokens.most_common(20)),
            "tasks": [{"task_id": i, "name": names[i], "language": langs[i], "category": categorize(names[i])}
                      for i in pick]}


def load_or_build(path, per_cat=30):
    if os.path.exists(path):
        return json.load(open(path))
    sub = build_subset(suite_name=SUITE, per_cat=per_cat)
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    json.dump(sub, open(path, "w"), indent=1)
    return sub


def load_rows(path):
    rows = []
    if os.path.exists(path):
        for line in open(path):
            line = line.strip()
            if line:
                try:
                    rows.append(json.loads(line))
                except Exception:
                    pass
    return rows


def append_row(path, row):
    """Append one result row and CLOSE the file. Drive uploads a file only when it is closed: a runner that keeps its
    results file open for a whole cell has nothing on Drive until the cell ends, and a killed runtime loses all of it."""
    with open(path, "a") as f:
        f.write(json.dumps(row, default=float) + "\n")


def load_shards(path):
    """Rows of `<stem>.jsonl` plus every per-session shard `<stem>.<shard>.jsonl` next to it, one row per
    (task, condition, arm, rep). Two Colab sessions that append to ONE Drive file overwrite each other (each uploads its
    own whole copy, the last upload wins; file 8 lost 576 rows that way on 2026-09-20), so every session writes its own
    shard and readers merge. A measured row beats one recovered from a log (`recovered`), which lacks the detail fields."""
    import glob
    best = {}
    # 0 = measured, 1 = measured before the shading gate (re-queued by the runner, its re-run lives in a later shard),
    # 2 = rebuilt from a log. The lowest rank wins; on a tie the first file does.
    rank = lambda r: 2 if r.get("recovered") else int(bool(r.get("photo_applied")) and "photo_shading" not in r)
    for p in [path] + sorted(glob.glob(path[:-len(".jsonl")] + ".*.jsonl")):
        for r in load_rows(p):
            k = (r.get("policy"), r["task_id"], r.get("cond"), r["arm"], r.get("rep", 0))
            if k not in best or rank(r) < rank(best[k]):
                best[k] = r
    return list(best.values())


def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0.0, c - h), min(1.0, c + h)


def summarize(rows, label=""):
    by = defaultdict(list)
    for r in rows:
        by[r["category"]].append(bool(r["success"]))
    out = {}
    print(f"\n{label}  ({len(rows)} episodes)")
    print(f"{'category':<14}{'n':>5}{'success':>9}{'95% CI':>16}")
    for cat in sorted(by) + ["ALL"]:
        v = [s for r in by.values() for s in r] if cat == "ALL" else by[cat]
        k, n = sum(v), len(v)
        lo, hi = wilson(k, n)
        out[cat] = {"n": n, "success": k / max(1, n), "ci": [lo, hi]}
        print(f"{cat:<14}{n:>5}{k / max(1, n):>9.3f}   [{lo:.2f}, {hi:.2f}]")
    return out

In [ ]:
%%writefile /content/scripts/wrapper_libero.py
"""Deployment-time canonicalization for a FROZEN policy in LIBERO / LIBERO-Plus.

Two channels, both using only what a real deployment knows (the robot's nominal start configuration, the
training camera's nominal pose and intrinsics, the robot's own geometry, and the RGB-D image):

  state channel   measure how far the arm's joints are from the nominal start; if off, drive the end effector
                  back to the nominal start pose with ordinary controller actions (no teleporting).
  camera channel  a short arm sweep, then render-and-compare on the robot's own silhouette to estimate the
                  actual camera pose; every step, reproject the RGB-D image into the nominal camera and give
                  the frozen policy that canonical view.

Sim-only stand-ins, disclosed: the observed robot mask comes from the renderer's segmentation (a perfect arm
segmenter), and depth from the renderer (an RGB-D camera). Upper-bound arms use ground truth explicitly.
"""
import os
import re

import cv2
import numpy as np

ROBOT_PREFIXES = ("robot0", "gripper0", "mount0")
TABLE_Z = 0.8                      # workspace plane used only to pick the orbit pivot on the nominal optical axis
SWEEP = [np.zeros(3), np.array([0.0, 0.12, -0.05]), np.array([0.08, -0.12, 0.0])]
# learned sensing has no metric depth, so the silhouettes alone must fix the camera's distance: more of the robot's own
# KNOWN motion is the metric baseline. Measured against the true camera on six hard tasks (2026-09-19, local): median
# calibration error 21.1 cm with 3 poses, 7.9 cm with 7, 7.1 cm with 7 and like-with-like mask scoring.
SWEEP_LEARNED = SWEEP + [np.array([-0.06, 0.0, 0.10]), np.array([0.10, 0.10, 0.08]), np.array([0.0, -0.15, -0.06]),
                         np.array([-0.05, 0.15, 0.06])]


def scene_of(name):
    """Scene prefix of a LIBERO task name, e.g. KITCHEN_SCENE3. Each scene has its own agentview camera."""
    m = re.match(r"([A-Z]+(?:_[A-Z]+)*_SCENE\d+)_", name)
    if not m:                   # libero_spatial / libero_object / libero_goal: one scene per suite, no prefix in the name;
        return "SUITE_SCENE"    # read_nominals still REQUIRES the unperturbed tasks of a scene to agree on the camera
    return m.group(1)


def seed_flip(suite, make_env, nominals, cam="agentview"):
    """Detect the raw-render -> observation flip ONCE on a clean, unperturbed frame (a default-view language task) and
    cache it. Sensor-noise tasks corrupt the image so heavily (mean abs error ~76 gray levels even downsampled) that
    detection on their frames cannot work."""
    if cam in _FLIP_CACHE:
        return
    from plus_common import DEFAULT_VIEW, categorize
    for i in range(suite.n_tasks):
        n = suite.get_task(i).name
        v = re.search(r"_view_(\d+)_(\d+)_(\d+)_(\d+)_(\d+)", n)
        s = re.search(r"_initstate_(\d+)", n)
        if categorize(n) == "language" and not (v and v.groups() != DEFAULT_VIEW) and not (s and s.group(1) != "0") \
                and scene_of(n) in nominals:
            break
    else:
        return
    env = make_env(i)
    try:
        obs = env.reset()
        Canonicalizer(env, nominals[scene_of(n)], cam=cam)._obs_convention(obs)
        print(f"image orientation for {cam}: {_FLIP_CACHE.get(cam)} (detected on clean task {i})", flush=True)
    finally:
        env.close()


def read_nominals(suite, make_env, cache=None, per_scene=2):
    """Nominal agentview camera and robot start PER SCENE. LIBERO-10 spans several rooms with different cameras,
    so a single global nominal is wrong for most tasks. Read from unperturbed tasks of each scene (language and
    sensor-noise variants with the default view and robot start) and require them to agree."""
    import json
    import os
    from plus_common import DEFAULT_VIEW, categorize
    if cache and os.path.exists(cache) and os.path.exists(cache + ".ref.npz"):
        out = json.load(open(cache))
        refs = np.load(cache + ".ref.npz")
        for s in out:
            if s in refs.files:
                out[s]["ref_image"] = refs[s]
        seed_flip(suite, make_env, out)
        if any("robot_shading_d" not in d or "table_z" not in d for d in out.values()):   # older cache: upgrade in place
            _add_shading_reference(suite, make_env, out, cache)
        return out
    src = {}
    for i in range(suite.n_tasks):
        n = suite.get_task(i).name
        v = re.search(r"_view_(\d+)_(\d+)_(\d+)_(\d+)_(\d+)", n)
        s = re.search(r"_initstate_(\d+)", n)
        if categorize(n) != "language" or (v and v.groups() != DEFAULT_VIEW) or (s and s.group(1) != "0"):
            continue
        ids = src.setdefault(scene_of(n), [])
        if len(ids) < per_scene:
            ids.append(i)
    out = {}
    for scene, ids in sorted(src.items()):
        got = []
        for i in ids:
            env = make_env(i)
            obs = env.reset()
            m = env.env.sim.model
            cid = m.camera_name2id("agentview")
            nom = dict(cam_pos=[float(x) for x in m.cam_pos[cid]], cam_quat_wxyz=[float(x) for x in m.cam_quat[cid]],
                       q_nom=[float(x) for x in env.env.robots[0].init_qpos])
            # photometric and noise references: the robot's own pixels and the image noise floor in the training setup
            mask = Canonicalizer(env, nom).obs_robot_mask(obs) > 0
            px = obs["agentview_image"][mask].reshape(-1, 3).astype(np.float64)
            nom.update(robot_rgb_mean=px.mean(0).tolist(), robot_rgb_std=px.std(0).tolist(),
                       noise={k: noise_level(obs[k]) for k in IMG_KEYS if k in obs})
            # training-view reference of the scene (a nominal-camera frame of an unperturbed task): fills reprojection
            # holes with what the training camera sees there, instead of inventing pixels by inpainting. The robot is
            # inpainted out of it (dilated mask) so a hole can never receive a ghost arm at the reset pose.
            ref = np.ascontiguousarray(obs["agentview_image"]).copy()
            robot = cv2.dilate(mask.astype(np.uint8), np.ones((9, 9), np.uint8))
            nom["ref_image"] = cv2.inpaint(ref, robot, 5, cv2.INPAINT_TELEA)
            got.append(nom)
            env.close()
        spread = max(float(np.abs(np.subtract(g["cam_pos"], got[0]["cam_pos"])).max()) for g in got)
        assert spread < 1e-4, f"{scene}: unperturbed tasks {ids} disagree on the camera by {spread:.4f} m"
        out[scene] = got[0]
        print(f"nominal {scene}: cam_pos {np.round(got[0]['cam_pos'], 4).tolist()} "
              f"q_nom {np.round(got[0]['q_nom'], 4).tolist()} (tasks {ids})", flush=True)
    if cache:
        json.dump({s: {k: v for k, v in d.items() if k != "ref_image"} for s, d in out.items()}, open(cache, "w"), indent=1)
        np.savez_compressed(cache + ".ref.npz", **{s: d["ref_image"] for s, d in out.items()})
    seed_flip(suite, make_env, out)
    _add_shading_reference(suite, make_env, out, cache)
    return out


def _add_shading_reference(suite, make_env, out, cache):
    """The body's shading statistic in the training setup, per scene (one unperturbed source task each), written into
    the cache. Cheap (one env per scene), so an older cache is upgraded in place rather than rebuilt."""
    import json
    from plus_common import DEFAULT_VIEW, categorize
    src = {}
    for i in range(suite.n_tasks):
        n = suite.get_task(i).name
        v = re.search(r"_view_(\d+)_(\d+)_(\d+)_(\d+)_(\d+)", n)
        s = re.search(r"_initstate_(\d+)", n)
        if categorize(n) != "language" or (v and v.groups() != DEFAULT_VIEW) or (s and s.group(1) != "0"):
            continue
        src.setdefault(scene_of(n), i)
    for scene, d in out.items():
        if ("robot_shading_d" in d and "table_z" in d) or scene not in src:
            continue
        env = make_env(src[scene])
        try:
            obs = env.reset()
            can = Canonicalizer(env, d)
            if "robot_shading_d" not in d:
                d["robot_shading_d"] = can.photometric_consistency(obs)
            if "table_z" not in d:
                d["table_z"] = can.training_plane_z()
        finally:
            env.close()
        print(f"training references {scene}: shading {d['robot_shading_d']:.3f}, table z {d['table_z']:.3f} m", flush=True)
    if cache:
        json.dump({s: {k: v for k, v in d.items() if k != "ref_image"} for s, d in out.items()}, open(cache, "w"), indent=1)


# ---------------------------------------------------------------- rotation helpers (pure numpy / cv2)
def quat_wxyz_to_mat(q):
    w, x, y, z = np.asarray(q, float) / np.linalg.norm(q)
    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
                     [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
                     [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


def mat_to_quat_wxyz(R):
    R = np.asarray(R, float)
    t = np.trace(R)
    if t > 0:
        s = 0.5 / np.sqrt(t + 1.0)
        q = np.array([0.25 / s, (R[2, 1] - R[1, 2]) * s, (R[0, 2] - R[2, 0]) * s, (R[1, 0] - R[0, 1]) * s])
    else:
        i = int(np.argmax(np.diag(R)))
        j, k = (i + 1) % 3, (i + 2) % 3
        s = np.sqrt(max(R[i, i] - R[j, j] - R[k, k] + 1.0, 1e-12)) * 2.0
        q = np.zeros(4)
        q[1 + i] = 0.25 * s
        q[0] = (R[k, j] - R[j, k]) / s
        q[1 + j] = (R[j, i] + R[i, j]) / s
        q[1 + k] = (R[k, i] + R[i, k]) / s
    return q / np.linalg.norm(q) * (1 if q[0] >= 0 else -1)


def quat_xyzw_to_mat(q):
    x, y, z, w = q
    return quat_wxyz_to_mat([w, x, y, z])


def rotvec_to_mat(r):
    return cv2.Rodrigues(np.asarray(r, np.float64).reshape(3, 1))[0]


def mat_to_rotvec(R):
    return cv2.Rodrigues(np.asarray(R, np.float64))[0].reshape(3)


def rot_z(deg):
    a = np.deg2rad(deg)
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def orbit(C0, R0, pivot, az_deg, el_deg, scale):
    """Camera pose after orbiting about `pivot`: azimuth about world z, elevation about the horizontal axis
    perpendicular to the viewing direction, then distance scaling. R columns are camera axes in world."""
    Rz = rot_z(az_deg)
    C = pivot + Rz @ (C0 - pivot)
    R = Rz @ R0
    v = C - pivot
    h = np.array([-v[1], v[0], 0.0])
    if np.linalg.norm(h) < 1e-9:
        h = R[:, 0].copy()
    h = h / np.linalg.norm(h)
    Rel = rotvec_to_mat(h * np.deg2rad(el_deg))
    C = pivot + Rel @ (C - pivot)
    R = Rel @ R
    C = pivot + scale * (C - pivot)
    return C, R


def optical_pivot(C0, R0, plane_z=TABLE_Z):
    d = -R0[:, 2]                                  # MuJoCo cameras look along -z
    t = (plane_z - C0[2]) / d[2] if abs(d[2]) > 1e-6 else 1.0
    return C0 + max(t, 0.1) * d


def edge(mask):
    m = mask.astype(np.uint8)
    return m & (1 - cv2.erode(m, np.ones((3, 3), np.uint8)))


def edge_dt(mask):
    """Distance from every pixel to the mask boundary."""
    return cv2.distanceTransform((1 - edge(mask)).astype(np.uint8), cv2.DIST_L2, 3)


def chamfer(obs_mask, ren_mask, dt_obs=None, trim=0.0):
    """Symmetric mean boundary distance in pixels between two binary masks. Boundary-based, so a shift of d pixels
    scores on the order of d rather than being diluted by the overlapping interior. trim > 0 drops that share of the
    largest boundary distances on each side: a learned mask that is wrong on one link then cannot drag the pose."""
    eo, er = edge(obs_mask), edge(ren_mask)
    if eo.sum() == 0 or er.sum() == 0:
        return 1e3
    if dt_obs is None:
        dt_obs = edge_dt(obs_mask)
    dt_ren = edge_dt(ren_mask)
    a, b = dt_ren[eo > 0], dt_obs[er > 0]
    if trim > 0:
        a = np.sort(a)[:max(1, int(round(len(a) * (1 - trim))))]
        b = np.sort(b)[:max(1, int(round(len(b) * (1 - trim))))]
    return 0.5 * (float(a.mean()) + float(b.mean()))


# the photometric fit on the robot's own body is trusted only when enough unclipped body pixels remain and the
# implied gain is sane; otherwise the channel abstains (Canonicalizer.photometric_fit)
PHOTO_MIN_PX, PHOTO_CLIP_LO, PHOTO_CLIP_HI, PHOTO_GAIN_MAX = 200, 3.0, 252.0, 2.5
# the photometric channel inverts a global gain/offset; when the body's shading PATTERN has changed (directional or
# grazing light) that model is wrong and the channel abstains. Ratio of the affine-invariant body statistic to its
# value in the training setup (photometric_consistency vs nominal["robot_shading_d"]): grazing scored 0.60 to 0.62 on
# the OFT sweep, every affine lighting change 0.68 or above.
PHOTO_SHADING_MIN = 0.65
IMG_KEYS = ("agentview_image", "robot0_eye_in_hand_image")
_FLIP_CACHE = {}                   # camera name -> confident raw-to-observation flip (fixed per camera)


def noise_level(img):
    """Robust per-pixel noise sigma in gray levels: scaled MAD of the residual after a 3x3 median filter."""
    g = cv2.cvtColor(np.ascontiguousarray(img), cv2.COLOR_RGB2GRAY)
    r = g.astype(np.float32) - cv2.medianBlur(g, 3).astype(np.float32)
    return float(1.4826 * np.median(np.abs(r - np.median(r))))


def denoise(img, level):
    h = float(max(3.0, 1.2 * level))
    return cv2.fastNlMeansDenoisingColored(np.ascontiguousarray(img), None, h, h, 3, 7)


def reproject(rgb, z, f, Ct, Rt, Cn, Rn, splat=2, inpaint=True, return_mask=False):
    """Warp an RGB-D image from camera (Ct, Rt) into camera (Cn, Rn). Raw OpenGL buffer convention: row 0 is the
    image bottom. z is metric depth along the optical axis. Nearest point wins; holes are inpainted."""
    H, W = z.shape
    rr, cc = np.meshgrid(np.arange(H), np.arange(W), indexing="ij")
    x = (cc + 0.5 - W / 2) * z / f
    y = (rr + 0.5 - H / 2) * z / f
    pc = np.stack([x, y, -z], -1).reshape(-1, 3)
    pw = Ct + pc @ Rt.T
    pn = (pw - Cn) @ Rn
    zn = -pn[:, 2]
    ok = zn > 1e-3
    col = np.round(f * pn[:, 0] / np.where(ok, zn, 1) + W / 2 - 0.5).astype(int)
    row = np.round(f * pn[:, 1] / np.where(ok, zn, 1) + H / 2 - 0.5).astype(int)
    colors = rgb.reshape(-1, 3)
    out = np.zeros_like(rgb)
    filled = np.zeros((H, W), bool)
    idx = np.where(ok)[0]
    idx = idx[np.argsort(-zn[idx])]                 # far first, near last: near overwrites
    for dr in range(splat):
        for dc in range(splat):
            r, c = row[idx] + dr, col[idx] + dc
            m = (r >= 0) & (r < H) & (c >= 0) & (c < W)
            out[r[m], c[m]] = colors[idx[m]]
            filled[r[m], c[m]] = True
    if inpaint and (~filled).any():
        out = cv2.inpaint(out, (~filled).astype(np.uint8), 3, cv2.INPAINT_TELEA)
    if return_mask:
        return out, float(filled.mean()), filled
    return out, float(filled.mean())


def plane_depth(H, W, f, C, R, plane_z, far=10.0):
    """Depth along the optical axis of the ray through every pixel (raw buffer convention, row 0 = bottom) to the
    horizontal plane z = plane_z in world coordinates, for a camera at C with axes R (MuJoCo: looks along -z).
    Rays that never meet the plane (pointing up, or the plane behind the camera) get `far`."""
    rr, cc = np.meshgrid(np.arange(H), np.arange(W), indexing="ij")
    d = np.stack([(cc + 0.5 - W / 2) / f, (rr + 0.5 - H / 2) / f, -np.ones((H, W))], -1)     # camera-frame ray, z = -1
    dw = d @ R.T                                                                                # world-frame ray
    dz = dw[..., 2]
    t = np.where(np.abs(dz) > 1e-6, (plane_z - C[2]) / np.where(np.abs(dz) > 1e-6, dz, 1.0), -1.0)
    return np.where(t > 1e-3, t, far)                                                           # depth = t since d_z = -1


def points_depth(pts, f, C, R, H, W, splat=2, grow=3, see_through=0.04):
    """Depth along the optical axis of a WORLD point cloud z-buffered into camera (C, R); raw buffer convention, same
    projection as `reproject`. Pixels no point reaches are inf. Nearest point wins; `grow` passes of a 3x3 minimum
    over EMPTY pixels close the gaps a cloud sampled from another viewpoint leaves."""
    pc = (np.asarray(pts, np.float64) - C) @ R
    zn = -pc[:, 2]
    idx = np.where(zn > 1e-3)[0]
    idx = idx[np.argsort(-zn[idx])]                 # far first, near last: near overwrites
    col = np.round(f * pc[idx, 0] / zn[idx] + W / 2 - 0.5).astype(int)
    row = np.round(f * pc[idx, 1] / zn[idx] + H / 2 - 0.5).astype(int)
    z = np.full((H, W), np.inf)
    for dr in range(splat):
        for dc in range(splat):
            r, c = row + dr, col + dc
            m = (r >= 0) & (r < H) & (c >= 0) & (c < W)
            z[r[m], c[m]] = zn[idx][m]
    for _ in range(grow):
        mn = cv2.erode(np.where(np.isfinite(z), z, 1e9).astype(np.float32), np.ones((3, 3), np.uint8))
        z = np.where(np.isfinite(z), z, np.where(mn < 1e8, mn, np.inf))
    # a point cloud is see-through: far points show between the samples of a near surface. A pixel lying well behind
    # the nearest depth in its 5x5 neighbourhood takes that depth (costs a 2 px fattening of near silhouettes)
    mn = cv2.erode(np.where(np.isfinite(z), z, 1e9).astype(np.float32), np.ones((5, 5), np.uint8))
    return np.where(np.isfinite(z) & (z > mn + see_through), mn, z)


def clean_mask(mask, min_px=30, keep_frac=0.15):
    """Speckle-free binary mask with its holes filled (uint8 0/1): every connected component at least `keep_frac` of
    the largest one is kept. Keeping ONLY the largest (before 2026-09-19) threw away half the robot whenever the image
    border cut the arm in two, which is the normal case under a large camera shift (task 768: mask IoU 0.50 on that
    frame, 31 cm calibration error). Tiny masks are returned as is."""
    m = (np.asarray(mask) > 0).astype(np.uint8)
    if m.sum() < min_px:
        return m
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    if n > 2:
        area = stats[1:, cv2.CC_STAT_AREA]
        keep = 1 + np.nonzero(area >= max(min_px, keep_frac * area.max()))[0]
        m = np.isin(lab, keep).astype(np.uint8)
    inv = (1 - m).astype(np.uint8)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(inv, connectivity=4)
    border = set(np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]])))
    for k in range(1, n):
        if k not in border:                    # a background component not touching the border is a hole
            m[lab == k] = 1
    return m


def backproject(z, mask, f):
    """Camera-frame 3D points (MuJoCo convention: looks along -z, raw buffer row 0 = bottom) of masked pixels."""
    H, W = z.shape
    rr, cc = np.nonzero(mask)
    zz = z[rr, cc]
    return np.stack([(cc + 0.5 - W / 2) * zz / f, (rr + 0.5 - H / 2) * zz / f, -zz], -1)


def kabsch(P, Q):
    """R, C minimising sum ||R p + C - q||^2 over paired rows of P and Q."""
    mp, mq = P.mean(0), Q.mean(0)
    Hm = (P - mp).T @ (Q - mq)
    U, _, Vt = np.linalg.svd(Hm)
    D = np.eye(3)
    D[2, 2] = np.sign(np.linalg.det(Vt.T @ U.T))
    R = Vt.T @ D @ U.T
    return R, mq - R @ mp


def _nearest(model, query):
    try:
        from scipy.spatial import cKDTree
        d, i = cKDTree(model).query(query)
        return d, i
    except Exception:
        d = np.empty(len(query)); i = np.empty(len(query), int)
        for s in range(0, len(query), 512):
            dd = ((query[s:s + 512, None] - model[None]) ** 2).sum(-1)
            i[s:s + 512] = dd.argmin(1)
            d[s:s + 512] = np.sqrt(dd[np.arange(len(dd)), i[s:s + 512]])
        return d, i


def icp(P_cam, Q_world, R0, C0, iters=40, keep=0.8):
    """Rigid camera pose (R, C) aligning camera-frame points to a world-frame model; trimmed point-to-point ICP."""
    R, C = R0.copy(), C0.copy()
    for _ in range(iters):
        Pw = P_cam @ R.T + C
        d, idx = _nearest(Q_world, Pw)
        k = d <= np.quantile(d, keep)
        R_new, C_new = kabsch(P_cam[k], Q_world[idx[k]])
        done = np.linalg.norm(C_new - C) < 1e-5 and np.linalg.norm(mat_to_rotvec(R_new @ R.T)) < 1e-5
        R, C = R_new, C_new
        if done:
            break
    d, _ = _nearest(Q_world, P_cam @ R.T + C)
    return R, C, float(np.mean(np.sort(d)[: max(1, int(keep * len(d)))]))




# ---------------------------------------------------------------- simulator-facing wrapper
class Canonicalizer:
    def __init__(self, env, nominal, cam="agentview", res=256, mask_res=96, point_res=128, sensing=None):
        """env: LIBERO OffScreenRenderEnv. nominal: dict(cam_pos, cam_quat_wxyz, q_nom) read from an
        unperturbed task (the training setup)."""
        import mujoco
        self.mj = mujoco
        self.env, self.sim, self.cam, self.res, self.mres, self.pres = env, env.env.sim, cam, res, mask_res, point_res
        self.sensing = sensing          # None = renderer stand-ins; LearnedSensing = trained segmenter + monocular depth
        ref = nominal.get("ref_image")
        self.ref_image = None if ref is None else np.asarray(ref, np.uint8)
        m = self.sim.model
        self.cid = m.camera_name2id(cam)
        assert int(m.cam_bodyid[self.cid]) == 0, "camera must be attached to the world body"
        self.robot = env.env.robots[0]
        self.arm_idx = list(self.robot._ref_joint_pos_indexes)
        self.grip_idx = list(getattr(self.robot, "_ref_gripper_joint_pos_indexes", []))
        self.robot_geoms = np.array([g for g in range(m.ngeom)
                                     if self._body_name(int(m.geom_bodyid[g])).startswith(ROBOT_PREFIXES)])
        assert len(self.robot_geoms) > 0, "no robot geoms found"
        self.Cn = np.asarray(nominal["cam_pos"], float)
        self.Rn = quat_wxyz_to_mat(nominal["cam_quat_wxyz"])
        self.q_nom = np.asarray(nominal["q_nom"], float)
        self.table_z = nominal.get("table_z")           # training-setup table plane (world z), for plane-depth sensing
        self.static_pts = nominal.get("static_pts")     # training-setup static geometry (world points), hybrid depth
        self.gripper_geoms = np.array([g for g in range(m.ngeom)
                                       if self._body_name(int(m.geom_bodyid[g])).startswith("gripper0")])
        self._grip_max = 0.0
        self.mask_diag = []                             # learned mask vs renderer mask at the true camera (logging only)
        self.fovy = float(m.cam_fovy[self.cid])
        self.flip = None
        self.C_est = self.R_est = None

    def _body_name(self, i):
        m = self.sim.model
        try:
            return m.body_id2name(i) or ""
        except Exception:
            return self.mj.mj_id2name(m._model, self.mj.mjtObj.mjOBJ_BODY, i) or ""

    # -- state save / restore (bit-exact, camera included)
    def _save(self):
        m = self.sim.model
        return (np.array(self.sim.get_state().flatten()), self.sim.data.qacc_warmstart.copy(),
                m.cam_pos[self.cid].copy(), m.cam_quat[self.cid].copy())

    def _restore(self, s):
        m = self.sim.model
        m.cam_pos[self.cid][:] = s[2]
        m.cam_quat[self.cid][:] = s[3]
        self.sim.set_state_from_flattened(s[0])
        self.sim.data.qacc_warmstart[:] = s[1]
        self.sim.forward()

    def true_camera(self):
        m = self.sim.model
        return m.cam_pos[self.cid].copy(), quat_wxyz_to_mat(m.cam_quat[self.cid])

    def _set_camera(self, C, R):
        m = self.sim.model
        m.cam_pos[self.cid][:] = C
        m.cam_quat[self.cid][:] = mat_to_quat_wxyz(R)

    def _focal(self, H):
        return 0.5 * H / np.tan(self.fovy * np.pi / 360)

    def grip_opening(self):
        """Finger opening from proprioception (sum of absolute finger joint positions)."""
        return float(np.abs(self.sim.data.qpos[self.grip_idx]).sum()) if self.grip_idx else 0.0

    def grip_closed(self, frac=0.75):
        """Closed = the fingers are well inside the widest opening seen this episode. Proprioception only."""
        o = self.grip_opening()
        self._grip_max = max(self._grip_max, o)
        return self._grip_max > 0 and o < frac * self._grip_max

    def _gripper_mask(self, res):
        seg = self.sim.render(width=res, height=res, camera_name=self.cam, segmentation=True)
        return ((seg[..., 0] == int(self.mj.mjtObj.mjOBJ_GEOM)) & np.isin(seg[..., 1], self.gripper_geoms)).astype(np.uint8)

    def _static_mask(self, res):
        """Pixels of geometry that cannot move: not the robot, and not a body whose root carries a free joint."""
        m = getattr(self.sim.model, "_model", self.sim.model)      # raw mujoco model: every field is there
        seg = self.sim.render(width=res, height=res, camera_name=self.cam, segmentation=True)
        gid = seg[..., 1].astype(int)
        isgeom = (seg[..., 0] == int(self.mj.mjtObj.mjOBJ_GEOM)) & (gid >= 0) & (gid < m.ngeom)
        free = int(self.mj.mjtJoint.mjJNT_FREE)
        fixed = np.ones(m.ngeom, bool)
        for g in range(m.ngeom):
            root = int(m.body_rootid[int(m.geom_bodyid[g])])
            if int(m.body_jntnum[root]) > 0 and int(m.jnt_type[int(m.body_jntadr[root])]) == free:
                fixed[g] = False
        fixed[self.robot_geoms] = False
        out = np.zeros(gid.shape, np.uint8)
        out[isgeom] = fixed[gid[isgeom]]
        return out

    def training_static_cloud(self, res=384, voxel=0.006):
        """World points of the training setup's STATIC geometry (table, fixtures, walls, floor) scanned once from
        the nominal camera and a ring of viewpoints around it, robot and movable objects removed, 6 mm voxels. Training-setup
        information, like the reference image and the table plane; never computed on a deployment frame."""
        pivot = optical_pivot(self.Cn, self.Rn, self.table_z if self.table_z is not None else TABLE_Z)
        views = [(self.Cn, self.Rn)] + [orbit(self.Cn, self.Rn, pivot, az, el, 1.0)
                                        for az in (-60, -30, 30, 60) for el in (-15, 0, 15)]
        out = []
        s = self._save()
        try:
            for C, R in views:              # a scan: a single view misses the faces a shifted camera will look at
                self._set_camera(C, R)
                self.sim.forward()
                z, _ = self._depth_and_mask(res)
                st = self._static_mask(res)
                pts = backproject(z, (st > 0) & (z < 5.0), self._focal(res))
                out.append(C + pts @ R.T)
        finally:
            self._restore(s)
        pts = np.concatenate(out)
        _, keep = np.unique(np.round(pts / voxel).astype(np.int64), axis=0, return_index=True)
        return pts[keep].astype(np.float32)

    def _robot_mask(self, res):
        seg = self.sim.render(width=res, height=res, camera_name=self.cam, segmentation=True)
        return ((seg[..., 0] == int(self.mj.mjtObj.mjOBJ_GEOM)) & np.isin(seg[..., 1], self.robot_geoms)).astype(np.uint8)

    def _obs_convention(self, obs):
        """Which flip maps the raw render onto the observation image. Compared on 32x32 area-downsampled images so
        heavy sensor noise averages out (full-resolution errors reach ~77 gray levels on LIBERO-Plus noise tasks).
        The flip is fixed per camera, so a confident detection is cached and reused if a later frame is ambiguous."""
        if self.cam in _FLIP_CACHE:         # fixed per camera: never re-detect on a (possibly corrupted) frame
            self.flip = _FLIP_CACHE[self.cam]
            return
        raw = self.sim.render(width=self.res, height=self.res, camera_name=self.cam)
        img = obs[f"{self.cam}_image"]
        small = lambda a: cv2.resize(np.ascontiguousarray(a).astype(np.float32), (32, 32), interpolation=cv2.INTER_AREA)
        ref = small(img)
        cands = {"id": raw, "ud": raw[::-1], "lr": raw[:, ::-1], "rot": raw[::-1, ::-1]}
        errs = {k: float(np.abs(small(v) - ref).mean()) for k, v in cands.items()}
        best = min(errs, key=errs.get)
        second = sorted(errs.values())[1]
        if errs[best] < 40.0 and second - errs[best] > 2.0:
            self.flip = best
            _FLIP_CACHE[self.cam] = best
        elif self.cam in _FLIP_CACHE:
            self.flip = _FLIP_CACHE[self.cam]
        else:
            raise AssertionError(f"cannot match raw render to observation image: {errs}")

    def _to_obs(self, raw):
        return {"id": raw, "ud": raw[::-1], "lr": raw[:, ::-1], "rot": raw[::-1, ::-1]}[self.flip].copy()

    # -- state channel
    def joint_dev(self):
        return float(np.abs(self.sim.data.qpos[self.arm_idx] - self.q_nom).max())

    def nominal_eef(self):
        s = self._save()
        self.sim.data.qpos[self.arm_idx] = self.q_nom
        self.sim.forward()
        ob = self.env.env._get_observations(force_update=True)
        p, q = np.array(ob["robot0_eef_pos"]), np.array(ob["robot0_eef_quat"])
        self._restore(s)
        self.env.env._get_observations(force_update=True)
        return p, q

    def servo(self, obs, p_t, q_t_xyzw, max_steps=150, tol_p=0.004, tol_r=0.03):
        Rt = quat_xyzw_to_mat(q_t_xyzw)
        for i in range(max_steps):
            dp = p_t - np.asarray(obs["robot0_eef_pos"])
            dr = mat_to_rotvec(Rt @ quat_xyzw_to_mat(obs["robot0_eef_quat"]).T)
            if np.linalg.norm(dp) < tol_p and np.linalg.norm(dr) < tol_r:
                return obs, i
            a = np.zeros(7)
            a[:3] = np.clip(dp / 0.05, -1, 1)
            a[3:6] = np.clip(dr / 0.5, -1, 1)
            a[6] = -1.0
            obs, _, _, _ = self.env.step(a.tolist())
        return obs, max_steps

    def settle(self, obs, p_t, q_t_xyzw, max_steps=120, tol_p=0.0008, tol_r=0.006, rest=0.01):
        """Finish a measurement the way an episode starts: AT the target and AT REST. `servo` stops inside 4 mm with the
        arm still moving (measured 2026-09-19: 2.6 mm off and 0.19 rad/s at hand-over, against 0.00 to 0.03 rad/s for an
        untouched start), so a `full` episode began from a different physical state than its `raw` twin even when the
        wrapper then applied nothing. Keeps servoing (never a zero action: under it the arm sags 6 mm) until the pose is
        inside 0.8 mm / 0.006 rad AND every arm joint is slower than `rest` rad/s. Measured: 0.16 mm, 0.005 rad/s, 19 steps.
        Returns (obs, dict(steps, err_mm, speed))."""
        Rt = quat_xyzw_to_mat(q_t_xyzw)
        i = 0
        for i in range(max_steps):
            dp = p_t - np.asarray(obs["robot0_eef_pos"])
            dr = mat_to_rotvec(Rt @ quat_xyzw_to_mat(obs["robot0_eef_quat"]).T)
            speed = float(np.abs(self.sim.data.qvel[self.arm_idx]).max())
            if np.linalg.norm(dp) < tol_p and np.linalg.norm(dr) < tol_r and speed < rest:
                break
            a = np.zeros(7)
            a[:3] = np.clip(dp / 0.05, -1, 1)
            a[3:6] = np.clip(dr / 0.5, -1, 1)
            a[6] = -1.0
            obs, _, _, _ = self.env.step(a.tolist())
        return obs, dict(steps=i, err_mm=float(1e3 * np.linalg.norm(p_t - np.asarray(obs["robot0_eef_pos"]))),
                         speed=float(np.abs(self.sim.data.qvel[self.arm_idx]).max()))

    # -- camera channel
    def sweep(self, obs, poses=None):
        """poses: optional list of end-effector offsets; default is the method's sweep (3 poses, 7 under learned masks).
        The `verify` arm passes the 7-pose list when the self-check rejects the first calibration."""
        p0, q0 = self.nominal_eef()
        recs = []
        for off in (poses if poses is not None else (SWEEP if self.sensing is None else SWEEP_LEARNED)):
            obs, _ = self.servo(obs, p0 + off, q0)
            img = np.ascontiguousarray(obs[f"{self.cam}_image"]).copy()
            if self.sensing is None:
                mask, pts = self._robot_mask(self.mres), self._robot_points_camera(self.pres)
            else:                           # learned: segment the observed frame; metric points come after the silhouette fit
                if self.flip is None:
                    self._obs_convention(obs)
                mask = cv2.resize(self._to_obs(self.sensing.mask_obs(img)), (self.mres, self.mres),
                                  interpolation=cv2.INTER_NEAREST)
                pts = None
                true = self._robot_mask(self.mres)           # LOGGING ONLY: never reaches the method
                u = float(((mask > 0) | (true > 0)).sum())
                self.mask_diag.append(dict(iou=float(((mask > 0) & (true > 0)).sum() / u) if u else 0.0,
                                           area_ratio=float((mask > 0).sum() / max(float((true > 0).sum()), 1.0))))
            recs.append((self.sim.data.qpos[self.arm_idx].copy(),
                         self.sim.data.qpos[self.grip_idx].copy() if self.grip_idx else None,
                         mask, pts, img))
        obs, _ = self.servo(obs, p0, q0)
        return obs, recs

    def static_residual(self, obs, area_w=4.0):
        """MOTION-FREE check: at the pose the arm is in NOW, does the robot's silhouette sit where the TRAINING camera says
        it must? Same silhouette score as the calibration (`cal_nominal_score`), from one record and no arm motion. Small:
        the camera has not moved, so the measurement sweep would be pure cost (it moves the arm, and a sampling policy then
        starts its episode from a different state). Logged rows, 2026-09-21: every one of 240 camera-shifted tasks reads
        above 3.0, every one of 140 others below 0.02 (sweep conditions: above 2.26 against below 0.86)."""
        if self.sensing is None:
            mask, pts = self._robot_mask(self.mres), self._robot_points_camera(self.pres)
        else:
            img = np.ascontiguousarray(obs[f"{self.cam}_image"]).copy()
            if self.flip is None:
                self._obs_convention(obs)
            mask = cv2.resize(self._to_obs(self.sensing.mask_obs(img)), (self.mres, self.mres), interpolation=cv2.INTER_NEAREST)
            pts = None
        rec = (self.sim.data.qpos[self.arm_idx].copy(), self.sim.data.qpos[self.grip_idx].copy() if self.grip_idx else None,
               mask, pts, None)
        s = self._save()
        try:
            return float(self._score([rec], [(edge_dt(mask), mask)], self.Cn, self.Rn, area_w, 0.0, clean=self.sensing is not None))
        finally:
            self._restore(s)

    def _score(self, recs, dts, C, R, area_w=0.0, trim=0.0, res=None, clean=False):
        """Mean boundary chamfer (pixels) over the sweep poses. area_w adds |log(rendered area / observed area)|:
        pixel chamfer alone shrinks as a candidate camera backs away and the robot gets small, so without the size
        term the search drifts to far cameras."""
        self._set_camera(C, R)
        total = 0.0
        for rec, dt in zip(recs, dts):
            qa, qg = rec[0], rec[1]
            self.sim.data.qpos[self.arm_idx] = qa
            if qg is not None:
                self.sim.data.qpos[self.grip_idx] = qg
            self.sim.forward()
            ren = self._robot_mask(res or self.mres)
            if clean:                       # compare like with like: the observed (learned) mask went through clean_mask,
                ren = clean_mask(ren)       # so stray robot geometry far from the arm must not count in the render either
            total += chamfer(dt[1], ren, dt_obs=dt[0], trim=trim)
            if area_w:
                total += area_w * abs(np.log((ren.sum() + 1.0) / (dt[1].sum() + 1.0)))
        return total / len(recs)

    def calibrate(self, recs, grid_az=range(-60, 61, 10), grid_el=range(-20, 21, 10),
                  grid_scale=(0.8, 1.0, 1.25, 1.5, 1.75, 2.0), cem_iters=8, pop=24, elite=6, seed=0):
        dts = [(edge_dt(rec[2]), rec[2]) for rec in recs]
        s = self._save()
        try:
            nominal_score = self._score(recs, dts, self.Cn, self.Rn)
            pivot = optical_pivot(self.Cn, self.Rn)
            grid = []
            for az in grid_az:
                for el in grid_el:
                    for sc in grid_scale:
                        C, R = orbit(self.Cn, self.Rn, pivot, az, el, sc)
                        grid.append((self._score(recs, dts, C, R), C, R))
            grid.sort(key=lambda g: g[0])
            best_score, Cb, Rb = grid[0]
            rng = np.random.default_rng(seed)
            mu, sd = np.zeros(6), np.array([0.06, 0.06, 0.06, 0.08, 0.08, 0.08])
            for _ in range(cem_iters):
                cand = mu + sd * rng.standard_normal((pop, 6))
                cand[0] = mu
                sc_ = []
                for c in cand:
                    C, R = Cb + c[:3], rotvec_to_mat(c[3:]) @ Rb
                    sc_.append(self._score(recs, dts, C, R))
                sc_ = np.array(sc_)
                el_ = cand[np.argsort(sc_)[:elite]]
                if sc_.min() < best_score:
                    best_score = float(sc_.min())
                mu, sd = el_.mean(0), el_.std(0) + 1e-4
            self.C_est, self.R_est = Cb + mu[:3], rotvec_to_mat(mu[3:]) @ Rb
            final = self._score(recs, dts, self.C_est, self.R_est)
        finally:
            self._restore(s)
        return dict(nominal_score=nominal_score, grid_best=grid[0][0], final_score=final)

    def calibrate_v2(self, recs, max_cm=110.0, max_deg=80.0, area_w=4.0, accept=0.8, seed=0):
        """Camera pose from the sweep. Silhouette grid + CEM with the size term and a plausibility bound, keeping the
        best candidate seen (not the CEM mean); then depth ICP refined from the best silhouette pose and from nominal.
        Every candidate is judged by the same silhouette score, and the nominal pose is kept unless beaten by the
        factor `accept`. The orbit pivot sits on the nominal optical axis at the robot's measured median depth."""
        dts = [(edge_dt(rec[2]), rec[2]) for rec in recs]
        learned = any(rec[3] is None for rec in recs)
        trim = 0.0      # measured 2026-09-18: trimming changes no synthetic fit (blob, shadow, missing link); kept at 0
        s = self._save()
        out = dict(method="nominal", icp_residual_cm=None)
        try:
            if learned:                     # pivot depth from the robot's known geometry rendered at the nominal camera
                self._set_camera(self.Cn, self.Rn)
                self.sim.data.qpos[self.arm_idx] = recs[0][0]
                self.sim.forward()
                z0, mk0 = self._depth_and_mask(self.pres)
                pivot_depth = float(np.median(z0[mk0 > 0])) if mk0.any() else 1.0
                P = None
            else:
                P = np.concatenate([rec[3] for rec in recs])
                pivot_depth = float(np.median(-P[:, 2]))
            pivot = self.Cn + pivot_depth * (-self.Rn[:, 2])
            def score(C, R):
                e = pose_error(C, R, self.Cn, self.Rn)
                if e[0] > max_cm or e[1] > max_deg:
                    return np.inf
                return self._score(recs, dts, C, R, area_w, trim, clean=learned)

            nominal = score(self.Cn, self.Rn)
            best = (nominal, self.Cn, self.Rn, "nominal")
            cands = []
            # measured LIBERO-Plus camera shifts reach 99 cm and 70 deg; calibration was exact up to 35 cm / 31 deg and
            # failed beyond, where the old 35 cm / 35 deg bound pinned it. Coarser steps keep the grid size (539 poses).
            for az in range(-75, 76, 15):
                for el in range(-45, 46, 15):
                    for sc in (0.7, 0.85, 1.0, 1.2, 1.45, 1.75, 2.1):
                        C, R = orbit(self.Cn, self.Rn, pivot, az, el, sc)
                        v = score(C, R)
                        if np.isfinite(v):
                            cands.append((v, C, R))
            cands.sort(key=lambda x: x[0])
            rng = np.random.default_rng(seed)
            for v0, Cb, Rb in cands[:3]:
                if v0 < best[0]:
                    best = (v0, Cb, Rb, "silhouette")
                mu, sd = np.zeros(6), np.array([0.04, 0.04, 0.04, 0.05, 0.05, 0.05])
                for _ in range(6):
                    X = mu + sd * rng.standard_normal((20, 6))
                    X[0] = mu
                    vals = []
                    for x in X:
                        C, R = Cb + x[:3], rotvec_to_mat(x[3:]) @ Rb
                        v = score(C, R)
                        vals.append(v)
                        if v < best[0]:
                            best = (v, C, R, "silhouette")
                    elite = X[np.argsort(np.array(vals))[:5]]
                    mu, sd = elite.mean(0), elite.std(0) + 1e-4
            if learned:                     # metric robot points: learned mask + monocular depth scaled at the silhouette pose
                P = self.sensing.robot_points(self, recs, best[1], best[2])
            Q = self._robot_model_world(recs, self.pres, pivot)
            for _, C0, R0, _ in [best, (nominal, self.Cn, self.Rn, "nominal")]:
                R, C, resid = icp(P, Q, R0, C0)
                v = score(C, R)
                if v < best[0]:
                    best = (v, C, R, "icp")
                    out["icp_residual_cm"] = resid * 100
            if learned and best[3] != "nominal" and getattr(self.sensing, "fine_cal", False):
                # fine stage at twice the mask resolution (the 96 px score leaves about 3 px of slack at 256): masks
                # re-segmented from the stored sweep frames, trimmed chamfer, CEM from 3 cm / 3 deg down
                F = 2 * self.mres
                fm = [cv2.resize(self._to_obs(self.sensing.mask_obs(rec[4])), (F, F), interpolation=cv2.INTER_NEAREST)
                      for rec in recs]
                fdts = [(edge_dt(k), k) for k in fm]
                fine = lambda C, R: self._score(recs, fdts, C, R, area_w, trim, res=F, clean=True)
                fb = (fine(best[1], best[2]), best[1], best[2])
                out["fine_score_before"] = float(fb[0])
                mu, sd = np.zeros(6), np.array([0.03, 0.03, 0.03, 0.05, 0.05, 0.05])
                for _ in range(5):
                    X = mu + sd * rng.standard_normal((24, 6))
                    X[0] = mu
                    vals = []
                    for x in X:
                        C, R = best[1] + x[:3], rotvec_to_mat(x[3:]) @ best[2]
                        v = fine(C, R)
                        vals.append(v)
                        if v < fb[0]:
                            fb = (v, C, R)
                    elite = X[np.argsort(np.array(vals))[:6]]
                    mu, sd = elite.mean(0), elite.std(0) + 1e-4
                out["fine_score_after"] = float(fb[0])
                v = score(fb[1], fb[2])
                if v <= best[0] + 0.3:              # never trade the coarse agreement away
                    best = (min(v, best[0]), fb[1], fb[2], best[3] + "+fine")
        finally:
            self._restore(s)
        # relative AND absolute margin: near a perfect nominal match (score ~0) a ratio alone fires on noise
        applied = best[3] != "nominal" and best[0] < accept * nominal and nominal - best[0] > 0.5
        self.C_est, self.R_est = (best[1], best[2]) if applied else (self.Cn.copy(), self.Rn.copy())
        shift = pose_error(self.C_est, self.R_est, self.Cn, self.Rn)
        out.update(nominal_score=float(nominal), final_score=float(best[0]), applied=bool(applied),
                   method=best[3] if applied else "nominal", est_shift_cm=shift[0], est_shift_deg=shift[1])
        return out

    # -- depth-based calibration (ICP of the robot's own surface)
    def _depth_and_mask(self, res):
        rgb, depth = self.sim.render(width=res, height=res, camera_name=self.cam, depth=True)
        m = self.sim.model
        extent = float(m.stat.extent)
        near, far = float(m.vis.map.znear) * extent, float(m.vis.map.zfar) * extent
        z = near / (1.0 - depth * (1.0 - near / far))
        return z, self._robot_mask(res)

    def training_plane_z(self, res=128):
        """World height of the dominant horizontal surface seen by the nominal camera in the training setup: the
        table. Renderer depth at the nominal camera, non-robot pixels, histogram mode over 1 cm bins. Training-setup
        information, like the reference image; never computed on a deployment frame."""
        s = self._save()
        try:
            self._set_camera(self.Cn, self.Rn)
            self.sim.forward()
            z, mk = self._depth_and_mask(res)
        finally:
            self._restore(s)
        pts = backproject(z, (mk == 0) & (z < 5.0), self._focal(res))
        zw = (self.Cn + pts @ self.Rn.T)[:, 2]
        hist, edges = np.histogram(zw, bins=np.arange(zw.min(), zw.max() + 0.01, 0.01))
        return float(edges[int(np.argmax(hist))] + 0.005)

    def _robot_points_camera(self, res):
        """Observed robot surface in the (unknown) actual camera's frame, from the RGB-D image and arm segmentation."""
        z, mk = self._depth_and_mask(res)
        pts = backproject(z, mk, self._focal(res))
        if len(pts) > 3000:
            pts = pts[np.random.default_rng(0).choice(len(pts), 3000, replace=False)]
        return pts

    def _robot_model_world(self, recs, res, pivot=None):
        """The robot's surface in world coordinates at the recorded joint angles, rendered from known virtual
        viewpoints around the nominal camera (robot geometry plus known joints only)."""
        s = self._save()
        pts = []
        try:
            pivot = optical_pivot(self.Cn, self.Rn) if pivot is None else pivot
            views = [(self.Cn, self.Rn)] + [orbit(self.Cn, self.Rn, pivot, az, el, 1.0)
                                            for az, el in ((-40, 0), (40, 0), (0, 20))]
            for rec in recs:
                self.sim.data.qpos[self.arm_idx] = rec[0]
                if rec[1] is not None:
                    self.sim.data.qpos[self.grip_idx] = rec[1]
                for C, R in views:
                    self._set_camera(C, R)
                    self.sim.forward()
                    z, mk = self._depth_and_mask(res)
                    pc = backproject(z, mk, self._focal(res))
                    pts.append(C + pc @ R.T)
        finally:
            self._restore(s)
        Q = np.concatenate(pts)
        if len(Q) > 12000:
            Q = Q[np.random.default_rng(1).choice(len(Q), 12000, replace=False)]
        return Q

    def obs_robot_mask(self, obs):
        """Robot mask in the observation image's orientation and resolution (segmentation stand-in)."""
        if self.flip is None:
            self._obs_convention(obs)
        if self.sensing is not None:
            return self.sensing.mask_obs(obs[f"{self.cam}_image"])
        return self._to_obs(self._robot_mask(obs[f"{self.cam}_image"].shape[0]))

    # -- photometric channel
    def photometric_fit(self, obs, nominal):
        """Per-channel gain and offset mapping the robot's current appearance onto its appearance in the training
        setup. The robot's own body is the reference, so table, background and layout changes do not enter the
        estimate. Call with the arm at its nominal pose.

        Clipped pixels carry no information about a linear map, so they are dropped from the fit. If too few pixels
        survive, or the implied gain leaves a sane range, the channel abstains (photo_ok False) rather than apply a
        map the robot's own measurements do not support: a saturated or strongly directional lighting change is not
        identifiable from the body alone. The reported shift and gain still use every robot pixel, so the
        identifiability score sees the full magnitude of the change either way."""
        mask = self.obs_robot_mask(obs) > 0
        allpx = obs[f"{self.cam}_image"][mask].reshape(-1, 3).astype(np.float32)
        rmu = np.asarray(nominal["robot_rgb_mean"], np.float32)
        rsd = np.asarray(nominal["robot_rgb_std"], np.float32)
        keep = ((allpx > PHOTO_CLIP_LO) & (allpx < PHOTO_CLIP_HI)).all(1)
        n_keep, n_all = int(keep.sum()), max(int(len(allpx)), 1)
        ok = n_keep >= PHOTO_MIN_PX
        px = allpx[keep] if ok else allpx
        mu, sd = px.mean(0), px.std(0) + 1e-3
        gain = rsd / sd
        ok = bool(ok and np.all(gain > 1.0 / PHOTO_GAIN_MAX) and np.all(gain < PHOTO_GAIN_MAX))
        d0 = nominal.get("robot_shading_d")
        shading = float("nan")
        if d0 is not None and np.isfinite(d0) and d0 > 0:
            d = self.photometric_consistency(obs)
            shading = float(min(d, d0) / max(d, d0)) if np.isfinite(d) and max(d, d0) > 0 else float("nan")
            ok = bool(ok and (np.isnan(shading) or shading >= PHOTO_SHADING_MIN))
        self.photo = (gain, rmu - gain * mu) if ok else None
        return dict(photo_ok=ok, photo_mean_shift=float(np.abs(allpx.mean(0) - rmu).max()),
                    photo_gain=[float(g) for g in gain], photo_px=n_keep,
                    photo_clipped=float(1.0 - n_keep / n_all), photo_shading=shading)

    def apply_photo(self, img):
        if self.photo is None:                 # abstained: leave the image as measured
            return img
        gain, off = self.photo
        return np.clip(img.astype(np.float32) * gain + off, 0, 255).astype(np.uint8)

    def photometric_consistency(self, obs, min_px=50):
        """Affine-invariant split-half statistic of the robot's own body: log ratio of per-channel spread between the
        upper and lower halves of the body plus their mean difference in units of the whole-body spread. A global
        gain/offset change (the only lighting change the photometric channel can invert) leaves it unchanged; a
        directional or grazing change alters the body's shading pattern and moves it. Compared against the same
        statistic at the unperturbed state, it says whether the affine inversion model is valid, from the body alone.
        Returns nan when the body is too small to split."""
        mask = self.obs_robot_mask(obs) > 0
        img = obs[f"{self.cam}_image"].astype(np.float32)
        rows = np.where(mask.any(1))[0]
        if len(rows) < 4:
            return float("nan")
        mid = int((rows.min() + rows.max()) // 2)
        top = img[:mid][mask[:mid]].reshape(-1, 3)
        bot = img[mid:][mask[mid:]].reshape(-1, 3)
        if len(top) < min_px or len(bot) < min_px:
            return float("nan")
        sd = img[mask].reshape(-1, 3).std(0) + 1e-3
        d = np.abs(np.log((top.std(0) + 1e-3) / (bot.std(0) + 1e-3))).sum() + np.abs((top.mean(0) - bot.mean(0)) / sd).sum()
        return float(d)

    def canonical_image(self, obs, C=None, R=None, return_mask=False, fill="ref"):
        """The OBSERVED image reprojected into the nominal camera, using the renderer's depth at the actual camera
        (RGB-D stand-in) and camera pose (C, R) (default: the self-calibrated estimate)."""
        if self.flip is None:
            self._obs_convention(obs)
        C = self.C_est if C is None else C
        R = self.R_est if R is None else R
        img = obs[f"{self.cam}_image"]
        H = img.shape[0]
        if self.sensing is not None:
            z = self.sensing.metric_depth(self, img, C, R)
        else:
            _, depth = self.sim.render(width=H, height=H, camera_name=self.cam, depth=True)
            m = self.sim.model
            extent = float(m.stat.extent)
            near, far = float(m.vis.map.znear) * extent, float(m.vis.map.zfar) * extent
            z = near / (1.0 - depth * (1.0 - near / far))
        # the flips are involutions, so _to_obs also maps the observation back to the raw buffer orientation
        ref = self.ref_image if (fill == "ref" and self.ref_image is not None and self.ref_image.shape == img.shape) else None
        out, cover, filled = reproject(self._to_obs(img), z, self._focal(H), C, R, self.Cn, self.Rn,
                                       inpaint=ref is None, return_mask=True)
        if ref is not None:                  # holes take the training camera's view of the scene, not invented pixels
            out[~filled] = self._to_obs(ref)[~filled]
        if return_mask:
            return self._to_obs(out), cover, self._to_obs(filled)
        return self._to_obs(out), cover

    def self_check(self, C=None, R=None, res=128):
        """VERIFY THE UNDO with the instrument that measured it (policy-free, no reference episode, no ground truth).
        Warp the robot's OBSERVED silhouette into the training camera with the same pose and depth the image correction
        uses, and compare it with where the robot's own kinematics say the silhouette must be in the training view. A
        correction that landed puts the arm where the policy expects it; one that did not, does not.
        Returns dict(precision, recall, f1): precision = warped robot pixels that land on the predicted silhouette,
        recall = predicted silhouette (inside the area the warp could fill) that is covered. Simulator sensing only."""
        if self.sensing is not None:
            return None
        C = self.C_est if C is None else C
        R = self.R_est if R is None else R
        m_obs = self._robot_mask(res)
        _, depth = self.sim.render(width=res, height=res, camera_name=self.cam, depth=True)
        m = self.sim.model
        extent = float(m.stat.extent)
        near, far = float(m.vis.map.znear) * extent, float(m.vis.map.zfar) * extent
        z = near / (1.0 - depth * (1.0 - near / far))
        rgb = np.repeat((m_obs * 255).astype(np.uint8)[..., None], 3, axis=-1)
        out, _, filled = reproject(rgb, z, self._focal(res), C, R, self.Cn, self.Rn, inpaint=False, return_mask=True)
        m_can = (out[..., 0] > 127) & filled
        s = self._save()
        try:
            self._set_camera(self.Cn, self.Rn)
            self.sim.forward()
            m_nom = self._robot_mask(res) > 0
        finally:
            self._restore(s)
        inter = float((m_can & m_nom).sum())
        prec = inter / max(float(m_can.sum()), 1.0)
        rec = inter / max(float((m_nom & filled).sum()), 1.0)
        return dict(precision=prec, recall=rec, f1=2 * prec * rec / max(prec + rec, 1e-9))

    def oracle_nominal_image(self, obs):
        """UPPER BOUND ONLY: the true render from the nominal camera."""
        if self.flip is None:
            self._obs_convention(obs)
        s = self._save()
        try:
            self._set_camera(self.Cn, self.Rn)
            self.sim.forward()
            raw = self.sim.render(width=self.res, height=self.res, camera_name=self.cam)
        finally:
            self._restore(s)
        return self._to_obs(raw)


HELD_HALO_M = 0.08                  # a held object stays within this distance of the gripper geometry


def hybrid_depth(z_plane, z_robot, mask_robot, mask_grip, static_pts, f, C, R, closed, img=None, ref=None,
                 Cn=None, Rn=None, same=30):
    """Non-robot depth for the 'hybrid' mode (see LearnedSensing). Pure function of training-setup references, the
    robot's rendered geometry at the calibrated pose, the observed frame and one proprioceptive bit. img / ref (raw
    buffer orientation) are optional: a halo pixel whose colour matches the training reference at the place the TABLE
    homography sends it is table, and stays on the plane; only what the reference cannot explain rides with the hand."""
    z = z_plane.copy()
    H, W = z.shape
    if static_pts is not None and len(static_pts):
        zs = points_depth(static_pts, f, C, R, H, W)
        z = np.where(np.isfinite(zs), zs, z)
    if closed and mask_grip is not None and mask_grip.sum() > 5:
        z_g = float(np.median(z_robot[mask_grip > 0]))
        r = int(np.clip(round(HELD_HALO_M * f / max(z_g, 0.05)), 2, H // 4))
        halo = cv2.dilate(mask_grip.astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1)))
        sel = (halo > 0) & (mask_robot == 0) & (z > z_g + 0.02)      # only what lies BEHIND the hand is pulled forward
        if img is not None and ref is not None and sel.any():
            rr, cc = np.nonzero(sel)
            zz = z_plane[rr, cc]
            pc = np.stack([(cc + 0.5 - W / 2) * zz / f, (rr + 0.5 - H / 2) * zz / f, -zz], -1)
            pn = ((C + pc @ R.T) - Cn) @ Rn
            zn = np.maximum(-pn[:, 2], 1e-6)
            c2 = np.round(f * pn[:, 0] / zn + W / 2 - 0.5).astype(int)
            r2 = np.round(f * pn[:, 1] / zn + H / 2 - 0.5).astype(int)
            inside = (r2 >= 0) & (r2 < H) & (c2 >= 0) & (c2 < W)
            a = cv2.GaussianBlur(img, (5, 5), 0).astype(np.int16)
            b = cv2.GaussianBlur(ref, (5, 5), 0).astype(np.int16)
            is_table = np.zeros(len(rr), bool)
            is_table[inside] = np.abs(a[rr[inside], cc[inside]] - b[r2[inside], c2[inside]]).max(-1) < same
            sel[rr[is_table], cc[is_table]] = False
        z[sel] = z_g
    return z


def read_static_clouds(suite, make_env, nominals, cache=None):
    """Attach `static_pts` (Canonicalizer.training_static_cloud) to every scene's nominal. Stored NEXT TO the nominal
    cache (`<cache>.cloud.npz`), never inside it, so runs that share the cache are not disturbed."""
    import os
    from plus_common import DEFAULT_VIEW, categorize
    path = (cache + ".cloud.npz") if cache else None
    have = dict(np.load(path)) if path and os.path.exists(path) else {}
    src = {}
    for i in range(suite.n_tasks):
        n = suite.get_task(i).name
        v = re.search(r"_view_(\d+)_(\d+)_(\d+)_(\d+)_(\d+)", n)
        s = re.search(r"_initstate_(\d+)", n)
        if categorize(n) != "language" or (v and v.groups() != DEFAULT_VIEW) or (s and s.group(1) != "0"):
            continue
        src.setdefault(scene_of(n), i)
    new = False
    for scene, d in nominals.items():
        if scene not in have and scene in src:
            env = make_env(src[scene])
            try:
                env.reset()
                have[scene] = Canonicalizer(env, d).training_static_cloud()
            finally:
                env.close()
            new = True
            zs = have[scene][:, 2]
            print(f"static cloud {scene}: {len(zs)} points, height {zs.min():.2f} to {zs.max():.2f} m "
                  f"(table {d.get('table_z', float('nan')):.2f})", flush=True)
        d["static_pts"] = have.get(scene)
    if path and new:
        np.savez_compressed(path, **have)
    return nominals


def fit_inverse_depth(disp, z_ref, sel, z_min=0.05, z_max=10.0):
    """Map relative disparity to metric depth, 1/z = a * disp + b, fitted on the selected pixels against reference
    metric depth. Falls back to a pure scale if fewer than 50 pixels are available."""
    inv_ref = 1.0 / np.maximum(z_ref[sel], 1e-3)
    if sel.sum() >= 50:
        A = np.stack([disp[sel], np.ones(int(sel.sum()))], 1)
        a, b = np.linalg.lstsq(A, inv_ref, rcond=None)[0]
    else:
        a, b = (float(np.median(inv_ref) / max(np.median(disp[sel]), 1e-6)) if sel.any() else 1.0), 0.0
    return 1.0 / np.clip(a * disp + b, 1.0 / z_max, 1.0 / z_min)


METRIC_DEPTH_MODELS = ("depth-anything/Depth-Anything-V2-Metric-Indoor-Small-hf", "Intel/zoedepth-nyu-kitti")


class MetricDepth:
    """Monocular metric depth in metres for an HxWx3 uint8 image. Loaded in this process when its transformers can read
    the model; otherwise (the OFT venv pins transformers 4.40, which cannot) served by depth_server.py in the system
    python, one persistent process, frames passed as .npy files in /tmp."""

    def __init__(self, device="cuda"):
        self.proc = self.server = None
        last = None
        for name in METRIC_DEPTH_MODELS:
            try:
                from transformers import AutoImageProcessor, AutoModelForDepthEstimation
                import torch
                self.proc = AutoImageProcessor.from_pretrained(name)
                self.model = AutoModelForDepthEstimation.from_pretrained(name).to(device).eval()
                self.name, self.where, self.torch, self.device = name, "in process", torch, device
                return
            except Exception as e:
                last = e
        import subprocess
        server = os.path.join(os.path.dirname(os.path.abspath(__file__)), "depth_server.py")
        py = os.environ.get("METRIC_DEPTH_PYTHON", "/usr/bin/python3")
        self.server = subprocess.Popen([py, server], stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True, bufsize=1)
        line = self.server.stdout.readline().strip()
        if not line.startswith("ready"):
            raise RuntimeError(f"no metric depth model: in process {last!r}; server said {line!r}")
        self.name, self.where = line.split(" ", 1)[1], f"server {py}"

    def __call__(self, img):
        img = np.ascontiguousarray(np.asarray(img)[..., :3]).astype(np.uint8)
        if self.server is None:
            import torch.nn.functional as F
            with self.torch.no_grad():
                inputs = self.proc(images=img, return_tensors="pt").to(self.device)
                pred = self.model(**inputs).predicted_depth
                pred = F.interpolate(pred[:, None].float(), size=img.shape[:2], mode="bicubic", align_corners=False)[0, 0]
            return pred.cpu().numpy().astype(np.float64)
        src, dst = "/tmp/_md_in.npy", "/tmp/_md_out.npy"
        np.save(src, img)
        self.server.stdin.write(src + " " + dst + chr(10))
        ans = self.server.stdout.readline().strip()
        assert ans == "ok", f"depth server: {ans}"
        return np.load(dst).astype(np.float64)


class LearnedSensing:
    """Replaces both simulator stand-ins. Robot mask: a segmenter trained on held-out tasks (train_seg.py). Depth: a
    pretrained monocular network (Depth Anything, relative disparity) mapped to metric depth on the robot's OWN pixels
    against its known geometry rendered at the estimated camera pose and current joint angles. No renderer depth or
    segmentation of the observed scene is used."""

    def __init__(self, seg_ckpt, depth_model="depth-anything/Depth-Anything-V2-Small-hf", device="cuda", depth="mono"):
        """depth: 'mono' = Depth Anything scaled on the robot's own pixels; 'plane' = no depth network: the table plane
        of the training setup (nominal cache) intersected with the calibrated camera's rays, the robot's own pixels
        taking the depth of its known geometry. 'hybrid' = plane, plus (a) the training setup's static geometry
        (fixtures, walls, floor: one depth capture of the training scene, z-buffered into the calibrated camera) and
        (b) when proprioception says the gripper is closed, the pixels around the gripper take the gripper's depth, so a
        held object travels with the hand instead of being torn off it by the table homography (2026-09-18: 12 to 55 px
        of held-object displacement under plane depth at the exact pose). All use only the learned segmenter."""
        import torch
        from train_seg import load as load_seg
        self.torch, self.device, self.depth_mode = torch, device, depth
        self.seg = load_seg(seg_ckpt, device)
        self.fine_cal = depth == "hybrid"      # 192 px refinement of the silhouette pose; off for the older arms so
        if depth == "metric":                  # POST HOC (2026-09-24): a monocular METRIC depth network, used as is
            self.depth, self.depth_name = MetricDepth(device), "metric"
            print(f"learned sensing: segmenter {seg_ckpt}, metric depth {self.depth.name} ({self.depth.where})", flush=True)
            return
        if depth in ("plane", "hybrid"):       # their rows stay comparable (cam_est_err_cm_deg is logged for both)
            self.depth, self.depth_name = None, depth
            print(f"learned sensing: segmenter {seg_ckpt}, depth from the training-setup table plane"
                  + (" + static geometry + held-object rule" if depth == "hybrid" else ""), flush=True)
            return
        from transformers import AutoImageProcessor, AutoModelForDepthEstimation
        last = None
        for name in (depth_model, "LiheYoung/depth-anything-small-hf"):
            try:
                self.proc = AutoImageProcessor.from_pretrained(name)
                self.depth = AutoModelForDepthEstimation.from_pretrained(name).to(device).eval()
                self.depth_name = name
                break
            except Exception as e:          # older transformers cannot read the V2 config: fall back to V1
                last = e
        else:
            raise RuntimeError(f"no monocular depth model could be loaded: {last}")
        print(f"learned sensing: segmenter {seg_ckpt}, depth {self.depth_name}", flush=True)

    def mask_obs(self, img):
        """Learned robot mask, cleaned for silhouette matching: the boundary chamfer that drives calibration chases
        speckle and pinholes, so keep the largest connected component and fill its holes (learned-mask calibration was
        5 to 30 cm off on a third of the camera tasks before this, 2026-09-18)."""
        from train_seg import predict
        return clean_mask(predict(self.seg, img, self.device))

    def disparity_obs(self, img):
        import torch.nn.functional as F
        with self.torch.no_grad():
            inputs = self.proc(images=np.ascontiguousarray(img), return_tensors="pt").to(self.device)
            pred = self.depth(**inputs).predicted_depth
            pred = F.interpolate(pred[:, None].float(), size=img.shape[:2], mode="bicubic", align_corners=False)[0, 0]
        return pred.cpu().numpy().astype(np.float64)

    def metric_depth(self, can, img_obs, C, R):
        H = img_obs.shape[0]
        s = can._save()
        try:
            can._set_camera(C, R)
            can.sim.forward()
            z_r, mask_r = can._depth_and_mask(H)          # the robot's OWN geometry at the calibrated pose
            grip = can._gripper_mask(H) if self.depth_mode == "hybrid" else None
        finally:
            can._restore(s)
        if self.depth_mode in ("plane", "hybrid"):
            assert can.table_z is not None, "plane depth needs table_z in the nominal cache (read_nominals upgrades it)"
            f = can._focal(H)
            z = plane_depth(H, img_obs.shape[1], f, C, R, can.table_z)
            if self.depth_mode == "hybrid":
                ref = can.ref_image if (can.ref_image is not None and can.ref_image.shape == img_obs.shape) else None
                z = hybrid_depth(z, z_r, mask_r, grip, can.static_pts, f, C, R, can.grip_closed(),
                                 img=np.ascontiguousarray(can._to_obs(img_obs)),
                                 ref=None if ref is None else np.ascontiguousarray(can._to_obs(ref)), Cn=can.Cn, Rn=can.Rn)
            return np.where(mask_r > 0, z_r, z)
        if self.depth_mode == "metric":
            # metres straight from the network, robot pixels included: no fit to the robot's rendered geometry, so the
            # calibration's point cloud is measured, not assumed at the candidate pose
            return np.clip(can._to_obs(self.depth(img_obs)), 0.05, 10.0)
        disp = can._to_obs(self.disparity_obs(img_obs))
        mask_l = can._to_obs(self.mask_obs(img_obs)) > 0
        sel = mask_l & (mask_r > 0)
        if sel.sum() < 50:
            sel = mask_r > 0
        return fit_inverse_depth(disp, z_r, sel)

    def robot_points(self, can, recs, C, R):
        out = []
        for rec in recs:
            can.sim.data.qpos[can.arm_idx] = rec[0]
            if rec[1] is not None:
                can.sim.data.qpos[can.grip_idx] = rec[1]
            can.sim.forward()
            img = rec[4]
            z = self.metric_depth(can, img, C, R)
            pts = backproject(z, can._to_obs(self.mask_obs(img)) > 0, can._focal(img.shape[0]))
            if len(pts) > 3000:
                pts = pts[np.random.default_rng(0).choice(len(pts), 3000, replace=False)]
            out.append(pts)
        return np.concatenate(out)


def pose_error(C1, R1, C2, R2):
    return float(np.linalg.norm(C1 - C2) * 100), float(np.rad2deg(np.linalg.norm(mat_to_rotvec(R1 @ R2.T))))

In [ ]:
%%writefile /content/scripts/eval_oft_wrapper.py
"""OpenVLA-OFT on LIBERO-Plus with the self-measured canonicalization wrapper. Run from the openvla-oft repo root.

  validate  a few camera tasks: self-calibration error vs ground truth, reprojection quality vs the true nominal
            render, side-by-side PNGs.
  run       arms, in PRIORITY order (so the most informative episodes finish first if compute runs out):
              full      deployable wrapper; every channel diagnoses from measurements and fires only when needed:
                        state (joint deviation), camera (calibrate_v2), photometric (robot's own pixels),
                        denoise (noise floor). Run on every category; on capability categories it is a no-harm control.
              state     state channel only (robot_init)
              cam_true  true camera pose + reprojection (upper bound, camera)
              oracle    true nominal-camera render (upper bound, camera)
              learned   learned canonicalizer (camera; needs --canon_ckpt). Checkpoints trained with --input rgb take the
                        raw frame only (no depth, no sweep); --input reproj checkpoints take the wrapper's own
                        self-calibrated reprojection + hole mask + frame (same sensing as `full`)
"""
import argparse
import gc
import json
import os
import sys
from collections import deque

import numpy as np

os.environ.setdefault("MUJOCO_GL", "egl")
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from plus_common import load_rows, wilson  # noqa: E402
import wrapper_libero as W  # noqa: E402

PRIORITY = [("camera", "full"), ("camera", "cam_true_ref"), ("robot_init", "full"), ("camera", "learned"),
            ("light", "full"), ("sensor_noise", "full"), ("layout", "full"), ("background", "full"),
            ("language", "full"), ("camera", "oracle"), ("camera", "cam_true"), ("robot_init", "state")]
PER_CAT = {"camera": 30, "robot_init": 30, "light": 10, "sensor_noise": 10, "layout": 10, "background": 10, "language": 10}
NOMINAL_VERSION = 4                # 4 = wider calibration search + training-view hole fill; rows >= 2 are kept
OLD_Q_NOM = [0.0, -0.161, 0.0, -2.4446, 0.0, 2.2268, 0.7854]
JOINT_THR = 0.05                   # rad
NOISE_MULT, NOISE_ADD = 2.0, 1.0   # denoise when the noise floor exceeds 2x the training floor + 1 gray level
PHOTO_SHIFT, PHOTO_GAIN = 15.0, 1.25


def setup(ckpt):
    from experiments.robot.libero.run_libero_eval import GenerateConfig, initialize_model
    from experiments.robot.robot_utils import get_image_resize_size, set_seed_everywhere
    cfg = GenerateConfig(pretrained_checkpoint=ckpt, task_suite_name="libero_10", center_crop=True)
    set_seed_everywhere(cfg.seed)
    parts = initialize_model(cfg)
    return cfg, parts, get_image_resize_size(cfg)


def photo_stale(r):
    """A row whose photometric map was applied before the shading gate existed (no photo_shading field)."""
    return bool(r.get("photo_applied")) and "photo_shading" not in r


def drop_stale(path, nominals, canon_tag="", learned_label="learned"):
    """Re-queue rows from older wrapper versions. A `state` row stays valid when its scene's robot start equals the
    old global one (the state channel never used the camera). A `learned` row is tied to the checkpoint that produced
    it (canon_tag), so a retrained canonicalizer re-queues them. The old file is backed up first."""
    rows = load_rows(path)
    keep = [r for r in rows if (r.get("nominal_version", 0) >= 2          # v2 = per-scene nominal, still valid
            or (r["arm"] == "state" and np.allclose(nominals[W.scene_of(r["name"])]["q_nom"], OLD_Q_NOM, atol=1e-3)))
            and (r["arm"] != learned_label or r.get("canon", "") == canon_tag)
            and not photo_stale(r)]
    if len(keep) < len(rows):
        with open(path + f".before_v{NOMINAL_VERSION}_backup", "w") as f:
            f.writelines(json.dumps(r) + "\n" for r in rows)
        with open(path, "w") as f:
            f.writelines(json.dumps(r) + "\n" for r in keep)
        print(f"re-queued {len(rows) - len(keep)} rows from older wrapper versions, kept {len(keep)}", flush=True)


def episode(cfg, parts, resize_size, env, desc, init_state, arm, nominal, canon=None, sensing=None):
    from experiments.robot.libero.libero_utils import get_libero_dummy_action
    from experiments.robot.libero.run_libero_eval import TASK_MAX_STEPS, prepare_observation, process_action
    from experiments.robot.robot_utils import get_action
    model, action_head, proprio_projector, noisy_action_projector, processor = parts
    env.reset()
    obs = env.set_init_state(init_state)
    for _ in range(cfg.num_steps_wait):
        obs, _, _, _ = env.step(get_libero_dummy_action(cfg.model_family))
    can = W.Canonicalizer(env, nominal, sensing=sensing if arm == "learned" else None)   # learned sensing: learned arm only
    Ct, Rt = can.true_camera()
    info = dict(arm=arm, joint_dev=can.joint_dev(), cam_true_shift_cm_deg=W.pose_error(Ct, Rt, can.Cn, can.Rn),
                sensing=("learned:" + sensing.depth_mode) if (sensing is not None and arm == "learned") else "renderer")
    use_cam, photo, denoise_keys = None, False, []
    if arm == "state" or (arm == "full" and info["joint_dev"] > JOINT_THR):
        p, q = can.nominal_eef()
        obs, info["home_steps"] = can.servo(obs, p, q, max_steps=200)
        info["state_applied"] = True
    if arm == "full":
        obs, recs = can.sweep(obs)
        cal = can.calibrate_v2(recs)
        info.update({f"cal_{k}": v for k, v in cal.items()})
        info["cam_est_err_cm_deg"] = W.pose_error(can.C_est, can.R_est, Ct, Rt)
        if cal["applied"]:
            use_cam = "est"
        for k in W.IMG_KEYS:
            if k in obs and k in nominal["noise"]:
                lvl = W.noise_level(obs[k])
                info[f"noise_{k}"] = lvl
                if lvl > NOISE_MULT * nominal["noise"][k] + NOISE_ADD:
                    denoise_keys.append(k)
        probe = dict(obs)
        for k in denoise_keys:
            probe[k] = W.denoise(obs[k], info[f"noise_{k}"])
        ph = can.photometric_fit(probe, nominal)
        info.update(ph)
        photo = ph["photo_ok"] and (ph["photo_mean_shift"] > PHOTO_SHIFT
                               or max(max(g, 1 / max(g, 1e-6)) for g in ph["photo_gain"]) > PHOTO_GAIN)
    elif arm in ("cam_true", "cam_true_ref", "oracle", "learned"):
        use_cam = {"cam_true": "true", "cam_true_ref": "true_ref", "oracle": "oracle", "learned": "learned"}[arm]
        if arm == "learned" and getattr(canon, "in_ch", 3) == 7:   # measured geometry in: same self-calibration as full
            obs, recs = can.sweep(obs)
            cal = can.calibrate_v2(recs)
            info.update({f"cal_{k}": v for k, v in cal.items()})
            info["cam_est_err_cm_deg"] = W.pose_error(can.C_est, can.R_est, Ct, Rt)
    info.update(cam_applied=use_cam, photo_applied=bool(photo), denoise_applied=denoise_keys)
    queue = deque(maxlen=cfg.num_open_loop_steps)
    success, covers = False, []
    for t in range(TASK_MAX_STEPS["libero_10"]):
        for k in denoise_keys:
            obs[k] = W.denoise(obs[k], info[f"noise_{k}"])
        if photo:
            for k in W.IMG_KEYS:
                if k in obs:
                    obs[k] = can.apply_photo(obs[k])
        if use_cam == "est":
            obs["agentview_image"], cv = can.canonical_image(obs)
            covers.append(cv)
        elif use_cam == "true":                          # legacy upper bound: true pose, holes inpainted
            obs["agentview_image"], cv = can.canonical_image(obs, Ct, Rt, fill="inpaint")
            covers.append(cv)
        elif use_cam == "true_ref":                      # true pose, holes from the training-view reference
            obs["agentview_image"], cv = can.canonical_image(obs, Ct, Rt, fill="ref")
            covers.append(cv)
        elif use_cam == "oracle":
            obs["agentview_image"] = can.oracle_nominal_image(obs)
        elif use_cam == "learned":
            import canon_net
            if getattr(canon, "in_ch", 3) == 7:
                rep, cv, filled = can.canonical_image(obs, return_mask=True)
                covers.append(cv)
                inp = np.concatenate([rep, (filled.astype(np.uint8) * 255)[..., None], obs["agentview_image"]], -1)
                obs["agentview_image"] = canon_net.apply(canon, inp)
            else:
                obs["agentview_image"] = canon_net.apply(canon, obs["agentview_image"])
        observation, _ = prepare_observation(obs, resize_size)
        if not queue:
            queue.extend(get_action(cfg, model, observation, desc, processor=processor, action_head=action_head,
                                    proprio_projector=proprio_projector, noisy_action_projector=noisy_action_projector,
                                    use_film=cfg.use_film))
        action = process_action(queue.popleft(), cfg.model_family)
        obs, _, done, _ = env.step(action.tolist())
        if done:
            success = True
            break
    if covers:
        info["reproj_coverage"] = float(np.mean(covers))
    return success, info


def validate(cfg, suite, sub, nominals, get_libero_env, out_dir, n):
    import cv2
    tasks = [t for t in sub["tasks"] if t["category"] == "camera"][:n]
    psnr = lambda a, b: float(10 * np.log10(255.0 ** 2 / max(np.mean((a.astype(float) - b.astype(float)) ** 2), 1e-9)))
    for t in tasks:
        env, _ = get_libero_env(suite.get_task(t["task_id"]), cfg.model_family, resolution=cfg.env_img_res)
        env.reset()
        obs = env.set_init_state(suite.get_task_init_states(t["task_id"])[0])
        for _ in range(cfg.num_steps_wait):
            obs, _, _, _ = env.step([0, 0, 0, 0, 0, 0, -1])
        can = W.Canonicalizer(env, nominals[W.scene_of(t["name"])])
        Ct, Rt = can.true_camera()
        obs, recs = can.sweep(obs)
        cal = can.calibrate_v2(recs)
        est_err = W.pose_error(can.C_est, can.R_est, Ct, Rt)
        shift = W.pose_error(Ct, Rt, can.Cn, can.Rn)
        oracle = can.oracle_nominal_image(obs)
        est, _ = can.canonical_image(obs)
        tru, cov_t = can.canonical_image(obs, Ct, Rt)
        print(f"task {t['task_id']}: true shift {shift[0]:.1f} cm / {shift[1]:.1f} deg | calibrate_v2 {cal['method']} error "
              f"{est_err[0]:.1f} cm / {est_err[1]:.1f} deg | PSNR vs true nominal render: raw "
              f"{psnr(obs['agentview_image'], oracle):.1f}, reproj(true pose) {psnr(tru, oracle):.1f}, "
              f"reproj(estimate) {psnr(est, oracle):.1f} dB | coverage {cov_t:.2f}", flush=True)
        panel = np.concatenate([obs["agentview_image"], est, tru, oracle], 1)
        cv2.imwrite(os.path.join(out_dir, f"validate_{t['task_id']}.png"), cv2.cvtColor(panel, cv2.COLOR_RGB2BGR))
        env.close(); del env; gc.collect()
    print("panels: observed | reprojected from estimate | reprojected from true pose | true nominal render")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("mode", choices=["validate", "run"])
    ap.add_argument("--subset", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--baseline", default="")
    ap.add_argument("--ckpt", default="moojink/openvla-7b-oft-finetuned-libero-10")
    ap.add_argument("--canon_ckpt", default="")
    ap.add_argument("--seg_ckpt", default="", help="learned sensing for the `learned` arm's reprojection (else renderer)")
    ap.add_argument("--depth", default="plane", choices=["plane", "mono"])
    ap.add_argument("--skip_arms", default="", help="comma-separated arms to skip, e.g. cam_true")
    ap.add_argument("--rerun_full2", action="store_true")
    ap.add_argument("--n", type=int, default=3)
    a = ap.parse_args()

    from libero.libero import benchmark
    from experiments.robot.libero.libero_utils import get_libero_env
    sub = json.load(open(a.subset))
    suite = benchmark.get_benchmark_dict()["libero_10"]()
    cfg, parts, resize_size = setup(a.ckpt)
    out_dir = os.path.dirname(a.out) or "."
    os.makedirs(out_dir, exist_ok=True)
    nominals = W.read_nominals(
        suite, lambda i: get_libero_env(suite.get_task(i), cfg.model_family, resolution=cfg.env_img_res)[0],
        cache=os.path.join(out_dir, "nominals_v3.json"))
    missing = sorted({W.scene_of(t["name"]) for t in sub["tasks"]} - set(nominals))
    assert not missing, f"no unperturbed source task for scenes {missing}"
    if a.mode == "validate":
        validate(cfg, suite, sub, nominals, get_libero_env, out_dir, a.n)
        return

    canon, canon_tag = None, ""
    if a.canon_ckpt and os.path.exists(a.canon_ckpt):
        import canon_net
        canon = canon_net.load(a.canon_ckpt)
        import torch
        ck = torch.load(a.canon_ckpt, map_location="cpu")
        canon_tag = f"{os.path.basename(a.canon_ckpt)}:{ck.get('input', 'rgb')}:{ck.get('target', 'reproj')}:{ck.get('epoch', 0)}"
        if a.seg_ckpt:
            canon_tag += f":learned-{a.depth}"
        print(f"learned canonicalizer loaded from {a.canon_ckpt} ({canon_tag})", flush=True)
    sensing = W.LearnedSensing(a.seg_ckpt, depth=a.depth) if (a.seg_ckpt and canon is not None) else None
    skip = set(filter(None, a.skip_arms.split(",")))
    if canon is None:
        skip.add("learned")
        print("no --canon_ckpt found: the learned arm is skipped", flush=True)

    learned_label = f"learned_{a.depth}" if sensing is not None else "learned"   # learned sensing: its own row label
    drop_stale(a.out, nominals, canon_tag, learned_label)
    done = {(r["task_id"], r["arm"]) for r in load_rows(a.out)}
    # v2 `full2` rows (per-scene camera + calibrate_v2, no photometric/denoise channels) count as `full` on robot_init
    # and camera, where those two channels are not expected to fire; pass --rerun_full2 to re-run them instead
    # robot_init full2 rows stand in for `full` (its state channel is unchanged). Camera and light are re-run: the camera
    # channel changed (wider calibration, training-view hole fill) and light now has the photometric channel.
    cat_of = {t["task_id"]: t["category"] for t in sub["tasks"]}
    if not a.rerun_full2:
        done |= {(tid, "full") for tid, arm in done if arm == "full2" and cat_of.get(tid) == "robot_init"}
    by_cat = {c: [t for t in sub["tasks"] if t["category"] == c][:k] for c, k in PER_CAT.items()}
    with open(a.out, "a") as f:
        for cat, arm in PRIORITY:
            if arm in skip:
                continue
            for t in by_cat[cat]:
                label = learned_label if arm == "learned" else arm
                if (t["task_id"], label) in done:
                    continue
                scene = W.scene_of(t["name"])
                env, desc = get_libero_env(suite.get_task(t["task_id"]), cfg.model_family, resolution=cfg.env_img_res)
                ok, info = episode(cfg, parts, resize_size, env, desc, suite.get_task_init_states(t["task_id"])[0], arm,
                                   nominals[scene], canon, sensing)
                env.close(); del env; gc.collect()
                info["arm"] = label
                row = dict(family="openvla_oft", **t, success=bool(ok), scene=scene, nominal_version=NOMINAL_VERSION,
                           canon=canon_tag if arm == "learned" else "", **info)
                f.write(json.dumps(row, default=float) + "\n")
                f.flush()
                print(f"{t['task_id']:5d} {cat:<12} {arm:<8} success={ok!s:<5} state={info.get('state_applied', False)!s:<5} "
                      f"cam={info['cam_applied']} photo={info['photo_applied']} denoise={len(info['denoise_applied'])} "
                      f"cam_err={np.round(info.get('cam_est_err_cm_deg', ()), 1).tolist()}", flush=True)

    rows = load_rows(a.out)
    base = {r["task_id"]: r["success"] for r in load_rows(a.baseline)} if a.baseline else {}
    print(f"\n{'category':<13}{'arm':<10}{'n':>4}{'success':>9}{'95% CI':>15}{'baseline':>10}")
    for cat, arm in PRIORITY + [("robot_init", "full2"), ("camera", "full2"), ("light", "full2")]:
        v = [r for r in rows if r["category"] == cat and r["arm"] == arm]
        if not v:
            continue
        k = sum(r["success"] for r in v)
        lo, hi = wilson(k, len(v))
        b = [base[r["task_id"]] for r in v if r["task_id"] in base]
        bs = f"{sum(b) / len(b):.3f}" if b else "-"
        print(f"{cat:<13}{arm:<10}{len(v):>4}{k / len(v):>9.3f}   [{lo:.2f}, {hi:.2f}]{bs:>10}")
    print("\ndiagnosis (full arm): fraction of episodes where each channel fired")
    for cat in PER_CAT:
        v = [r for r in rows if r["category"] == cat and r["arm"] == "full"]
        if v:
            print(f"  {cat:<13} state {np.mean([bool(r.get('state_applied')) for r in v]):.2f} | camera "
                  f"{np.mean([r.get('cam_applied') == 'est' for r in v]):.2f} | photometric "
                  f"{np.mean([bool(r.get('photo_applied')) for r in v]):.2f} | denoise "
                  f"{np.mean([bool(r.get('denoise_applied')) for r in v]):.2f}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/scripts/calib_check.py
"""Calibration-only check (no policy): camera pose error of the old and new calibration on LIBERO-Plus tasks.

Needs only the LIBERO-Plus install, so it can run beside a policy run. For every task: true camera shift, the pose
each method would feed the reprojection (nominal when its decision rule does not fire), and that pose's error.
  old  calibrate()     silhouette grid + CEM mean, fires when nominal > 1.5 px and final < 0.7 x nominal
  new  calibrate_v2()  size-aware silhouette + best tracking + depth ICP, nominal unless beaten by 0.8x
"""
import argparse
import gc
import json
import os
import sys
import time

import numpy as np

os.environ.setdefault("MUJOCO_GL", "egl")

# LIBERO stores init states as numpy pickles and loads them with a bare torch.load; torch >= 2.6 defaults to
# weights_only=True and refuses them. Every script that starts an env imports this module, so patch once here.
import torch  # noqa: E402
_torch_load = torch.load


def _load_trusted(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _torch_load(*args, **kwargs)


torch.load = _load_trusted
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import wrapper_libero as W  # noqa: E402

JOINT_THR, CAM_THR = 0.05, 1.5


def make_env(suite, tid, res=256):
    from libero.libero import get_libero_path
    from libero.libero.envs import OffScreenRenderEnv
    task = suite.get_task(tid)
    bddl = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
    env = OffScreenRenderEnv(bddl_file_name=bddl, camera_heights=res, camera_widths=res)
    env.seed(0)
    return env


def start(env, suite, tid):
    env.reset()
    obs = env.set_init_state(suite.get_task_init_states(tid)[0])
    for _ in range(10):
        obs, _, _, _ = env.step([0, 0, 0, 0, 0, 0, -1])
    return obs


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--subset", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--n_robot", type=int, default=6)
    ap.add_argument("--n_camera", type=int, default=10)
    a = ap.parse_args()

    from libero.libero import benchmark
    sub = json.load(open(a.subset))
    suite = benchmark.get_benchmark_dict()["libero_10"]()
    os.makedirs(os.path.dirname(a.out) or ".", exist_ok=True)
    nominals = W.read_nominals(suite, lambda i: make_env(suite, i),
                               cache=os.path.join(os.path.dirname(a.out) or ".", "nominals_v3.json"))
    tasks = ([t for t in sub["tasks"] if t["category"] == "robot_init"][:a.n_robot]
             + [t for t in sub["tasks"] if t["category"] == "camera"][:a.n_camera])
    done = set()
    if os.path.exists(a.out):
        done = {json.loads(l)["task_id"] for l in open(a.out) if l.strip()}
    with open(a.out, "a") as f:
        for t in tasks:
            if t["task_id"] in done:
                continue
            env = make_env(suite, t["task_id"])
            obs = start(env, suite, t["task_id"])
            can = W.Canonicalizer(env, nominals[W.scene_of(t["name"])])
            Ct, Rt = can.true_camera()
            shift = W.pose_error(Ct, Rt, can.Cn, can.Rn)
            if can.joint_dev() > JOINT_THR:
                p, q = can.nominal_eef()
                obs, _ = can.servo(obs, p, q, max_steps=200)
            obs, recs = can.sweep(obs)

            t0 = time.time()
            c1 = can.calibrate(recs)
            fire1 = c1["nominal_score"] > CAM_THR and c1["final_score"] < 0.7 * c1["nominal_score"]
            raw1 = W.pose_error(can.C_est, can.R_est, Ct, Rt)
            used1 = raw1 if fire1 else shift
            s1 = time.time() - t0

            t0 = time.time()
            c2 = can.calibrate_v2(recs)
            used2 = W.pose_error(can.C_est, can.R_est, Ct, Rt)
            s2 = time.time() - t0

            row = dict(task_id=t["task_id"], category=t["category"], true_shift=shift,
                       old=dict(raw_err=raw1, fired=bool(fire1), used_err=used1, sec=s1, **c1),
                       new=dict(used_err=used2, sec=s2, **c2))
            f.write(json.dumps(row, default=float) + "\n")
            f.flush()
            print(f"{t['task_id']:5d} {t['category']:<10} shift {shift[0]:5.1f}cm/{shift[1]:4.1f}deg | "
                  f"OLD estimate {raw1[0]:5.1f}cm/{raw1[1]:4.1f}deg fired={fire1!s:<5} used {used1[0]:5.1f}cm/{used1[1]:4.1f}deg "
                  f"({s1:.0f}s) | NEW {c2['method']:<10} used {used2[0]:5.1f}cm/{used2[1]:4.1f}deg "
                  f"score {c2['nominal_score']:.2f}->{c2['final_score']:.2f} ({s2:.0f}s)", flush=True)
            env.close()
            del env, can
            gc.collect()

    rows = [json.loads(l) for l in open(a.out) if l.strip()]
    print(f"\n{'category':<11}{'n':>3}  {'true shift':>12}  {'OLD used err':>14}  {'NEW used err':>14}  "
          f"{'OLD fired':>9}  {'NEW fired':>9}   (median cm / deg)")
    for cat in ("robot_init", "camera"):
        v = [r for r in rows if r["category"] == cat]
        if not v:
            continue
        med = lambda key: (np.median([k[0] for k in key]), np.median([k[1] for k in key]))
        ts, o, n = med([r["true_shift"] for r in v]), med([r["old"]["used_err"] for r in v]), med([r["new"]["used_err"] for r in v])
        print(f"{cat:<11}{len(v):>3}  {ts[0]:5.1f}/{ts[1]:5.1f}   {o[0]:6.1f}/{o[1]:5.1f}   {n[0]:6.1f}/{n[1]:5.1f}   "
              f"{np.mean([r['old']['fired'] for r in v]):9.2f}  {np.mean([r['new']['applied'] for r in v]):9.2f}")
    print("go bar for the camera channel: NEW used error on camera tasks well below the true shift "
          "(target <= 3 cm / 3 deg), and robot_init (nominal camera) NEW fired ~0")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/scripts/canon_net.py
"""Learned canonicalizer: an RGB frame from an unknown camera -> the same scene seen from the nominal (training)
camera. Warp + fill: the network predicts a dense sampling field for the input image, a fill image, and a blend
weight, so large viewpoint changes are a geometric resampling rather than something a U-Net must copy through skips.

Two input modes (in_ch):
  3   the raw RGB frame only: the network must infer the camera pose from pixels and warp accordingly. Learning
      alone; on held-out poses this reached +1.4 dB (2026-09-16).
  7   the frame REPROJECTED by the wrapper's own self-calibration (3) + its hole mask (1) + the raw frame (3): the
      measured geometry is given, the network completes holes and repairs resampling artefacts. Measurement plus
      learning. The warp head is then a small residual (max_flow 0.15)."""
import torch
import torch.nn as nn
import torch.nn.functional as F


def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.GroupNorm(8, o), nn.SiLU(),
                         nn.Conv2d(o, o, 3, padding=1), nn.GroupNorm(8, o), nn.SiLU())


class CanonNet(nn.Module):
    def __init__(self, width=32, max_flow=0.6, in_ch=3):
        super().__init__()
        ch = [width, 2 * width, 4 * width, 8 * width, 8 * width]
        self.max_flow, self.in_ch = max_flow, in_ch
        self.enc = nn.ModuleList()
        prev = in_ch + 2                                           # input channels + 2 coordinate channels
        for c in ch:
            self.enc.append(block(prev, c))
            prev = c
        self.dec = nn.ModuleList()
        for c in reversed(ch[:-1]):
            self.dec.append(block(prev + c, c))
            prev = c
        self.head = nn.Conv2d(prev, 6, 1)                          # flow (2), fill RGB (3), blend (1)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)
        with torch.no_grad():
            self.head.bias[5] = 4.0                                # start as identity: blend fully toward the warp

    def forward(self, x):
        """x: (B, in_ch, H, W) in [0, 1]; the first 3 channels are the image that gets warped. Returns (image in
        [0, 1], blend weight)."""
        B, _, H, W = x.shape
        yy, xx = torch.meshgrid(torch.linspace(-1, 1, H, device=x.device, dtype=x.dtype),
                                torch.linspace(-1, 1, W, device=x.device, dtype=x.dtype), indexing="ij")
        h = torch.cat([x, xx.expand(B, 1, H, W), yy.expand(B, 1, H, W)], 1)
        skips = []
        for i, b in enumerate(self.enc):
            h = b(h)
            if i < len(self.enc) - 1:
                skips.append(h)
                h = F.max_pool2d(h, 2)
        for b, s in zip(self.dec, reversed(skips)):
            h = F.interpolate(h, size=s.shape[-2:], mode="bilinear", align_corners=False)
            h = b(torch.cat([h, s], 1))
        out = self.head(h)
        flow = torch.tanh(out[:, :2]) * self.max_flow
        grid = torch.stack([xx, yy], -1).expand(B, H, W, 2) + flow.permute(0, 2, 3, 1)
        warped = F.grid_sample(x[:, :3], grid, mode="bilinear", padding_mode="border", align_corners=True)
        fill = torch.sigmoid(out[:, 2:5])
        alpha = torch.sigmoid(out[:, 5:6])
        return alpha * warped + (1 - alpha) * fill, alpha


def load(path, device="cuda"):
    ck = torch.load(path, map_location="cpu")
    net = CanonNet(**ck.get("net_args", {}))
    net.load_state_dict(ck["state_dict"])
    return net.to(device).eval()


@torch.no_grad()
def apply(net, img_uint8, device="cuda"):
    """uint8 HxWxC numpy (C = net.in_ch) -> uint8 HxWx3 numpy."""
    import numpy as np
    x = torch.from_numpy(np.ascontiguousarray(img_uint8)).to(device).permute(2, 0, 1)[None].float() / 255.0
    y, _ = net(x)
    return (y[0].permute(1, 2, 0).clamp(0, 1) * 255.0).round().byte().cpu().numpy()

In [ ]:
%%writefile /content/scripts/collect_canon.py
"""Self-labelled training pairs for the learned canonicalizer (claim 3).

Held-out LIBERO-Plus tasks only (camera-perturbed and language tasks that are NOT in the evaluation subset). Per task:
self-calibrate the camera with calibrate_v2 (the same measurement the wrapper uses), move the arm to scripted random
end-effector targets, and store (observed image, observed image reprojected into the nominal camera, validity mask,
the TRUE nominal-camera render). The self-labelled reprojection needs no privileged information. The oracle render is
a training-time-only target for `train_canon.py --target oracle`: the network trained on it still takes only the
observed RGB frame at deployment, so nothing privileged crosses into the wrapper. The true pose is saved only for
reporting.

With --seg_ckpt (and --depth plane|mono) the reprojection and its hole mask come from LEARNED sensing (trained
segmenter, table-plane or monocular depth), so a canonicalizer trained on these shards learns to repair the artefacts
of the deployable sensing rather than of the renderer's. Segmenter labels (`r`) always come from the renderer.

Resumable: one .npz per task. Run several workers in parallel with --worker i --workers k.
"""
import argparse
import json
import os
import sys

import numpy as np

os.environ.setdefault("MUJOCO_GL", "egl")
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import wrapper_libero as W  # noqa: E402
from calib_check import make_env, start  # noqa: E402
from plus_common import categorize  # noqa: E402


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--subset", required=True)
    ap.add_argument("--nominals", required=True, help="nominals_v3.json cache (built on first use)")
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--n_camera", type=int, default=120)
    ap.add_argument("--n_language", type=int, default=30)
    ap.add_argument("--frames", type=int, default=12)
    ap.add_argument("--worker", type=int, default=0)
    ap.add_argument("--workers", type=int, default=1)
    ap.add_argument("--seg_ckpt", default="", help="learned sensing for the reprojection (train_seg.py output)")
    ap.add_argument("--depth", default="plane", choices=["plane", "mono"])
    a = ap.parse_args()

    from libero.libero import benchmark
    suite = benchmark.get_benchmark_dict()["libero_10"]()
    sub = json.load(open(a.subset))
    held_in = {t["task_id"] for t in sub["tasks"]}
    small = a.subset.replace("subset_libero10.json", "subset_libero10_small.json")
    if os.path.exists(small):
        held_in |= {t["task_id"] for t in json.load(open(small))["tasks"]}
    rng = np.random.default_rng(0)
    pool = {"camera": [], "language": []}
    for i in range(suite.n_tasks):
        c = categorize(suite.get_task(i).name)
        if c in pool and i not in held_in:
            pool[c].append(i)
    ids = sorted(rng.permutation(pool["camera"])[:a.n_camera].tolist()
                 + rng.permutation(pool["language"])[:a.n_language].tolist())
    ids = ids[a.worker::a.workers]
    os.makedirs(a.out_dir, exist_ok=True)
    nominals = W.read_nominals(suite, lambda i: make_env(suite, i), cache=a.nominals)
    sensing = W.LearnedSensing(a.seg_ckpt, depth=a.depth) if a.seg_ckpt else None
    print(f"worker {a.worker}/{a.workers}: {len(ids)} held-out tasks (none in the evaluation subsets); sensing "
          f"{'learned, ' + a.depth + ' depth' if sensing else 'renderer'}", flush=True)

    for tid in ids:
        path = os.path.join(a.out_dir, f"task_{tid:05d}.npz")
        if os.path.exists(path):
            continue
        name = suite.get_task(tid).name
        env = make_env(suite, tid)
        try:
            obs = start(env, suite, tid)
            can = W.Canonicalizer(env, nominals[W.scene_of(name)], sensing=sensing)
            if can.joint_dev() > 0.05:
                p, q = can.nominal_eef()
                obs, _ = can.servo(obs, p, q, max_steps=200)
            obs, recs = can.sweep(obs)
            cal = can.calibrate_v2(recs)
            Ct, Rt = can.true_camera()
            p0, q0 = can.nominal_eef()
            r = np.random.default_rng(tid)
            X, Y, M, RM, O = [], [], [], [], []
            for _ in range(a.frames):
                obs, _ = can.servo(obs, p0 + r.uniform([-0.12, -0.15, -0.08], [0.12, 0.15, 0.08]), q0, max_steps=40)
                tgt, _, filled = can.canonical_image(obs, return_mask=True)
                X.append(obs["agentview_image"].copy())
                Y.append(tgt)
                M.append(filled)
                RM.append(can._to_obs(can._robot_mask(obs["agentview_image"].shape[0])))   # labels: renderer, always
                O.append(can.oracle_nominal_image(obs))         # privileged target, training time only
            meta = dict(task_id=tid, name=name, category=categorize(name), scene=W.scene_of(name),
                        sensing=("learned:" + a.depth) if sensing else "renderer",
                        calib=dict(cal), est_err_cm_deg=W.pose_error(can.C_est, can.R_est, Ct, Rt),
                        true_shift_cm_deg=W.pose_error(Ct, Rt, can.Cn, can.Rn))
            np.savez_compressed(path + ".tmp.npz", x=np.stack(X), y=np.stack(Y), m=np.stack(M), r=np.stack(RM), o=np.stack(O),
                                meta=json.dumps(meta, default=float))
            os.replace(path + ".tmp.npz", path)
            print(f"{tid:5d} {meta['category']:<9} shift {meta['true_shift_cm_deg'][0]:5.1f}cm | calib {cal['method']:<10}"
                  f" err {meta['est_err_cm_deg'][0]:4.1f}cm/{meta['est_err_cm_deg'][1]:4.1f}deg | "
                  f"mask {np.mean(M):.2f}", flush=True)
        finally:
            env.close()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/scripts/train_canon.py
"""Train the learned canonicalizer (claim 3) on the self-labelled pairs from collect_canon.py.

Targets (--target):
  reproj   the observation reprojected with the wrapper's own camera estimate (self-labelled). The network then
           distills the geometric pipeline into one RGB-only pass; its ceiling is the `full` arm.
  oracle   the true nominal-camera render stored by collect_canon.py (privileged at training time only). Every pixel
           is supervised, so the network can fill what reprojection cannot see; this is the arm that can beat `full`.
Inputs (--input):
  rgb      the raw frame only (learning alone; the 2026-09-16 result)
  reproj   the wrapper's reprojection + hole mask + raw frame (7 channels): measurement plus learning. The network
           only has to complete and repair, which is what the reprojection gets wrong.
Loss: L1 on observed pixels (weight 1) and on holes (weight --hole_w; every pixel counts as observed for oracle), plus
a feature loss in a frozen DINOv2-small on patches that are mostly observed. Split by task, so validation scenes and
camera poses are unseen. Reports PSNR against the chosen target for the network and for the do-nothing baseline.
"""
import argparse
import glob
import json
import os
import sys

import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from canon_net import CanonNet  # noqa: E402


def net_input(d, mode):
    """uint8 HxWxC network input from a shard: the raw frame, or reprojection + hole mask + raw frame."""
    if mode == "rgb":
        return d["x"]
    hole = (d["m"].astype(np.uint8) * 255)[..., None]
    return np.concatenate([d["y"], hole, d["x"]], -1)


def load_pairs(data_dir, target="reproj", mode="rgb"):
    X, Y, M, T, META = [], [], [], [], []
    for p in sorted(glob.glob(os.path.join(data_dir, "task_*.npz"))):
        d = np.load(p)
        meta = json.loads(str(d["meta"]))
        if target == "oracle":
            if "o" not in d.files:
                sys.exit(f"{p} has no oracle render: re-run collect_canon.py into a fresh --out_dir (shards from "
                         f"before 2026-09-16 carry only the reprojection target)")
            X.append(net_input(d, mode)); Y.append(d["o"]); M.append(np.ones(d["m"].shape, bool))
        else:
            X.append(net_input(d, mode)); Y.append(d["y"]); M.append(d["m"])
        T += [meta["task_id"]] * len(d["x"])
        META.append(meta)
    assert X, f"no task_*.npz in {data_dir}"
    return np.concatenate(X), np.concatenate(Y), np.concatenate(M), np.array(T), META


def to_t(a, dev):
    return torch.from_numpy(a).to(dev).permute(0, 3, 1, 2).float() / 255.0


def masked_psnr(pred, y, m):
    mse = ((pred - y) ** 2 * m).sum() / (m.sum() * 3 + 1e-6)
    return float(10 * torch.log10(1.0 / (mse + 1e-10)))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--epochs", type=int, default=30)
    ap.add_argument("--bs", type=int, default=16)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--hole_w", type=float, default=0.1)
    ap.add_argument("--feat_w", type=float, default=0.2)
    ap.add_argument("--val_frac", type=float, default=0.15)
    ap.add_argument("--target", default="reproj", choices=["reproj", "oracle"])
    ap.add_argument("--input", default="rgb", choices=["rgb", "reproj"])
    a = ap.parse_args()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0)

    X, Y, M, T, META = load_pairs(a.data_dir, a.target, a.input)
    net_args = dict(in_ch=X.shape[-1], max_flow=0.15 if a.input == "reproj" else 0.6)
    tasks = np.unique(T)
    rng = np.random.default_rng(0)
    val_tasks = set(rng.permutation(tasks)[:max(1, int(a.val_frac * len(tasks)))].tolist())
    va = np.array([t in val_tasks for t in T])
    cat = {m["task_id"]: m["category"] for m in META}
    print(f"target {a.target}, input {a.input} ({X.shape[-1]} ch): {len(X)} pairs from {len(tasks)} tasks; "
          f"train {int((~va).sum())} / val {int(va.sum())} "
          f"(val tasks {len(val_tasks)}); mean observed fraction {M.mean():.2f}", flush=True)

    net = CanonNet(**net_args).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=a.lr, weight_decay=1e-4)
    steps = a.epochs * max(1, int((~va).sum()) // a.bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=a.lr, total_steps=steps, pct_start=0.05)
    dino = None
    if a.feat_w > 0:
        from transformers import AutoModel
        dino = AutoModel.from_pretrained("facebook/dinov2-small").to(dev).eval()
        for p in dino.parameters():
            p.requires_grad_(False)
        mean = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)

    def feats(img):
        z = F.interpolate(img, size=(224, 224), mode="bilinear", align_corners=False)
        return dino(pixel_values=(z - mean) / std).last_hidden_state[:, 1:]          # (B, 256, D)

    tr_idx = np.where(~va)[0]
    best = -1.0
    for ep in range(a.epochs):
        net.train()
        rng.shuffle(tr_idx)
        tot = 0.0
        for s in range(0, len(tr_idx) - a.bs + 1, a.bs):
            b = tr_idx[s:s + a.bs]
            x, y = to_t(X[b], dev), to_t(Y[b], dev)
            m = torch.from_numpy(M[b]).to(dev).float()[:, None]
            with torch.autocast(dev, dtype=torch.bfloat16, enabled=dev == "cuda"):
                pred, _ = net(x)
                pred = pred.float()
                w = m + a.hole_w * (1 - m)
                loss = (torch.abs(pred - y) * w).sum() / (w.sum() * 3)
                if dino is not None:
                    fp, fy = feats(pred), feats(y)
                    pm = (F.adaptive_avg_pool2d(m, 16).flatten(1) > 0.9).float()
                    cos = 1 - F.cosine_similarity(fp.float(), fy.float(), dim=-1)
                    loss = loss + a.feat_w * (cos * pm).sum() / (pm.sum() + 1e-6)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
            sched.step()
            tot += float(loss.detach())
        net.eval()
        res = {"camera": [[], []], "language": [[], []]}
        with torch.no_grad():
            for i in np.where(va)[0]:
                x, y = to_t(X[i:i + 1], dev), to_t(Y[i:i + 1], dev)
                m = torch.from_numpy(M[i:i + 1]).to(dev).float()[:, None]
                pred, _ = net(x)
                c = cat[int(T[i])]
                res[c][0].append(masked_psnr(pred, y, m))
                res[c][1].append(masked_psnr(x[:, :3], y, m))     # do-nothing baseline: the frame the net starts from
        line = " | ".join(f"{c}: net {np.mean(v[0]):.1f} dB vs raw {np.mean(v[1]):.1f} dB" for c, v in res.items() if v[0])
        print(f"epoch {ep + 1:3d} loss {tot / max(1, len(tr_idx) // a.bs):.4f} | val PSNR vs {a.target} target: {line}", flush=True)
        score = np.mean(res["camera"][0]) if res["camera"][0] else -np.mean([0])
        if score > best:
            best = score
            os.makedirs(os.path.dirname(a.out) or ".", exist_ok=True)
            torch.save(dict(state_dict=net.state_dict(), net_args=net_args, val_tasks=sorted(val_tasks), epoch=ep + 1,
                            val_camera_psnr=float(score), target=a.target, input=a.input), a.out)
    print(f"best val camera PSNR {best:.1f} dB -> {a.out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/scripts/invertibility_report.py
"""Claims 1 and 2: how much of the LIBERO-Plus robustness gap a FROZEN policy recovers from measurements of its own
body, per perturbation class, with paired per-task statistics.

  python invertibility_report.py --family openvla_oft:rows_oft.jsonl:wrapper_oft/rows.jsonl --out report.json

Each --family is NAME:BASELINE_ROWS:WRAPPER_ROWS. Rows pair by task_id. The wrapper arm `full` is the deployable
method; `oracle`, `cam_true` and `learned` are reported next to it when present.
"""
import argparse
import json
import math
import os
import sys
from collections import defaultdict

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from plus_common import load_rows, wilson  # noqa: E402

TAXONOMY = {
    "robot_init": ("invertible", "proprioception: joint angles vs the training start"),
    "camera": ("invertible", "self-calibration: robot silhouette + depth"),
    "light": ("partly invertible", "photometric gain fitted on the robot's own pixels"),
    "sensor_noise": ("partly invertible", "noise floor estimate + denoising"),
    "layout": ("non-invertible", "new object configuration: a capability test"),
    "background": ("non-invertible", "new scene appearance: a capability test"),
    "language": ("non-invertible", "new instruction: a capability test"),
}
ORDER = ["robot_init", "camera", "light", "sensor_noise", "layout", "background", "language"]


def mcnemar_p(b, c):
    """Exact two-sided McNemar p-value from the discordant counts b (base fail -> wrapped success) and c."""
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    tail = sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n
    return min(1.0, 2 * tail)


def family_report(name, base_path, wrap_path):
    base = {r["task_id"]: bool(r["success"]) for r in load_rows(base_path) if r.get("arm", "raw") == "raw"}
    arms = defaultdict(dict)
    cats = {}
    for r in load_rows(wrap_path):
        arms[r["arm"]][r["task_id"]] = bool(r["success"])
        cats[r["task_id"]] = r["category"]
    for tid, ok in arms.get("full2", {}).items():          # v2 rows stand in for `full` where no `full` row exists
        arms["full"].setdefault(tid, ok)
    out = {"family": name, "categories": {}, "classes": {}}
    print(f"\n== {name} ==")
    print(f"{'category':<13}{'class':<19}{'n':>3}{'base':>7}{'full':>7}{'delta':>7}{'recovered':>11}{'harmed':>8}"
          f"{'p':>8}   upper bounds")
    agg = defaultdict(lambda: [0, 0, 0, 0])            # class -> [base fails, recovered, base successes, harmed]
    for cat in ORDER:
        ids = [t for t, c in cats.items() if c == cat and t in base and t in arms.get("full", {})]
        if not ids:
            continue
        b0 = [base[t] for t in ids]
        f1 = [arms["full"][t] for t in ids]
        rec = sum(1 for x, y in zip(b0, f1) if not x and y)
        harm = sum(1 for x, y in zip(b0, f1) if x and not y)
        fails, succ = b0.count(False), b0.count(True)
        cls = TAXONOMY[cat][0]
        a = agg[cls]
        a[0] += fails; a[1] += rec; a[2] += succ; a[3] += harm
        ub = {}
        for arm in ("oracle", "cam_true", "learned", "learned_plane", "full_learned", "full_plane"):
            v = [arms[arm][t] for t in ids if t in arms.get(arm, {})]
            if v:
                ub[arm] = round(sum(v) / len(v), 3)
        entry = dict(cls=cls, n=len(ids), base=sum(b0) / len(ids), full=sum(f1) / len(ids),
                     full_ci=wilson(sum(f1), len(ids)), recovered=rec, base_fails=fails, harmed=harm,
                     base_successes=succ, mcnemar_p=mcnemar_p(rec, harm), upper_bounds=ub)
        out["categories"][cat] = entry
        print(f"{cat:<13}{cls:<19}{len(ids):>3}{entry['base']:>7.2f}{entry['full']:>7.2f}"
              f"{entry['full'] - entry['base']:>+7.2f}{f'{rec}/{fails}':>11}{f'{harm}/{succ}':>8}"
              f"{entry['mcnemar_p']:>8.3f}   {ub if ub else ''}")
    print("\nper class: share of baseline failures the wrapper recovers, and share of baseline successes it breaks")
    for cls in ("invertible", "partly invertible", "non-invertible"):
        f, r, s, h = agg[cls]
        if f + s == 0:
            continue
        out["classes"][cls] = dict(recovered=r, base_fails=f, harmed=h, base_successes=s)
        print(f"  {cls:<19} recovered {r}/{f} ({r / max(1, f):.0%})   harmed {h}/{s} ({h / max(1, s):.0%})")
    inv = [c for c in out["categories"] if TAXONOMY[c][0] != "non-invertible"]
    non = [c for c in out["categories"] if TAXONOMY[c][0] == "non-invertible"]
    if inv and non:
        gap_b = sum(1 - out["categories"][c]["base"] for c in inv) / len(inv)
        gap_f = sum(1 - out["categories"][c]["full"] for c in inv) / len(inv)
        out["invertible_gap_closed"] = (gap_b - gap_f) / gap_b if gap_b > 0 else None
        out["non_invertible_success"] = sum(out["categories"][c]["full"] for c in non) / len(non)
        print(f"\ninvertible-class failure rate {gap_b:.2f} -> {gap_f:.2f} "
              f"(gap closed {out['invertible_gap_closed']:.0%}); capability-class success after wrapping "
              f"{out['non_invertible_success']:.2f}")
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--family", action="append", required=True, help="NAME:BASELINE_ROWS:WRAPPER_ROWS")
    ap.add_argument("--out", required=True)
    a = ap.parse_args()
    reports = []
    for spec in a.family:
        name, base_path, wrap_path = spec.split(":", 2)
        reports.append(family_report(name, base_path, wrap_path))
    json.dump(dict(taxonomy=TAXONOMY, families=reports), open(a.out, "w"), indent=1, default=float)
    print(f"\nwrote {a.out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/scripts/identifiability.py
"""Self-identifiability of LIBERO-Plus perturbations: can the robot measure a perturbation from its own body?
No policy is run.

  local  per scene: finite-difference sensitivity of the robot's SELF-observation (silhouette, body colour,
         proprioception, image noise floor) to each perturbation family's parameters, and the singular values of
         the camera block with 1 vs 3 sweep poses (how well-conditioned self-calibration is).
  tasks  per task in the evaluation subset: the self-measured shift on every channel and an identifiability index
         (>1 means at least one channel sees the perturbation above its noise floor). With --wrapper and --baseline,
         tests whether identifiability predicts which baseline failures the wrapper recovers.
"""
import argparse
import json
import os
import sys
from collections import defaultdict

import numpy as np

os.environ.setdefault("MUJOCO_GL", "egl")
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import wrapper_libero as W  # noqa: E402
from plus_common import load_rows  # noqa: E402

FLOOR = dict(silhouette_px=0.5, body_colour=2.0, proprio_rad=0.005, noise=1.0)   # per-channel noise floors
TASK_THR = dict(joint=0.05, silhouette=1.5, photo_shift=15.0, photo_gain=0.25)


def spearman(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    if len(x) < 3 or np.ptp(x) == 0 or np.ptp(y) == 0:
        return float("nan")
    rx, ry = np.argsort(np.argsort(x)), np.argsort(np.argsort(y))
    return float(np.corrcoef(rx, ry)[0, 1])


# ------------------------------------------------------------------ local sensitivity
def self_obs(can, recs, res=96, arm_offset=None):
    import cv2
    masks, cols, blur = [], [], []
    for rec in recs:
        q = rec[0] if arm_offset is None else rec[0] + arm_offset
        can.sim.data.qpos[can.arm_idx] = q
        if rec[1] is not None:
            can.sim.data.qpos[can.grip_idx] = rec[1]
        can.sim.forward()
        mk = can._robot_mask(res)
        rgb = can.sim.render(width=res, height=res, camera_name=can.cam)
        px = rgb[mk > 0].reshape(-1, 3).astype(np.float64)
        masks.append(mk)
        cols.append(px.mean(0) if len(px) else np.zeros(3))
        blur.append(cv2.GaussianBlur(mk.astype(np.float32), (0, 0), 2.0).ravel())
    joints = np.array([rec[0] if arm_offset is None else rec[0] + arm_offset for rec in recs])
    return masks, np.array(cols), joints, np.concatenate(blur)


def compare(base, pert):
    m0, c0, j0, b0 = base
    m1, c1, j1, b1 = pert
    sil = float(np.mean([W.chamfer(a, b) if (a.any() and b.any()) else 0.0 for a, b in zip(m0, m1)]))
    return dict(silhouette_px=sil, body_colour=float(np.abs(c1 - c0).max()),
                proprio_rad=float(np.abs(j1 - j0).max())), b1 - b0


def local_scene(env, obs, nominal):
    can = W.Canonicalizer(env, nominal)
    obs, recs = can.sweep(obs)
    m, sim = can.sim.model, can.sim
    s = can._save()
    light0, rgba0 = m.light_diffuse.copy(), m.geom_rgba.copy()
    C0, R0 = can.true_camera()

    def undo():
        m.light_diffuse[:] = light0
        m.geom_rgba[:] = rgba0
        can._restore(s)

    base = self_obs(can, recs)
    fam = defaultdict(list)          # family -> list of (param, unit, channels)
    cam_cols = {1: [], 3: []}

    def record(family, param, unit, pert, jac=False):
        ch, dvec = compare(base, pert)
        fam[family].append(dict(param=param, unit=unit, **ch))
        if jac:
            per = len(dvec) // len(recs)
            cam_cols[3].append(dvec)
            cam_cols[1].append(dvec[:per])

    for i, ax in enumerate("xyz"):
        d = np.zeros(3); d[i] = 0.01
        can._set_camera(C0 + d, R0)
        record("camera", f"pos_{ax}", "1 cm", self_obs(can, recs), jac=True)
        undo()
    for i, ax in enumerate("xyz"):
        r = np.zeros(3); r[i] = np.deg2rad(1.0)
        can._set_camera(C0, W.rotvec_to_mat(r) @ R0)
        record("camera", f"rot_{ax}", "1 deg", self_obs(can, recs), jac=True)
        undo()
    if m.nlight > 0:
        m.light_diffuse[:] = np.clip(light0 * 1.1, 0, None)
        record("light", "diffuse", "+10%", self_obs(can, recs))
        undo()
    for k in range(len(can.arm_idx)):
        off = np.zeros(len(can.arm_idx)); off[k] = 0.02
        record("robot_init", f"joint_{k}", "0.02 rad", self_obs(can, recs, arm_offset=off))
        undo()
    robot_q = set(int(sim.model.jnt_qposadr[j]) for j in range(sim.model.njnt)
                  if can._body_name(int(sim.model.jnt_bodyid[j])).startswith(W.ROBOT_PREFIXES))
    free = [int(sim.model.jnt_qposadr[j]) for j in range(sim.model.njnt)
            if int(sim.model.jnt_type[j]) == 0 and int(sim.model.jnt_qposadr[j]) not in robot_q]
    for adr in free[:4]:
        for ax in range(2):
            sim.data.qpos[adr + ax] += 0.02
            sim.forward()
            record("layout", f"object_qpos{adr}_{'xy'[ax]}", "2 cm", self_obs(can, recs))
            undo()
    table = [g for g in range(m.ngeom) if "table" in (m.geom_id2name(g) or "").lower()]
    if table:
        m.geom_rgba[table, :3] = np.clip(rgba0[table, :3] + 0.1, 0, 1)
        record("background", "table_rgb", "+0.1", self_obs(can, recs))
        undo()
    img = obs["agentview_image"]
    lvl0 = W.noise_level(img)
    noisy = np.clip(img + np.random.default_rng(0).normal(0, 5, img.shape), 0, 255).astype(np.uint8)
    fam["sensor_noise"].append(dict(param="gaussian", unit="sigma 5", noise=W.noise_level(noisy) - lvl0))
    undo()

    sv = {}
    for n, cols in cam_cols.items():
        if cols:
            svals = np.linalg.svd(np.stack(cols, 1), compute_uv=False)
            sv[f"{n}_pose"] = dict(singular_values=svals.tolist(), condition=float(svals[0] / max(svals[-1], 1e-9)))
    return dict(families=fam, camera_svd=sv)


def identifiable(entry):
    return any(entry.get(k, 0.0) > f for k, f in FLOOR.items())


# ------------------------------------------------------------------ per-task self-measured shift
def task_signals(env, obs, nominal):
    can = W.Canonicalizer(env, nominal)
    jd = can.joint_dev()
    if jd > TASK_THR["joint"]:
        p, q = can.nominal_eef()
        obs, _ = can.servo(obs, p, q, max_steps=200)
    obs, recs = can.sweep(obs)
    dts = [(W.edge_dt(rec[2]), rec[2]) for rec in recs]
    s = can._save()
    try:
        sil = can._score(recs, dts, can.Cn, can.Rn)
    finally:
        can._restore(s)
    ph = can.photometric_fit(obs, nominal)
    nz = W.noise_level(obs["agentview_image"])
    ref = nominal["noise"]["agentview_image"]
    gain_dev = max(abs(g - 1.0) for g in ph["photo_gain"])
    parts = dict(joint=jd / TASK_THR["joint"], silhouette=sil / TASK_THR["silhouette"],
                 photometric=max(ph["photo_mean_shift"] / TASK_THR["photo_shift"], gain_dev / TASK_THR["photo_gain"]),
                 noise=nz / (2.0 * ref + 1.0))
    return dict(joint_dev=jd, silhouette_px=sil, photo_mean_shift=ph["photo_mean_shift"], photo_gain_dev=gain_dev,
                noise_level=nz, index_parts=parts, index=max(parts.values()))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("mode", choices=["local", "tasks"])
    ap.add_argument("--subset", required=True)
    ap.add_argument("--nominals", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--per_cat", type=int, default=30)
    ap.add_argument("--wrapper", default="")
    ap.add_argument("--baseline", default="")
    a = ap.parse_args()

    from libero.libero import benchmark
    from calib_check import make_env, start
    from plus_common import categorize
    suite = benchmark.get_benchmark_dict()["libero_10"]()
    sub = json.load(open(a.subset))
    nominals = W.read_nominals(suite, lambda i: make_env(suite, i), cache=a.nominals)
    os.makedirs(os.path.dirname(a.out) or ".", exist_ok=True)

    if a.mode == "local":
        res = {}
        for i in range(suite.n_tasks):
            name = suite.get_task(i).name
            scene = W.scene_of(name)
            if scene in res or categorize(name) != "language":
                continue
            env = make_env(suite, i)
            try:
                res[scene] = local_scene(env, start(env, suite, i), nominals[scene])
            finally:
                env.close()
            print(f"scene {scene} done", flush=True)
        json.dump(res, open(a.out, "w"), indent=1, default=float)
        print(f"\n{'family':<13}{'identifiable':>13}  median per-unit sensitivity across scenes and parameters")
        for f in ("robot_init", "camera", "light", "sensor_noise", "layout", "background"):
            ent = [e for r in res.values() for e in r["families"].get(f, [])]
            if not ent:
                continue
            frac = np.mean([identifiable(e) for e in ent])
            med = {k: round(float(np.median([e.get(k, 0.0) for e in ent])), 3) for k in FLOOR}
            print(f"{f:<13}{frac:>13.2f}  {med}")
        print(f"{'language':<13}{0.0:>13.2f}  no physical parameter: zero coupling to the self-observation by construction")
        for scene, r in res.items():
            sv = r["camera_svd"]
            if sv:
                print(f"camera self-calibration conditioning {scene}: 1 pose {sv['1_pose']['condition']:.1f}, "
                      f"3 poses {sv['3_pose']['condition']:.1f}")
        return

    done = {}
    if os.path.exists(a.out):
        done = {r["task_id"]: r for r in load_rows(a.out)}
    tasks = [t for c in ("robot_init", "camera", "light", "sensor_noise", "layout", "background", "language")
             for t in [x for x in sub["tasks"] if x["category"] == c][:a.per_cat]]
    with open(a.out, "a") as f:
        for t in tasks:
            if t["task_id"] in done:
                continue
            env = make_env(suite, t["task_id"])
            try:
                sig = task_signals(env, start(env, suite, t["task_id"]), nominals[W.scene_of(t["name"])])
            finally:
                env.close()
            row = dict(task_id=t["task_id"], category=t["category"], **sig)
            f.write(json.dumps(row, default=float) + "\n")
            f.flush()
            print(f"{t['task_id']:5d} {t['category']:<13} index {sig['index']:6.2f} "
                  f"{ {k: round(v, 2) for k, v in sig['index_parts'].items()} }", flush=True)
    rows = load_rows(a.out)
    print(f"\n{'category':<13}{'n':>4}{'identifiable':>13}{'median index':>14}")
    for c in ("robot_init", "camera", "light", "sensor_noise", "layout", "background", "language"):
        v = [r for r in rows if r["category"] == c]
        if v:
            print(f"{c:<13}{len(v):>4}{np.mean([r['index'] > 1 for r in v]):>13.2f}"
                  f"{np.median([r['index'] for r in v]):>14.2f}")
    if a.wrapper and a.baseline:
        base = {r["task_id"]: bool(r["success"]) for r in load_rows(a.baseline)}
        full = {}
        for r in load_rows(a.wrapper):
            if r["arm"] in ("full", "full2"):
                full.setdefault(r["task_id"], bool(r["success"]))
        idx = {r["task_id"]: r["index"] for r in rows}
        cats, ident, recov = [], [], []
        for c in ("robot_init", "camera", "light", "sensor_noise", "layout", "background", "language"):
            fails = [t for t, r in ((r["task_id"], r) for r in rows if r["category"] == c)
                     if t in base and t in full and not base[t]]
            if not fails:
                continue
            cats.append(c)
            ident.append(np.mean([idx[t] > 1 for t in fails]))
            recov.append(np.mean([full[t] for t in fails]))
            print(f"  {c:<13} baseline failures {len(fails):>3} | identifiable {ident[-1]:.2f} | recovered {recov[-1]:.2f}")
        pairs = [(idx[t], full[t]) for t in idx if t in base and t in full and not base[t]]
        rec_i = [i for i, ok in pairs if ok]
        non_i = [i for i, ok in pairs if not ok]
        print(f"\nprediction test: Spearman(identifiable fraction, recovery rate) across {len(cats)} categories = "
              f"{spearman(ident, recov):.2f}; median index of recovered failures {np.median(rec_i) if rec_i else float('nan'):.2f} "
              f"vs unrecovered {np.median(non_i) if non_i else float('nan'):.2f}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/scripts/sweep_conditions.py
"""Graded perturbation grid for the within-type identifiability test.

Every condition is applied programmatically to an UNPERTURBED LIBERO-10 scene (a LIBERO-Plus language variant with the
default view and robot start, run with the original instruction), so strength is controlled per type:
  camera      orbit azimuth 10/20/30/45 deg, pull-back x1.3, and look-away tilts 35/55 deg (arm leaves the frame)
  light       directional-light azimuth 90/180 deg, grazing elevation, diffuse gain 0.4/1.8, warm tint
  noise       Gaussian sensor noise sigma 5/10/20/40 on both cameras
  joint       start-pose offsets with max |dq| 0.10/0.20/0.35 rad
  background  table recolour red/dark (no coupling to the robot), and a "coloured room" (table recolour + matching
              light tint, which does reach the robot's appearance)
`override` marks conditions where the category label and the self-identifiability score are expected to disagree:
look-away camera shifts are labelled invertible but the arm is not visible; the coloured room is labelled background
(capability) but the colour cast is visible on the robot. MuJoCo has no global illumination, so colour bleed from a
coloured room is emulated by tinting the lights; this is disclosed as a simulation stand-in.
"""
import re

import numpy as np

INVERTIBLE_LABEL = {"nominal": 1, "camera": 1, "light": 1, "noise": 1, "joint": 1, "background": 0}


def C(family, name, level, override=False, **params):
    return dict(family=family, name=name, level=float(level), override=bool(override), params=params)


CONDITIONS = [
    C("nominal", "nominal", 0),
    C("camera", "orbit_az10", 10, az=10), C("camera", "orbit_az20", 20, az=20),
    C("camera", "orbit_az30", 30, az=30), C("camera", "orbit_az45", 45, az=45),
    C("camera", "pullback_1.3", 1.3, scale=1.3),
    C("camera", "look_away_35", 35, override=True, tilt=35), C("camera", "look_away_55", 55, override=True, tilt=55),
    C("light", "dir_az90", 90, dir_az=90), C("light", "dir_az180", 180, dir_az=180),
    C("light", "grazing", 1, grazing=True),
    C("light", "gain_0.4", 0.4, gain=0.4), C("light", "gain_1.8", 1.8, gain=1.8),
    C("light", "tint_warm", 1, tint=[1.35, 0.85, 0.6]),
    C("noise", "sigma5", 5, sigma=5), C("noise", "sigma10", 10, sigma=10),
    C("noise", "sigma20", 20, sigma=20), C("noise", "sigma40", 40, sigma=40),
    C("joint", "offset0.10", 0.10, offset=0.10), C("joint", "offset0.20", 0.20, offset=0.20),
    C("joint", "offset0.35", 0.35, offset=0.35),
    C("background", "table_red", 1, table_rgb=[0.8, 0.1, 0.1]),
    C("background", "table_dark", 1, table_rgb=[0.08, 0.08, 0.08]),
    C("background", "colored_room", 1, override=True, table_rgb=[0.8, 0.1, 0.1], tint=[1.3, 0.8, 0.8]),
]
# boundary conditions (--extended): levels between the base grid's steps, where a predictor is actually tested.
# Added 2026-09-18 after the OFT development analysis; every family runs them, none is tuned on them.
EXTENDED = [
    C("camera", "orbit_az15", 15, az=15), C("camera", "orbit_az25", 25, az=25), C("camera", "orbit_az38", 38, az=38),
    C("camera", "look_away_15", 15, override=True, tilt=15), C("camera", "look_away_25", 25, override=True, tilt=25),
    C("light", "gain_1.4", 1.4, gain=1.4),
    C("noise", "sigma30", 30, sigma=30),
    C("joint", "offset0.50", 0.50, offset=0.50),
]
BY_NAME = {c["name"]: c for c in CONDITIONS + EXTENDED}
DEFAULT_VIEW = ("0", "0", "100", "0", "0")


def base_tasks(suite, n):
    """n unperturbed base tasks spread over scenes: a language variant with the default view and robot start,
    paired with the ORIGINAL LIBERO-10 instruction recovered from the task name."""
    from plus_common import categorize
    from wrapper_libero import scene_of
    first = {}
    for i in range(suite.n_tasks):
        nm = suite.get_task(i).name
        if categorize(nm) != "language":
            continue
        v = re.search(r"_view_(\d+)_(\d+)_(\d+)_(\d+)_(\d+)", nm)
        s = re.search(r"_initstate_(\d+)", nm)
        if (v and v.groups() != DEFAULT_VIEW) or (s and s.group(1) != "0"):
            continue
        base = re.sub(r"_(view_\d+_\d+_\d+_\d+_\d+|initstate_\d+|language_\d+)", "", nm)
        first.setdefault(base, i)
    by_scene = {}
    for b in sorted(first):
        by_scene.setdefault(scene_of(b + "_"), []).append(b)
    order, k = [], 0
    while len(order) < len(first):
        for sc in sorted(by_scene):
            if k < len(by_scene[sc]):
                order.append(by_scene[sc][k])
        k += 1
    out = []
    for b in order[:n]:
        instr = re.sub(r"^[A-Z]+(?:_[A-Z]+)*_SCENE\d+_", "", b).replace("_", " ")
        out.append(dict(task_id=first[b], base=b, scene=scene_of(b + "_"), instruction=instr))
    return out


def center_depth(sim, cam, res=64):
    _, depth = sim.render(width=res, height=res, camera_name=cam, depth=True)
    m = sim.model
    extent = float(m.stat.extent)
    near, far = float(m.vis.map.znear) * extent, float(m.vis.map.zfar) * extent
    z = near / (1.0 - depth * (1.0 - near / far))
    c = res // 2
    return float(np.median(z[c - 4:c + 4, c - 4:c + 4]))


def _table_geoms(m):
    out = []
    for g in range(m.ngeom):
        gname = (m.geom_id2name(g) or "").lower()
        try:
            bname = (m.body_id2name(int(m.geom_bodyid[g])) or "").lower()
        except Exception:
            bname = ""
        if "table" in gname or "table" in bname:
            out.append(g)
    return out


def apply_scene(env, cond, cam="agentview", scale=1.0):
    """Model-level perturbation (camera, lights, table colour), `scale` in [0, 1+] multiplies the strength (used by
    the finite-difference score). Returns a dict describing what was changed."""
    import wrapper_libero as W
    sim = env.env.sim
    m = sim.model
    p = cond["params"]
    info = {}
    cid = m.camera_name2id(cam)
    C0, R0 = m.cam_pos[cid].copy(), W.quat_wxyz_to_mat(m.cam_quat[cid])
    if "tilt" in p:
        R = R0 @ W.rotvec_to_mat(np.array([np.deg2rad(p["tilt"] * scale), 0.0, 0.0]))
        m.cam_quat[cid][:] = W.mat_to_quat_wxyz(R)
        info["cam_shift_cm_deg"] = W.pose_error(C0, R, C0, R0)
    elif "az" in p or "scale" in p:
        pivot = C0 + center_depth(sim, cam) * (-R0[:, 2])
        C, R = W.orbit(C0, R0, pivot, p.get("az", 0) * scale, 0.0, 1.0 + (p.get("scale", 1.0) - 1.0) * scale)
        m.cam_pos[cid][:] = C
        m.cam_quat[cid][:] = W.mat_to_quat_wxyz(R)
        info["cam_shift_cm_deg"] = W.pose_error(C, R, C0, R0)
    if m.nlight > 0:
        if "dir_az" in p:
            m.light_dir[:] = (W.rot_z(p["dir_az"] * scale) @ m.light_dir.T).T
        if p.get("grazing"):
            d = m.light_dir.copy()
            horiz = np.where(np.linalg.norm(d[:, :2], axis=1, keepdims=True) > 1e-3, d[:, :2], np.array([[1.0, 0.0]]))
            horiz = horiz / np.linalg.norm(horiz, axis=1, keepdims=True)
            target = np.concatenate([horiz * 0.99, np.full((len(d), 1), -0.12)], 1)
            new = (1 - scale) * d + scale * target
            m.light_dir[:] = new / np.linalg.norm(new, axis=1, keepdims=True)
        if "gain" in p:
            m.light_diffuse[:] = m.light_diffuse * (1.0 + (p["gain"] - 1.0) * scale)
        if "tint" in p:
            m.light_diffuse[:] = m.light_diffuse * (1.0 + (np.asarray(p["tint"]) - 1.0) * scale)
    if "table_rgb" in p:
        for g in _table_geoms(m):
            tgt = np.asarray(p["table_rgb"])
            m.geom_rgba[g, :3] = m.geom_rgba[g, :3] + (tgt - m.geom_rgba[g, :3]) * min(scale, 1.0)
            mat = int(m.geom_matid[g])
            if mat >= 0:
                m.mat_rgba[mat, :3] = m.mat_rgba[mat, :3] + (tgt - m.mat_rgba[mat, :3]) * min(scale, 1.0)
        info["table_geoms"] = len(_table_geoms(m))
    sim.forward()
    return info


def joint_offset_vector(cond, seed, n=7):
    v = np.random.default_rng(seed).normal(size=n)
    return v / np.abs(v).max() * cond["params"]["offset"]


def apply_joint(env, cond, seed):
    robot = env.env.robots[0]
    idx = list(robot._ref_joint_pos_indexes)
    sim = env.env.sim
    sim.data.qpos[idx] = sim.data.qpos[idx] + joint_offset_vector(cond, seed, len(idx))
    sim.data.qvel[:] = 0.0
    sim.forward()
    try:
        robot.controller.update(force=True)
        robot.controller.reset_goal()
    except Exception:
        pass


def add_noise(obs, sigma, rng):
    if not sigma:
        return obs
    for k in ("agentview_image", "robot0_eye_in_hand_image"):
        if k in obs:
            obs[k] = np.clip(obs[k].astype(np.float32) + rng.normal(0.0, sigma, obs[k].shape), 0, 255).astype(np.uint8)
    return obs

In [ ]:
%%writefile /content/scripts/policy_adapters.py
"""One interface over the policy families used in the graded sweep:
    adapter.new_episode(instruction, seed)
    adapter.chunk(obs) -> list of 7-D env actions (one forward pass; chunked policies return several)
OFT and OpenVLA must be run from their repo roots (experiments.robot.*); Octo needs its calibration file from plus_5."""
import json

import numpy as np


class OFTAdapter:
    name = "openvla_oft"

    def __init__(self, ckpt=None):
        from experiments.robot.libero.run_libero_eval import GenerateConfig, initialize_model
        from experiments.robot.robot_utils import get_image_resize_size, set_seed_everywhere
        from plus_common import OFT_CKPT, SUITE
        ckpt = ckpt or OFT_CKPT[SUITE]                    # one released checkpoint per suite
        self.cfg = GenerateConfig(pretrained_checkpoint=ckpt, task_suite_name=SUITE, center_crop=True)
        set_seed_everywhere(self.cfg.seed)
        self.parts = initialize_model(self.cfg)
        self.resize = get_image_resize_size(self.cfg)
        self.instr = ""

    def new_episode(self, instruction, seed):
        self.instr = instruction

    def chunk(self, obs):
        from experiments.robot.libero.run_libero_eval import prepare_observation, process_action
        from experiments.robot.robot_utils import get_action
        model, action_head, proprio_projector, noisy_action_projector, processor = self.parts
        observation, _ = prepare_observation(obs, self.resize)
        acts = get_action(self.cfg, model, observation, self.instr, processor=processor, action_head=action_head,
                          proprio_projector=proprio_projector, noisy_action_projector=noisy_action_projector,
                          use_film=self.cfg.use_film)
        return [np.asarray(process_action(a, self.cfg.model_family), float) for a in acts]


class OpenVLAAdapter:
    name = "openvla"

    def __init__(self, ckpt="openvla/openvla-7b-finetuned-libero-10"):
        import inspect
        from experiments.robot.libero.libero_utils import get_libero_image
        from experiments.robot.libero.run_libero_eval import GenerateConfig
        from experiments.robot.openvla_utils import get_processor
        from experiments.robot.robot_utils import get_image_resize_size, get_model, set_seed_everywhere
        self.cfg = GenerateConfig(pretrained_checkpoint=ckpt, task_suite_name="libero_10", center_crop=True)
        set_seed_everywhere(self.cfg.seed)
        self.model = get_model(self.cfg)
        self.cfg.unnorm_key = self.cfg.task_suite_name
        if self.cfg.unnorm_key not in self.model.norm_stats and f"{self.cfg.unnorm_key}_no_noops" in self.model.norm_stats:
            self.cfg.unnorm_key = f"{self.cfg.unnorm_key}_no_noops"
        self.processor = get_processor(self.cfg)
        self.resize = get_image_resize_size(self.cfg)
        self.img_takes_resize = len(inspect.signature(get_libero_image).parameters) >= 2
        self.instr = ""

    def new_episode(self, instruction, seed):
        self.instr = instruction

    def chunk(self, obs):
        from experiments.robot.libero.libero_utils import get_libero_image, quat2axisangle
        from experiments.robot.robot_utils import get_action, invert_gripper_action, normalize_gripper_action
        img = get_libero_image(obs, self.resize) if self.img_takes_resize else get_libero_image(obs)
        observation = {"full_image": img,
                       "state": np.concatenate((obs["robot0_eef_pos"], quat2axisangle(obs["robot0_eef_quat"]),
                                                obs["robot0_gripper_qpos"]))}
        a = get_action(self.cfg, self.model, observation, self.instr, processor=self.processor)
        a = invert_gripper_action(normalize_gripper_action(a, binarize=True))
        return [np.asarray(a, float)]


class OctoAdapter:
    name = "octo"

    def __init__(self, calib_file, step=30000):
        from eval_octo import load_model
        self.model, self.stats = load_model(step)
        cal = json.load(open(calib_file))
        self.flip, self.gmode = bool(cal["flip"]), cal["gmode"]
        self.task, self.rng = None, None

    def new_episode(self, instruction, seed):
        import jax
        self.task = self.model.create_tasks(texts=[instruction])
        self.rng = jax.random.PRNGKey(seed)

    def chunk(self, obs):
        import jax
        from PIL import Image
        from eval_octo import to_env_gripper
        img = obs["agentview_image"]
        img = img[::-1, ::-1] if self.flip else img
        img = np.asarray(Image.fromarray(np.ascontiguousarray(img)).resize((256, 256)))
        o = {"image_primary": img[None, None], "timestep_pad_mask": np.ones((1, 1), dtype=bool)}
        self.rng, key = jax.random.split(self.rng)
        acts = np.asarray(self.model.sample_actions(o, self.task, unnormalization_statistics=self.stats, rng=key))[0]
        out = []
        for a in acts:
            a = np.asarray(a, dtype=np.float64).copy()
            a[6] = to_env_gripper(float(a[6]), self.gmode)
            out.append(a)
        return out


class LeRobotAdapter:
    """pi0.5 / pi0 through LeRobot's own pipeline (verified against lerobot 0.6.1 lerobot_eval.rollout): the raw
    robosuite observation is formatted exactly like envs/libero.LiberoEnv._format_raw_obs, then
    preprocess_observation -> env preprocessor (LiberoProcessorStep: 180 deg image flip, eef pos + axis-angle +
    gripper qpos state) -> policy preprocessor -> select_action -> postprocessor -> env postprocessor.
    select_action keeps its own action queue, so chunk() returns one action per call."""

    def __init__(self, policy_path):
        import torch
        from lerobot.configs.policies import PreTrainedConfig
        from lerobot.envs.configs import LiberoEnv
        from lerobot.envs.factory import make_env_pre_post_processors
        from lerobot.policies.factory import make_policy, make_pre_post_processors
        self.name = "pi05" if "pi05" in policy_path else ("pi0" if "pi0" in policy_path else "lerobot")
        cfg = PreTrainedConfig.from_pretrained(policy_path)
        cfg.pretrained_path = policy_path
        if hasattr(cfg, "compile_model"):
            cfg.compile_model = False
        env_cfg = LiberoEnv(task="libero_10")
        self.policy = make_policy(cfg=cfg, env_cfg=env_cfg)
        self.policy.eval()
        self.pre, self.post = make_pre_post_processors(
            policy_cfg=cfg, pretrained_path=policy_path,
            preprocessor_overrides={"device_processor": {"device": str(self.policy.config.device)},
                                    "rename_observations_processor": {"rename_map": {}}})
        self.env_pre, self.env_post = make_env_pre_post_processors(env_cfg=env_cfg, policy_cfg=cfg)
        self.torch = torch
        self.instr = ""

    def new_episode(self, instruction, seed):
        self.instr = instruction
        self.policy.reset()
        self.torch.manual_seed(seed)

    def chunk(self, obs):
        from lerobot.envs.utils import preprocess_observation
        b = lambda x: np.asarray(x, np.float64)[None]
        raw = {"pixels": {"image": np.ascontiguousarray(obs["agentview_image"])[None],
                          "image2": np.ascontiguousarray(obs["robot0_eye_in_hand_image"])[None]},
               "robot_state": {"eef": {"pos": b(obs["robot0_eef_pos"]), "quat": b(obs["robot0_eef_quat"]),
                                       "mat": np.eye(3)[None]},
                               "gripper": {"qpos": b(obs["robot0_gripper_qpos"]), "qvel": b(obs["robot0_gripper_qvel"])},
                               "joints": {"pos": b(obs["robot0_joint_pos"]), "vel": b(obs["robot0_joint_vel"])}}}
        o = preprocess_observation(raw)
        o["task"] = [self.instr]
        o = self.pre(self.env_pre(o))
        with self.torch.inference_mode():
            a = self.policy.select_action(o)
        a = self.post(a)
        a = self.env_post({"action": a})["action"]
        return [np.asarray(a.to("cpu").numpy(), float)[0]]


class RemoteAdapter:
    """A policy that lives in another process (policy_server.py): for machines where the simulator's pins and the
    policy's pins cannot share one Python. Same interface as every adapter."""
    KEYS = ("agentview_image", "robot0_eye_in_hand_image", "robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos",
            "robot0_gripper_qvel", "robot0_joint_pos", "robot0_joint_vel")

    def __init__(self, name, port=6006):
        import os
        from multiprocessing.connection import Client
        self.name = name
        self.conn = Client(("127.0.0.1", int(os.environ.get("POLICY_PORT", port))), authkey=b"self-motion")

    def new_episode(self, instruction, seed):
        self.conn.send(("new", instruction, int(seed)))
        self.conn.recv()

    def chunk(self, obs):
        self.conn.send(("chunk", {k: np.ascontiguousarray(obs[k]) for k in self.KEYS}))
        return [np.asarray(a, float) for a in self.conn.recv()]


def make_adapter(family, octo_calib=""):
    if family == "smolvla":                 # served by policy_server.py --policy lerobot/smolvla_libero
        return RemoteAdapter("smolvla")
    if family == "pi05":
        return LeRobotAdapter("lerobot/pi05_libero_finetuned_v044")
    if family == "pi0":
        return LeRobotAdapter("lerobot/pi0_libero_finetuned_v044")
    if family == "oft":
        return OFTAdapter()
    if family == "openvla":
        return OpenVLAAdapter()
    if family == "octo":
        assert octo_calib, "--octo_calib is required for octo (run the plus_5 calibration cell first)"
        return OctoAdapter(octo_calib)
    raise ValueError(family)

In [ ]:
%%writefile /content/scripts/sweep_run.py
"""Graded sweep: does self-identifiability predict which perturbations a frozen policy recovers from?

  score     policy-free. For every (base task, condition): apply the perturbation, run the arm sweep, and compute the
            self-identifiability of the condition's own parameters: the smallest singular value of the finite-
            difference Jacobian of the robot's self-observation (silhouette, body colour, joint angles, noise floor)
            with respect to those parameters, at the perturbed state, relative to the same block at the unperturbed
            state (stored as sigma_min_ref; scores_v2.jsonl). Also records the invertibility components (training-view
            coverage from the self-calibrated pose, body shading consistency, denoiser efficacy) and the composite
            recoverability score (sweep_analysis.recoverability).
  episodes  one policy family. For every (condition, base task): a RAW episode (perturbed, no wrapper) and a FULL
            episode (perturbed + self-measured wrapper). The raw episode also records a policy-internal probe: the
            change of the policy's first action chunk against the unperturbed first chunk.
Both are resumable (jsonl, one row per unit of work).
"""
import argparse
import json
import os
import sys
import time

import numpy as np

os.environ.setdefault("MUJOCO_GL", "egl")
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import sweep_conditions as S  # noqa: E402
import wrapper_libero as W  # noqa: E402
from calib_check import make_env  # noqa: E402
from plus_common import append_row, load_rows, load_shards  # noqa: E402

from plus_common import MAX_STEPS as _HORIZON, SUITE  # noqa: E402
MAX_STEPS, WAIT = _HORIZON[SUITE], 10
JOINT_THR, NOISE_MULT, NOISE_ADD, PHOTO_SHIFT, PHOTO_GAIN = 0.05, 2.0, 1.0, 15.0, 1.25


def start(env, suite, task, cond, seed):
    env.reset()
    obs = env.set_init_state(suite.get_task_init_states(task["task_id"])[0])
    for _ in range(WAIT):
        obs, _, _, _ = env.step([0, 0, 0, 0, 0, 0, -1])
    info = S.apply_scene(env, cond)
    if cond["family"] == "joint":
        S.apply_joint(env, cond, seed)
    for _ in range(3):                              # re-render with the perturbed model and let contacts settle
        obs, _, _, _ = env.step([0, 0, 0, 0, 0, 0, -1])
    return obs, info


# ------------------------------------------------------------------ policy-free self-identifiability
def obs_vector(can, recs, sigma, arm_offset=None):
    import identifiability as I
    _, cols, joints, blur = I.self_obs(can, recs, arm_offset=arm_offset)
    img = can.sim.render(width=128, height=128, camera_name=can.cam)
    img = S.add_noise({"agentview_image": img}, sigma, np.random.default_rng(0))["agentview_image"]
    return np.concatenate([blur, cols.ravel() / 255.0 * 10.0, joints.ravel() * 10.0, [W.noise_level(img) / 255.0 * 10.0]])


def block_deltas(can, cond):
    """(label, apply) pairs: one small step of each parameter of the condition's own family, applied on top of the
    current (perturbed) state. Undo is handled by the caller's save/restore of model arrays."""
    m = can.sim.model
    fam, p = cond["family"], cond["params"]
    C0, R0 = can.true_camera()
    out = []
    if fam in ("camera", "nominal"):
        for i in range(3):
            d = np.zeros(3); d[i] = 0.01
            out.append((f"cam_pos{i}", lambda d=d: can._set_camera(C0 + d, R0)))
        for i in range(3):
            r = np.zeros(3); r[i] = np.deg2rad(1.0)
            out.append((f"cam_rot{i}", lambda r=r: can._set_camera(C0, W.rotvec_to_mat(r) @ R0)))
    elif fam == "light":
        if "dir_az" in p or p.get("grazing"):
            out.append(("light_dir_az", lambda: m.light_dir.__setitem__(slice(None), (W.rot_z(5.0) @ m.light_dir.T).T)))

            def tilt():
                d = m.light_dir.copy()
                d[:, 2] = d[:, 2] * np.cos(np.deg2rad(5.0)) - 0.087
                m.light_dir[:] = d / np.linalg.norm(d, axis=1, keepdims=True)
            out.append(("light_dir_el", tilt))
        else:
            for c in range(3):
                g = np.ones(3); g[c] = 1.1
                out.append((f"light_gain{c}", lambda g=g: m.light_diffuse.__setitem__(slice(None), m.light_diffuse * g)))
    elif fam == "background":
        if "tint" in p:                                   # coloured room: one scalar drives table colour AND light tint
            step = dict(family="background", params=dict(table_rgb=p["table_rgb"], tint=p["tint"]))
            out.append(("room_colour", lambda: S.apply_scene(can.env, step, scale=0.1)))
        else:
            for c in range(3):
                def recolour(c=c):
                    for g in S._table_geoms(m):
                        m.geom_rgba[g, c] = min(1.0, m.geom_rgba[g, c] + 0.1)
                        if int(m.geom_matid[g]) >= 0:
                            m.mat_rgba[int(m.geom_matid[g]), c] = min(1.0, m.mat_rgba[int(m.geom_matid[g]), c] + 0.1)
                out.append((f"table_rgb{c}", recolour))
    return out


def score_instance(env, obs, cond, nominal, seed):
    can = W.Canonicalizer(env, nominal)
    joint_dev = can.joint_dev()
    obs, recs = can.sweep(obs)
    m = can.sim.model
    sigma = cond["params"].get("sigma", 0.0)
    snap = (can._save(), m.light_dir.copy(), m.light_diffuse.copy(), m.geom_rgba.copy(), m.mat_rgba.copy())

    def undo():
        m.light_dir[:], m.light_diffuse[:], m.geom_rgba[:], m.mat_rgba[:] = snap[1], snap[2], snap[3], snap[4]
        can._restore(snap[0])

    base = obs_vector(can, recs, sigma)
    cols, labels = [], []
    if cond["family"] == "joint":
        for k in range(len(can.arm_idx)):
            off = np.zeros(len(can.arm_idx)); off[k] = 0.02
            cols.append(obs_vector(can, recs, sigma, arm_offset=off) - base); labels.append(f"joint{k}")
            undo()
    elif cond["family"] == "noise":
        cols.append(obs_vector(can, recs, sigma + 2.0) - base); labels.append("sigma")
    else:
        for lab, apply in block_deltas(can, cond):
            apply()
            can.sim.forward()
            cols.append(obs_vector(can, recs, sigma) - base); labels.append(lab)
            undo()
    J = np.stack(cols, 1)
    sv = np.linalg.svd(J, compute_uv=False)
    area = float(np.mean([rec[2].mean() for rec in recs]))
    # invertibility components, all policy-free and self-measured (see sweep_analysis.recoverability):
    #   coverage   fraction of the training view the reprojection from the self-calibrated pose can reconstruct
    #   photo_d    affine-invariant body shading statistic (valid affine inversion <=> unchanged vs the reference)
    #   noise_inv  share of the added noise the denoiser removes, judged by the noise floor of the robot's own pixels
    can.calibrate_v2(recs)
    _, coverage = can.canonical_image(obs)
    photo_d = can.photometric_consistency(obs)
    noisy = S.add_noise({"agentview_image": obs["agentview_image"]}, sigma, np.random.default_rng(seed))["agentview_image"]
    lvl, floor = W.noise_level(noisy), float(nominal["noise"]["agentview_image"])
    if lvl <= NOISE_MULT * floor + NOISE_ADD:
        noise_inv = 1.0
    else:
        resid = max(W.noise_level(W.denoise(noisy, lvl)) - floor, 0.0)
        noise_inv = float(np.clip(1.0 - resid / max(lvl - floor, 1e-6), 0.0, 1.0))
    return dict(sigma_min=float(sv[-1]), sigma_max=float(sv[0]), n_params=len(labels), params=labels,
                robot_area=area, joint_dev=joint_dev, coverage=float(coverage), photo_d=photo_d, noise_inv=noise_inv)


def block_key(cond):
    """Which finite-difference block a condition is scored on (its own family's parameters)."""
    fam, p = cond["family"], cond["params"]
    if fam == "light":
        return (fam, "dir" if ("dir_az" in p or p.get("grazing")) else "gain")
    if fam == "background":
        return (fam, "tint" if "tint" in p else "table")
    return (fam,)


def run_score(a, suite, tasks, nominals):
    """scores_v2: every condition is scored on its OWN block and normalised by the SAME block at the unperturbed
    state of the same base task (sigma_min_ref). The v1 file normalised every family by the camera block, which
    made light, noise and joint scores incomparable to camera scores. Also stores the invertibility components."""
    from sweep_analysis import recoverability
    path = os.path.join(a.out_dir, "scores_v2.jsonl")
    rows = load_rows(path)
    done = {(r["task_id"], r["cond"]) for r in rows}
    refs = {(r["task_id"], tuple(r["block"])): r["ref"] for r in rows if r.get("ref")}
    with open(path, "a") as f:
        for cond in S.CONDITIONS + (S.EXTENDED if a.extended else []):
            for t in tasks:
                if (t["task_id"], cond["name"]) in done:
                    continue
                key = (t["task_id"], block_key(cond))
                if key not in refs:                                  # same block, unperturbed state, once per task
                    cond_ref = dict(cond, params={k: v for k, v in cond["params"].items() if k != "sigma"})
                    env = make_env(suite, t["task_id"])
                    try:
                        obs, _ = start(env, suite, t, S.BY_NAME["nominal"], seed=t["task_id"])
                        refs[key] = score_instance(env, obs, cond_ref, nominals[t["scene"]], seed=t["task_id"])
                    finally:
                        env.close()
                env = make_env(suite, t["task_id"])
                try:
                    obs, info = start(env, suite, t, cond, seed=t["task_id"])
                    sc = score_instance(env, obs, cond, nominals[t["scene"]], seed=t["task_id"])
                finally:
                    env.close()
                row = dict(task_id=t["task_id"], base=t["base"], cond=cond["name"], family=cond["family"],
                           level=cond["level"], override=cond["override"], block=list(key[1]),
                           sigma_min_ref=refs[key]["sigma_min"], photo_d_ref=refs[key]["photo_d"],
                           ref=refs[key], **sc, **info)
                row.update(recoverability(row))
                f.write(json.dumps(row, default=float) + "\n")
                f.flush()
                print(f"score {t['task_id']:5d} {cond['name']:<14} ident {row['ident']:.2f} invert {row['invert']:.2f} "
                      f"recover {row['recover']:.2f} | sigma_min {sc['sigma_min']:.4f} / ref {refs[key]['sigma_min']:.4f} "
                      f"coverage {sc['coverage']:.2f} area {sc['robot_area']:.3f}", flush=True)


# ------------------------------------------------------------------ policy episodes
# ---- the do-no-harm gate (pre-registered 2026-09-18, RESULTS.md) -------------------------------------------------
# A correction is an estimate. When the policy is already robust to the nuisance, the corrected frame can be FURTHER
# from what the policy handles than the raw one (pi0.5: light 0.90 -> 0.50 under the wrapper). The gate asks the frozen
# policy itself, without labels: is it less self-consistent on the corrected frame than on the raw frame? Self-
# consistency = how far its action moves under benign jitters of the input (2 px shifts, 4 percent brightness). A
# channel is VETOED only on positive evidence: the action deviation is larger on the corrected frame for at least
# GATE_MIN_WORSE of the GATE jitters, paired. Ties and mixed evidence leave the wrapper's own decision standing.
# No threshold is tuned on any policy; the state servo is never gated (no harm in either family).
GATE_JITTERS = (("shift", 2, 0), ("shift", -2, 0), ("shift", 0, 2), ("shift", 0, -2), ("gain", 1.04), ("gain", 0.96))
GATE_MIN_WORSE = 5


def _jitter(o, j):
    import cv2
    out = dict(o)
    for k in W.IMG_KEYS:
        if k not in out:
            continue
        img = np.asarray(out[k])
        if j[0] == "shift":
            M = np.float32([[1, 0, j[1]], [0, 1, j[2]]])
            out[k] = cv2.warpAffine(img, M, (img.shape[1], img.shape[0]), borderMode=cv2.BORDER_REPLICATE)
        else:
            out[k] = np.clip(img.astype(np.float32) * j[1], 0, 255).astype(np.uint8)
    return out


def gate_profile(act, o):
    """The policy's action on o and its deviations under the jitters. act: observation -> action vector."""
    a0 = np.asarray(act(o), float)
    return a0, np.array([float(np.linalg.norm(np.asarray(act(_jitter(o, j)), float) - a0)) for j in GATE_JITTERS])


def gate_decide(prof_raw, prof_cor):
    (a_raw, d_raw), (a_cor, d_cor) = prof_raw, prof_cor
    worse = int(np.sum(d_cor > d_raw))
    veto = bool(worse >= GATE_MIN_WORSE and d_cor.mean() > d_raw.mean())
    return dict(veto=veto, worse=worse, s_raw=float(d_raw.mean()), s_cor=float(d_cor.mean()),
                action_change=float(np.linalg.norm(a_cor - a_raw)))


def generic_correction(o):
    """BASELINE `generic`: the corrections anyone would reach for without measuring anything: gray-world white balance,
    CLAHE on the lightness channel, a light edge-preserving denoise. Applied to every frame, no abstention."""
    import cv2
    for k in W.IMG_KEYS:
        if k not in o:
            continue
        img = np.asarray(o[k]).astype(np.float32)
        img *= img.mean() / np.maximum(img.reshape(-1, 3).mean(0), 1.0)              # gray world
        lab = cv2.cvtColor(np.clip(img, 0, 255).astype(np.uint8), cv2.COLOR_RGB2LAB)
        lab[..., 0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lab[..., 0])
        o[k] = cv2.bilateralFilter(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB), 5, 25, 3)     # edge-preserving, ~2 ms a frame
    return o


LAZY_RESIDUAL = 1.5      # POST HOC (2026-09-21), chosen from logged rows, NOT pre-registered: the silhouette residual at the
#                          training camera is under 0.86 on every unshifted unit and over 2.26 on every camera-shifted one
SELFCHECK_MIN = 0.75     # frozen 2026-09-19 from policy-free data (self-check F1 0.82 at 5 cm of true error, 0.63 at 10 cm)


def prepare_full(env, obs, nominal, noise, sensing=None, pose=None, gate=None, cam_mode="full", settle=False,
                 verify=False, lazy=False):
    """The deployable wrapper (same rules as eval_oft_wrapper `full`); returns the per-step transform.
    pose="true" is a DIAGNOSTIC upper bound: the reprojection uses the simulator's camera pose instead of the
    calibrated one (rows are labelled *_truepose and never reported as deployable)."""
    can = W.Canonicalizer(env, nominal, sensing=sensing)
    info = dict(joint_dev=can.joint_dev())
    start_pose = ((np.asarray(obs["robot0_eef_pos"]).copy(), np.asarray(obs["robot0_eef_quat"]).copy())
                  if "robot0_eef_pos" in obs and "robot0_eef_quat" in obs else None)
    if hasattr(can, "nominal_eef") and "robot0_eef_pos" in obs:      # logging only: how far the untouched start is from the training pose
        info["start_off_mm"] = float(1e3 * np.linalg.norm(np.asarray(obs["robot0_eef_pos"]) - can.nominal_eef()[0]))
    if info["joint_dev"] > JOINT_THR:
        p, q = can.nominal_eef()
        obs, info["home_steps"] = can.servo(obs, p, q, max_steps=200)
        info["state_applied"] = True
    skipped = False
    if lazy:
        # MEASURE ONLY WHEN THE BODY SAYS THE CAMERA MOVED: one motion-free look at the robot's own silhouette. When it
        # sits where the training camera puts it, the sweep is skipped and the episode starts from its untouched state.
        info["static_residual"] = can.static_residual(obs)
        skipped = info["static_residual"] <= LAZY_RESIDUAL
        info["lazy_skipped"] = bool(skipped)
    if skipped:
        can.C_est, can.R_est = can.Cn.copy(), can.Rn.copy()
        cal = dict(method="not measured: the silhouette sits where the training camera puts it", applied=False)
    else:
        obs, recs = can.sweep(obs)
        cal = can.calibrate_v2(recs)
    info.update({f"cal_{k}": v for k, v in cal.items()})
    arm_speed = lambda: float(np.abs(can.sim.data.qvel[can.arm_idx]).max())
    if hasattr(can, "arm_idx"):
        info["handover_speed"] = arm_speed()                     # logged for every arm: how still the arm is handed over
    if verify and not skipped and hasattr(can, "self_check"):
        # MEASURE UNTIL VERIFIED: the self-check (policy-free) rejects the calibration -> measure again with the richer
        # 7-pose sweep and keep whichever calibration the robot's own silhouette agrees with more.
        sc0 = can.self_check()
        if sc0 is not None and sc0["f1"] < SELFCHECK_MIN:
            keep = (can.C_est.copy(), can.R_est.copy(), cal, sc0["f1"])
            obs, recs2 = can.sweep(obs, poses=W.SWEEP_LEARNED)
            cal2 = can.calibrate_v2(recs2)
            sc1 = can.self_check()
            info.update(verify_retried=True, verify_f1_before=sc0["f1"], verify_f1_after=sc1["f1"])
            if sc1["f1"] > keep[3]:
                cal = cal2
                info.update({f"cal_{k}": v for k, v in cal.items()})
            else:
                can.C_est, can.R_est = keep[0], keep[1]
    if (settle or verify) and not (skipped and not info.get("state_applied")):   # nothing moved the arm: nothing to settle
        # target: the training pose when the state channel moved the arm there, otherwise the pose the episode STARTED
        # from (on some tasks that is 17 to 25 mm from the training pose: `full` silently moved the arm by that much)
        tgt = can.nominal_eef() if (info.get("state_applied") or start_pose is None) else start_pose
        obs, st = can.settle(obs, *tgt)
        info.update(settle_steps=st["steps"], settle_err_mm=st["err_mm"], handover_speed=st["speed"],
                    settle_target="training" if info.get("state_applied") or start_pose is None else "start")
    use_cam = bool(cal["applied"])
    try:                                                         # logging only, and never allowed to stop a run
        sc = can.self_check() if hasattr(can, "self_check") else None
        if sc:
            info.update(selfcheck_f1=sc["f1"], selfcheck_precision=sc["precision"], selfcheck_recall=sc["recall"])
    except Exception as e:
        info["selfcheck_error"] = str(e)[:120]
    Ct, Rt = can.true_camera()                                   # logging only, except under pose="true"
    info["cam_est_err_cm_deg"] = list(W.pose_error(can.C_est, can.R_est, Ct, Rt))
    info["cam_true_shift_cm_deg"] = list(W.pose_error(Ct, Rt, can.Cn, can.Rn))
    info["cam_est_pose"] = [*can.C_est.tolist(), *W.mat_to_rotvec(can.R_est).tolist()]      # position + rotation vector
    info["cam_nom_pose"] = [*can.Cn.tolist(), *W.mat_to_rotvec(can.Rn).tolist()]
    if can.mask_diag:
        info["mask_iou"] = float(np.mean([d["iou"] for d in can.mask_diag]))
        info["mask_area_ratio"] = float(np.mean([d["area_ratio"] for d in can.mask_diag]))
    if pose == "true":
        can.C_est, can.R_est, use_cam = Ct, Rt, True
    elif pose in ("refine", "refine2", "fiducial"):
        # POST HOC pose-source ablation (TASKS.md P47, pose_sources.py): same sweep, state channel and hand-over as
        # `full`; only the pose that drives the reprojection changes. cam_est_err_cm_deg is the error of the pose USED.
        assert not skipped, "the pose-source arms always measure"
        import pose_sources as PS
        img_hw = np.asarray(obs["agentview_image"]).shape[:2]
        C, R, use_cam, pinfo = PS.estimate(pose, can, recs, img_hw, W.TABLE_Z)
        can.C_est, can.R_est = C, R
        info.update(pinfo, cam_pose_applied=bool(use_cam))
        info["cam_est_err_cm_deg"] = list(W.pose_error(C, R, Ct, Rt))
        info["cam_est_pose"] = [*C.tolist(), *W.mat_to_rotvec(R).tolist()]
    cov = []
    obs = noise(obs)
    den = []
    for k in W.IMG_KEYS:
        if k in obs and k in nominal["noise"]:
            lvl = W.noise_level(obs[k])
            info[f"noise_{k}"] = lvl
            if lvl > NOISE_MULT * nominal["noise"][k] + NOISE_ADD:
                den.append(k)
    probe = dict(obs)
    for k in den:
        probe[k] = W.denoise(obs[k], info[f"noise_{k}"])
    ph = can.photometric_fit(probe, nominal)
    photo = ph["photo_ok"] and (ph["photo_mean_shift"] > PHOTO_SHIFT
                           or max(max(g, 1 / max(g, 1e-6)) for g in ph["photo_gain"]) > PHOTO_GAIN)
    if gate is not None:                   # gate = act(observation) -> action; channels are judged in pipeline order
        info.update(wanted_cam=use_cam, wanted_photo=bool(photo), wanted_denoise=list(den))
        copy = lambda o: {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in o.items()}

        def ch_den(o):
            for k in den:
                o[k] = W.denoise(o[k], info[f"noise_{k}"])
            return o

        def ch_photo(o):
            for k in W.IMG_KEYS:
                if k in o:
                    o[k] = can.apply_photo(o[k])
            return o

        def ch_cam(o):
            o["agentview_image"], _ = can.canonical_image(o)
            return o

        base, prof, vetoes = copy(obs), None, []
        for name, want, fn in (("denoise", bool(den), ch_den), ("photo", bool(photo), ch_photo), ("cam", use_cam, ch_cam)):
            if not want:
                continue
            if prof is None:
                prof = gate_profile(gate, base)
            cand = fn(copy(base))
            prof_c = gate_profile(gate, cand)
            dec = gate_decide(prof, prof_c)
            info[f"gate_{name}"] = dec
            if dec["veto"]:
                vetoes.append(name)
            else:
                base, prof = cand, prof_c
        if "denoise" in vetoes:
            den = []
        if "photo" in vetoes:
            photo = False
        if "cam" in vetoes:
            use_cam = False
        info["gate_vetoes"] = vetoes
    info.update(ph, cam_applied=use_cam, photo_applied=bool(photo), denoise_applied=den)

    def transform(o):
        for k in den:
            o[k] = W.denoise(o[k], info[f"noise_{k}"])
        if photo:
            for k in W.IMG_KEYS:
                if k in o:
                    o[k] = can.apply_photo(o[k])
        if cam_mode == "oracle":
            # UPPER BOUND, never a method: the simulator's own render from the training camera replaces the frame. Same
            # measurement sweep, state channel and hand-over as `full`, so the ONLY difference is a perfect camera undo.
            # Top rung of the attribution ladder (attribution_ladder.py): what still fails here is not the camera.
            o["agentview_image"] = can.oracle_nominal_image(o)
        elif use_cam and cam_mode == "homog":
            # BASELINE: the measured pose with NO depth: one plane-induced homography (the training setup's table
            # plane) for every pixel, the robot included; holes inpainted. What a calibrated camera alone buys.
            img = o["agentview_image"]
            Hh = img.shape[0]
            z = W.plane_depth(Hh, img.shape[1], can._focal(Hh), can.C_est, can.R_est, can.table_z)
            out, c = W.reproject(can._to_obs(img), z, can._focal(Hh), can.C_est, can.R_est, can.Cn, can.Rn, inpaint=True)
            o["agentview_image"] = can._to_obs(out)
            cov.append(c)
        elif use_cam:
            o["agentview_image"], c = can.canonical_image(o, fill="ref" if cam_mode == "full" else "inpaint")
            cov.append(c)
        if cov:
            info["reproj_coverage"] = float(np.mean(cov))
        return o
    return obs, transform, info


def run_episode(adapter, env, suite, task, cond, arm, nominal, sensing=None, seed_offset=0, pose=None,
                gate_skip=False):
    """arm: raw | full | gated (full + the do-no-harm gate) | baselines: tta (the raw policy, actions averaged over
    benign jitters of the frame), generic (gray world + CLAHE + denoise on every frame), homog (full with a depth-free
    homography as the camera channel), nofill (full with inpainted holes instead of the training-view reference),
    settle (full, but the measurement ends AT REST at the pose the episode started from, or at the training pose
    when the state channel moved the arm), verify (settle + measure again with 7 poses when the self-check rejects the
    calibration), full_tta (full, with the policy averaged over benign jitters of the corrected frame).
    gate_skip: a gated episode in which the gate vetoes nothing is identical to the full episode, so it returns
    (None, 0, None, info) and the caller copies the full row."""
    assert not (arm in ("tta", "full_tta") and adapter.name in ("pi05", "pi0", "smolvla", "lerobot")), "tta needs a chunk-returning adapter"
    seed = task["task_id"] + seed_offset
    obs, info = start(env, suite, task, cond, seed)
    rng = np.random.default_rng(seed)
    sigma = cond["params"].get("sigma", 0.0)
    noise = lambda o: S.add_noise(o, sigma, rng)
    adapter.new_episode(task["instruction"], seed)
    first_chunk = None
    transform = None
    if arm in ("homog", "nofill", "settle", "verify", "oracle", "lazy"):
        obs, transform, winfo = prepare_full(env, obs, nominal, noise, sensing=sensing, pose=pose,
                                             cam_mode=arm if arm in ("homog", "nofill", "oracle") else "full",
                                             settle=arm in ("settle", "lazy"), verify=arm == "verify", lazy=arm == "lazy")
        info.update(winfo)
    elif arm == "generic":
        obs = noise(obs)
        transform = generic_correction
        winfo = {}
    elif arm == "nominal_view":
        # DIAGNOSTIC base rate for the camera category: nothing moves and nothing is measured; every frame the policy sees
        # is the simulator's own render from the TRAINING camera (the ladder's top-rung render without the measurement
        # sweep, the state channel or the hand-over). The camera shift changes only the image, so this is the unperturbed
        # task at the same layout, start state and seed: the success any camera correction can at best return to.
        can_nv = W.Canonicalizer(env, nominal)
        obs = noise(obs)

        def transform(o):
            o["agentview_image"] = can_nv.oracle_nominal_image(o)
            return o
        winfo = {}
        info["nominal_view"] = True
    elif arm == "nominal_start":
        # DIAGNOSTIC base rate for the robot-init category: the arm is PLACED at the training start q_nom (no servo, no
        # measurement, no correction of anything), then the raw policy runs. Only the joint perturbation is removed, so
        # this is the unperturbed task at the same object layout and seed: what the servo arm can at best return to.
        robot = env.env.robots[0]
        qi, vi = list(robot._ref_joint_pos_indexes), list(robot._ref_joint_vel_indexes)
        env.env.sim.data.qpos[qi] = np.asarray(nominal["q_nom"], float)
        env.env.sim.data.qvel[vi] = 0.0
        env.env.sim.forward()
        for _ in range(3):                          # re-render and let contacts settle, as `start` does
            obs, _, _, _ = env.step([0, 0, 0, 0, 0, 0, -1])
        obs = noise(obs)
        info["nominal_start"] = True
        info["joint_dev_after"] = float(np.abs(env.env.sim.data.qpos[qi] - np.asarray(nominal["q_nom"], float)).max())
    elif arm in ("full", "gated", "full_tta"):
        gate = None
        if arm == "gated":
            def gate(o):                        # same instruction, same seed for every probe: only the frame differs
                adapter.new_episode(task["instruction"], seed)
                return np.mean([np.asarray(q, float)[:6] for q in adapter.chunk(dict(o))], 0)
        obs, transform, winfo = prepare_full(env, obs, nominal, noise, sensing=sensing, pose=pose, gate=gate)
        info.update(winfo)
        if arm == "gated":
            adapter.new_episode(task["instruction"], seed)          # the probes must leave no trace in the episode
            if gate_skip and not info.get("gate_vetoes"):
                info["gate_same_as_full"] = True
                return None, 0, None, info
    else:
        obs = noise(obs)
    queue, success, steps = [], False, 0
    while steps < MAX_STEPS:
        o = dict(obs)
        if transform is not None:
            o = transform(o)
        if not queue and arm in ("tta", "full_tta"):   # tta: the raw frame; full_tta: the CORRECTED frame (o is post-transform)
            chunks = [list(adapter.chunk(dict(o)))] + [list(adapter.chunk(_jitter(dict(o), j))) for j in GATE_JITTERS[:4]]
            n = min(len(c) for c in chunks)
            queue = [np.mean([np.asarray(c[i], float) for c in chunks], 0) for i in range(n)]
            for q in queue:
                q[6] = np.sign(chunks[0][0][6]) if abs(q[6]) < 1e-6 else np.sign(q[6])       # the gripper command is binary
        elif not queue:
            queue = list(adapter.chunk(o))
        if first_chunk is None:
            first_chunk = np.mean([q[:6] for q in queue], 0).tolist()
        obs, _, done, _ = env.step(queue.pop(0).tolist())
        obs = noise(obs)
        steps += 1
        if done:
            success = True
            break
    if transform is not None:
        info.update(winfo)                  # per-step diagnostics (reprojection coverage) accumulate during the episode
    return success, steps, first_chunk, info


def run_episodes(a, suite, tasks, nominals):
    from policy_adapters import make_adapter
    adapter = make_adapter(a.family, a.octo_calib)
    base = os.path.join(a.out_dir, f"episodes_{adapter.name}.jsonl")
    import socket
    shard = socket.gethostname()[-6:] if a.shard == "auto" else a.shard          # same suffix as this runtime's mirrored logs
    path = base[:-len(".jsonl")] + f".{shard}.jsonl" if shard else base          # this session appends ONLY to its own shard
    print(f"results shard: {os.path.basename(path)} (rows of every other session count as done)", flush=True)
    rows = load_rows(path)
    is_stale = lambda r: r.get("photo_applied") and "photo_shading" not in r and not r.get("recovered")
    stale = [r for r in rows if is_stale(r)]
    if stale:                                             # photometric map applied before the shading gate: redo
        keep = [r for r in rows if not is_stale(r)]
        with open(path + ".before_shading_gate", "w") as f:
            f.writelines(json.dumps(r) + "\n" for r in rows)
        with open(path, "w") as f:
            f.writelines(json.dumps(r) + "\n" for r in keep)
        rows = keep
        print(f"re-queued {len(stale)} episodes whose photometric map predates the shading gate", flush=True)
    rows = [r for r in load_shards(base) if not is_stale(r)]          # what EVERY session has finished counts as done
    done = {(r["task_id"], r["cond"], r["arm"], r.get("rep", 0)) for r in rows}
    full_rows = {(r["task_id"], r["cond"], r.get("rep", 0)): r for r in rows if r["arm"] == "full" and "steps" in r}
    nominal_chunk = {r["task_id"]: r["first_chunk"] for r in rows
                     if r["cond"] == "nominal" and r["arm"] == "raw" and r.get("first_chunk") is not None}
    arms = a.arms.split(",")
    conds = S.CONDITIONS + (S.EXTENDED if a.extended else [])
    todo = [(c, t, arm, rep) for c in conds if not a.conditions or c["name"] in a.conditions.split(",")
            for t in tasks for arm in arms for rep in range(a.repeats)
            if (t["task_id"], c["name"], arm, rep) not in done]
    print(f"{adapter.name}: {len(todo)} episodes to run", flush=True)
    t0 = time.time()
    for n, (cond, t, arm, rep) in enumerate(todo):
        env = make_env(suite, t["task_id"])
        try:
            fr = full_rows.get((t["task_id"], cond["name"], rep)) if arm == "gated" else None
            ok, steps, chunk, info = run_episode(adapter, env, suite, t, cond, arm, nominals[t["scene"]],
                                                 seed_offset=1000 * rep, gate_skip=fr is not None)
            if ok is None:                                  # the gate vetoed nothing: same episode as `full`
                ok, steps, chunk = fr["success"], fr["steps"], fr.get("first_chunk")
                info["copied_from_full"] = True
        finally:
            env.close()
        if cond["name"] == "nominal" and arm == "raw":
            nominal_chunk[t["task_id"]] = chunk
        probe = None
        if arm == "raw" and chunk is not None and t["task_id"] in nominal_chunk:
            probe = float(np.linalg.norm(np.subtract(chunk, nominal_chunk[t["task_id"]])))
        row = dict(policy=adapter.name, task_id=t["task_id"], base=t["base"], cond=cond["name"],
                   family=cond["family"], level=cond["level"], override=cond["override"], arm=arm, rep=rep,
                   success=bool(ok), steps=steps, first_chunk=chunk, probe=probe, **info)
        append_row(path, row)
        if arm == "full":
            full_rows[(t["task_id"], cond["name"], rep)] = row
        el = (time.time() - t0) / 60
        print(f"{adapter.name} {t['task_id']:5d} {cond['name']:<14} {arm:<5} success={ok!s:<5} veto={info.get('gate_vetoes', '-')} "
              f"cam={info.get('cam_applied')} photo={info.get('photo_applied')} "
              f"denoise={len(info.get('denoise_applied', []))} | {n + 1}/{len(todo)} "
              f"eta {el / (n + 1) * (len(todo) - n - 1):.0f} min", flush=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("mode", choices=["score", "episodes", "list"])
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--nominals", required=True)
    ap.add_argument("--n_tasks", type=int, default=3)
    ap.add_argument("--family", default="oft", choices=["oft", "openvla", "octo", "pi05", "pi0", "smolvla"])
    ap.add_argument("--octo_calib", default="")
    ap.add_argument("--arms", default="raw,full")
    ap.add_argument("--conditions", default="", help="comma-separated condition names (default: all)")
    ap.add_argument("--extended", action="store_true", help="also run the boundary conditions (sweep_conditions.EXTENDED)")
    ap.add_argument("--shard", default="auto", help="per-session results file episodes_<policy>.<shard>.jsonl (readers merge the "
                    "shards); auto = this runtime's hostname suffix, empty = the one shared file (never with two sessions)")
    ap.add_argument("--task_ids", default="", help="comma-separated base task ids: keep only these of the first --n_tasks "
                    "base tasks (a post hoc extension picks its tasks by a rule written in TASKS.md before it runs)")
    ap.add_argument("--repeats", type=int, default=1, help="episodes per (condition, task, arm) with different seeds")
    a = ap.parse_args()

    from libero.libero import benchmark
    suite = benchmark.get_benchmark_dict()[SUITE]()
    tasks = S.base_tasks(suite, a.n_tasks)
    if a.task_ids:
        keep = {int(x) for x in a.task_ids.split(",") if x.strip()}
        tasks = [t for t in tasks if t["task_id"] in keep]
        assert tasks, f"--task_ids {a.task_ids}: none of them is among the first {a.n_tasks} base tasks"
    os.makedirs(a.out_dir, exist_ok=True)
    for t in tasks:
        print(f"base task {t['task_id']} [{t['scene']}]: {t['instruction']}", flush=True)
    if a.mode == "list":
        print(f"{len(S.CONDITIONS)} conditions:", [c["name"] for c in S.CONDITIONS])
        return
    nominals = W.read_nominals(suite, lambda i: make_env(suite, i), cache=a.nominals)
    (run_score if a.mode == "score" else run_episodes)(a, suite, tasks, nominals)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/scripts/sweep_analysis.py
"""Does a policy-free, self-measured score predict which perturbations the wrapper recovers, better than the category
label and a policy-internal probe?

Unit of analysis: (policy, base task, condition) where the policy succeeds on the unperturbed task and FAILS on the
perturbed one (raw). Outcome: the self-measured wrapper recovers it (full succeeds). Predictors:
  recover  RECOVERABILITY = ident x invert (pre-registered 2026-09-17 on OFT, to be tested on held-out families):
           ident   sigma_min(condition block, perturbed) / sigma_min(SAME block, unperturbed), clipped to [0, 1]:
                   can the robot still measure the shift?
           invert  can the wrapper's inversion reconstruct the training observation from what it measured?
                   camera: training-view coverage of the reprojection from the self-calibrated pose;
                   light: body shading consistency vs the unperturbed state (a global gain/offset is the only
                   lighting change the photometric channel inverts); noise: share of the added noise the denoiser
                   removes; joint: 1 (servoing restores the encoders exactly); scene content: 0 (no inversion exists).
  ident    the identifiability factor alone
  score    v1 relative identifiability (normalised by the CAMERA block for every family; kept for the record, it is
           the predictor that failed on OFT with AUROC 0.48)
  label    category label: camera, light, noise, joint -> invertible (1); background -> not (0)
  probe    policy-internal: minus the change of the first action chunk vs the unperturbed first chunk
Reported: AUROC per policy and pooled with bootstrap CIs; within-type Spearman across levels; the override
conditions; identifiable-but-unrecovered cases; and the paired-comparison noise floor (episodes whose outcome changed
although the wrapper applied nothing).
"""
import argparse
import glob
import json
import os
import sys
from collections import defaultdict

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from plus_common import load_rows, load_shards  # noqa: E402
from sweep_conditions import BY_NAME, INVERTIBLE_LABEL  # noqa: E402


def auroc(scores, labels):
    s, y = np.asarray(scores, float), np.asarray(labels, int)
    ok = ~np.isnan(s)
    s, y = s[ok], y[ok]
    pos, neg = s[y == 1], s[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return float("nan")
    allv = np.concatenate([pos, neg])
    ranks = np.argsort(np.argsort(allv, kind="mergesort"), kind="mergesort") + 1.0
    for v in np.unique(allv):                                   # average ranks for ties
        idx = allv == v
        ranks[idx] = ranks[idx].mean()
    return float((ranks[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg)))


def boot_ci(scores, labels, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    s, y = np.asarray(scores, float), np.asarray(labels, int)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(s), len(s))
        v = auroc(s[i], y[i])
        if not np.isnan(v):
            vals.append(v)
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))) if vals else (float("nan"),) * 2


def spearman(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    if len(x) < 3 or np.ptp(x) == 0 or np.ptp(y) == 0:
        return float("nan")
    return float(np.corrcoef(np.argsort(np.argsort(x)), np.argsort(np.argsort(y)))[0, 1])


def recoverability(row):
    """ident x invert from a scores_v2 row (see the module docstring). Every input is policy-free and self-measured."""
    ref = row.get("sigma_min_ref")
    if ref is None:
        ident = float("nan")
    elif ref <= 0:                                     # the body never responds to this block: unmeasurable, not unknown
        ident = 0.0
    else:
        ident = float(np.clip(row["sigma_min"] / ref, 0.0, 1.0))
    fam = row["family"]
    if fam in ("camera", "nominal"):
        invert = float(row.get("coverage", np.nan))
    elif fam == "joint":
        invert = 1.0
    elif fam == "noise":
        invert = float(row.get("noise_inv", np.nan))
    elif fam == "light":
        d, d0 = row.get("photo_d"), row.get("photo_d_ref")
        invert = float(min(d, d0) / max(d, d0)) if d and d0 and not (np.isnan(d) or np.isnan(d0)) and max(d, d0) > 0 else float("nan")
    else:                                              # scene content: the wrapper has no inversion for it
        invert = 0.0
    return dict(ident=ident, invert=invert, recover=ident * invert)


def load_scores(out_dir):
    """scores_v2 (own-block reference + invertibility components) when present, else the v1 file. Returns
    (rows by (task, cond), predictors by (task, cond), version)."""
    v2 = os.path.join(out_dir, "scores_v2.jsonl")
    if os.path.exists(v2) and load_rows(v2):
        rows = {(r["task_id"], r["cond"]): r for r in load_rows(v2)}
        scale = {tid: r["sigma_min"] for (tid, c), r in rows.items() if c == "nominal" and r["sigma_min"] > 0}
        pred = {}
        for k, r in rows.items():
            rc = recoverability(r)
            pred[k] = dict(score=r["sigma_min"] / scale[k[0]] if k[0] in scale else np.nan, **rc)
        return rows, pred, "v2"
    rows = {(r["task_id"], r["cond"]): r for r in load_rows(os.path.join(out_dir, "scores.jsonl"))}
    scale = {tid: r["sigma_min"] for (tid, c), r in rows.items() if c == "nominal" and r["sigma_min"] > 0}
    pred = {k: dict(score=r["sigma_min"] / scale[k[0]], ident=np.nan, invert=np.nan, recover=np.nan)
            for k, r in rows.items() if k[0] in scale}
    return rows, pred, "v1"


def applied_nothing(row):
    """True when the full arm left the observation untouched (no channel fired)."""
    return (not row.get("state_applied") and not row.get("cam_applied") and not row.get("photo_applied")
            and not row.get("denoise_applied"))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--exclude", default="octo", help="policies whose episodes are not evidence (comma separated)")
    a = ap.parse_args()
    scores, pred, version = load_scores(a.out_dir)
    print(f"scores: {version} ({len(scores)} rows)" + ("" if version == "v2" else
          "  WARNING: v1 normalises every family by the camera block; re-run the score cell for scores_v2"))
    rel = {k: p["score"] for k, p in pred.items()}
    report = {"policies": {}}
    pooled = defaultdict(list)
    for path in sorted({os.path.join(a.out_dir, os.path.basename(p).split(".")[0] + ".jsonl")
                        for p in glob.glob(os.path.join(a.out_dir, "episodes_*.jsonl"))}):
        rows = load_shards(path)                          # the policy's main file plus its per-session shards
        if not rows:
            continue
        pol = rows[0]["policy"]
        if pol in a.exclude.split(","):      # a policy that cannot do the clean task has no recovery to measure
            print(f"skipping {pol}: excluded")
            continue
        ep = {}
        for r in rows:                                    # repeats (rep) average into one row per (task, cond, arm)
            k = (r["task_id"], r["cond"], r["arm"])
            if k in ep:
                n = ep[k]["_n"]
                ep[k]["success"] = (ep[k]["success"] * n + float(r["success"])) / (n + 1)
                ep[k]["_n"] = n + 1
            else:
                ep[k] = dict(r, success=float(r["success"]), _n=1)
        for r in ep.values():
            r["success"] = r["success"] >= 0.5 if r["_n"] == 1 else r["success"]
        tasks = sorted({r["task_id"] for r in rows})
        nominal_ok = {t: ep.get((t, "nominal", "raw"), {}).get("success") for t in tasks}
        print(f"\n===== {pol} =====  unperturbed success on base tasks: "
              f"{sum(bool(v) for v in nominal_ok.values())}/{len(tasks)}")
        print(f"{'condition':<15}{'label':>6}{'recover':>8}{'ident':>7}{'invert':>7}{'v1':>7}{'raw':>6}{'full':>6}  n")
        nm = lambda v: float(np.nanmean(v)) if not np.all(np.isnan(v)) else float("nan")
        for cname, cond in BY_NAME.items():
            raw = [ep[(t, cname, 'raw')]["success"] for t in tasks if (t, cname, "raw") in ep]
            full = [ep[(t, cname, 'full')]["success"] for t in tasks if (t, cname, "full") in ep]
            pp = [pred.get((t, cname), {}) for t in tasks]
            if raw or full:
                print(f"{cname:<15}{INVERTIBLE_LABEL[cond['family']]:>6}"
                      f"{nm([p.get('recover', np.nan) for p in pp]):>8.3f}{nm([p.get('ident', np.nan) for p in pp]):>7.2f}"
                      f"{nm([p.get('invert', np.nan) for p in pp]):>7.2f}{nm([p.get('score', np.nan) for p in pp]):>7.3f}"
                      f"{np.mean(raw) if raw else float('nan'):>6.2f}{np.mean(full) if full else float('nan'):>6.2f}  {len(raw)}")
        flips = [(t, c) for (t, c, arm), r in ep.items() if arm == "full" and applied_nothing(r)
                 and (t, c, "raw") in ep and ep[(t, c, "raw")]["success"] != r["success"]]
        idle = [(t, c) for (t, c, arm), r in ep.items() if arm == "full" and applied_nothing(r) and (t, c, "raw") in ep]
        print(f"noise floor: {len(flips)} of {len(idle)} episodes changed outcome with NOTHING applied"
              + (f" -> {flips}" if flips else ""))
        units = []
        for t in tasks:
            if not nominal_ok.get(t):
                continue
            for cname, cond in BY_NAME.items():
                if cname == "nominal":
                    continue
                r, fu = ep.get((t, cname, "raw")), ep.get((t, cname, "full"))
                if r is None or fu is None or float(r["success"]) >= 0.5:
                    continue
                pp = pred.get((t, cname), {})
                units.append(dict(task=t, cond=cname, family=cond["family"], override=cond["override"],
                                  recovered=int(float(fu["success"]) >= 0.5), score=rel.get((t, cname), np.nan),
                                  recover=pp.get("recover", np.nan), ident=pp.get("ident", np.nan),
                                  invert=pp.get("invert", np.nan), label=INVERTIBLE_LABEL[cond["family"]],
                                  probe=-r["probe"] if r.get("probe") is not None else np.nan))
        y = [u["recovered"] for u in units]
        res = {"n_units": len(units), "recovered": int(sum(y))}
        print(f"\nprediction units (unperturbed success, perturbed raw failure): {len(units)}, recovered {sum(y)}")
        for name in ("recover", "ident", "score", "label", "probe"):
            x = [u[name] for u in units]
            if np.all(np.isnan(np.asarray(x, float))):
                continue
            au, ci = auroc(x, y), boot_ci(x, y)
            res[name] = dict(auroc=au, ci=ci)
            pooled[name] += list(zip(x, y))
            print(f"  AUROC {name:<8} {au:.3f}  [{ci[0]:.2f}, {ci[1]:.2f}]")
        use = "recover" if version == "v2" else "score"
        print(f"\nwithin-type: Spearman({use}, full success - raw success) across levels and tasks")
        for fam in ("camera", "light", "noise", "joint", "background"):
            xs, ds = [], []
            for t in tasks:
                for cname, cond in BY_NAME.items():
                    if cond["family"] != fam or (t, cname, "raw") not in ep or (t, cname, "full") not in ep:
                        continue
                    xs.append(pred.get((t, cname), {}).get(use, np.nan))
                    ds.append(int(ep[(t, cname, "full")]["success"]) - int(ep[(t, cname, "raw")]["success"]))
            ok = ~np.isnan(np.asarray(xs, float))
            rho = spearman(np.asarray(xs)[ok], np.asarray(ds)[ok]) if ok.sum() else float("nan")
            # the Spearman is dominated by episodes that never failed (diff 0); the units-only AUROC within the
            # family is the number a reviewer asks for ("does it predict within a perturbation type?")
            fu = [u for u in units if u["family"] == fam and not np.isnan(u[use])]
            au_f = auroc([u[use] for u in fu], [u["recovered"] for u in fu]) if fu else float("nan")
            res.setdefault("within_type", {})[fam] = dict(rho=rho, auroc=au_f, units=len(fu),
                                                         recovered=int(sum(u["recovered"] for u in fu)))
            print(f"  {fam:<11} rho {rho:+.2f}  (n {int(ok.sum())})   units {len(fu):>2}, recovered "
                  f"{int(sum(u['recovered'] for u in fu)):>2}, AUROC {au_f:.3f}")
        print("\noverride conditions: label call vs score vs outcome")
        for u in [u for u in units if u["override"]]:
            print(f"  task {u['task']} {u['cond']:<14} label {u['label']} recover {u['recover']:.3f} "
                  f"(ident {u['ident']:.2f} invert {u['invert']:.2f}) recovered {u['recovered']}")
        hi = [u for u in units if not np.isnan(u[use]) and u[use] > 0.5 and not u["recovered"]]
        print(f"\npredicted recoverable ({use} > 0.5) but NOT recovered: {len(hi)} -> "
              f"{sorted({u['cond'] for u in hi})}")
        lo = [u for u in units if not np.isnan(u[use]) and u[use] <= 0.5 and u["recovered"]]
        print(f"predicted NOT recoverable ({use} <= 0.5) but recovered: {len(lo)} -> {sorted({u['cond'] for u in lo})}")
        res["units"] = units
        report["policies"][pol] = res
    if pooled:
        print("\n===== pooled over policies =====")
        for name, xy in pooled.items():
            x, yy = [p[0] for p in xy], [p[1] for p in xy]
            au, ci = auroc(x, yy), boot_ci(x, yy)
            report[f"pooled_{name}"] = dict(auroc=au, ci=ci, n=len(xy))
            print(f"  AUROC {name:<8} {au:.3f}  [{ci[0]:.2f}, {ci[1]:.2f}]  (n {len(xy)})")
    json.dump(report, open(os.path.join(a.out_dir, "sweep_report.json"), "w"), indent=1, default=float)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/scripts/wrap_run.py
"""The self-measured wrapper on ANY policy family, over the shared LIBERO-Plus subset (claim 1 across families).

For each family adapter (oft, openvla, octo, pi05, pi0) and each subset task, arms:
  raw           the perturbed task through the same rollout loop, no wrapper (paired baseline)
  full          deployable wrapper, simulator stand-ins for robot segmentation and depth
  full_learned  deployable wrapper with LEARNED sensing: a trained robot segmenter + monocular depth
                (Depth Anything) scaled on the robot's own pixels (needs --seg_ckpt)
  full_plane    deployable wrapper with the learned segmenter and NO depth network: the training setup's table
                plane meets the calibrated camera's rays; the robot's pixels take its known geometry (needs --seg_ckpt)
  full_gated    full plus the do-no-harm gate (sweep_run.gate_decide): a channel is vetoed when the frozen policy is
                less self-consistent on the corrected frame than on the raw one. Run AFTER `full` in the same --arms list:
                an episode in which nothing is vetoed is identical to `full` and is copied, not re-run.
  BASELINES (OFT / OpenVLA; pi0.5 keeps its own action queue, so `tta` is not defined for it):
  tta           the raw policy, its action chunk averaged over the raw frame and four 2 px shifts of it
  generic       gray-world white balance + CLAHE + denoise on every frame, nothing measured, no abstention
  full_homog    full, but the camera channel is a depth-free plane homography with inpainting (calibration alone)
  full_settle   full, but the measurement sweep ends with the arm at the target and AT REST (plain `full` hands over
                inside 4 mm and still moving, which alone can flip a chaotic policy's outcome)
  full_verify   full_settle, plus: when the robot's own self-check rejects the calibration (F1 < 0.75), measure again
                with the 7-pose sweep and keep the calibration the silhouette agrees with more
  full_tta      full, with the policy's actions averaged over benign jitters of the CORRECTED frame (wrapper + tta)
  full_nofill   full, but reprojection holes are inpainted instead of filled from the training-view reference
  full_hybrid   full_plane plus the training setup's static geometry and the held-object rule (needs --seg_ckpt)
  nominal_view  DIAGNOSTIC base rate for camera tasks: every frame is the simulator's render from the TRAINING camera, the
                arm is untouched and nothing is measured; the unperturbed task at the same layout and seed
  full_metric   POST HOC: full with LEARNED sensing: the trained segmenter plus a monocular METRIC depth network
                (Depth Anything V2 metric, used as is, no fit to the robot's geometry); needs --seg_ckpt. Tests whether
                learned sensing with metric depth brings self-calibration inside the policy's ~1 cm tolerance
  nominal_start DIAGNOSTIC base rate for robot_init: the arm is placed at the training start (no servo, no correction),
                then the raw policy runs; the success an ideal joint reset can reach on the same layout and seed
  full_refine   POST HOC (TASKS.md P47, pose-source ablation): full, but the silhouette pose is refined EasyHeC-style
                (dense soft-mask loss, Adam from the same initial pose, finite-difference gradients; pose_sources.py)
  full_fiducial POST HOC (P47): full, but the pose comes from PnP on four table fiducials with oracle (never occluded)
                corners and 0.5 px corner noise, the AnyCamVLA pose source (pose_sources.py)
  *_truepose    DIAGNOSTIC upper bounds (full_truepose, full_plane_truepose, full_hybrid_truepose): same sensing, but
                the reprojection uses the simulator's camera pose. Separates depth error from calibration error.
Rows are written in the invertibility_report format (arm, task_id, category, success), so
  invertibility_report.py --family NAME:wrap_NAME.jsonl:wrap_NAME.jsonl
uses the paired raw rows as the baseline.
"""
import argparse
import json
import os
import sys
import time

import numpy as np

os.environ.setdefault("MUJOCO_GL", "egl")
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import sweep_conditions as S  # noqa: E402
import sweep_run as R  # noqa: E402
import wrapper_libero as W  # noqa: E402
from calib_check import make_env  # noqa: E402
from plus_common import append_row, load_rows  # noqa: E402

PER_CAT = {"camera": 30, "robot_init": 30, "light": 10, "sensor_noise": 10, "layout": 10, "background": 10, "language": 10}
POSE_SOURCE = {"full_refine": "refine", "full_fiducial": "fiducial"}      # POST HOC P47 (pose_sources.py)
PRIORITY = ["camera", "robot_init", "light", "sensor_noise", "layout", "background", "language"]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--family", required=True, choices=["oft", "openvla", "octo", "pi05", "pi0", "smolvla"])
    ap.add_argument("--subset", required=True)
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--nominals", required=True)
    ap.add_argument("--arms", default="raw,full")
    ap.add_argument("--per_cat_scale", type=float, default=1.0, help="multiply PER_CAT (e.g. 0.34 for a cheap pass)")
    ap.add_argument("--categories", default=",".join(PRIORITY))
    ap.add_argument("--octo_calib", default="")
    ap.add_argument("--seg_ckpt", default="")
    ap.add_argument("--rep", type=int, default=0,
                    help="replicate index: episode seeds become task_id + 1000 * rep (start state, noise and the policy's "
                         "sampler all change). Give every replicate its OWN --out_dir; rep 0 is the main table")
    ap.add_argument("--tasks", default="", help="comma-separated task ids: run only these (diagnostics)")
    ap.add_argument("--only_failed_in", default="",
                    help="a finished wrap file: run ONLY the tasks that fail there under BOTH raw and full (the units the "
                         "attribution ladder needs upper rungs for); arms already present in that file are not repeated. "
                         "Use with an --out_dir of its own")
    ap.add_argument("--redo", action="store_true",
                    help="with --tasks: drop the existing rows of these tasks for the requested arms that carry no "
                         "cam_est_err_cm_deg (made before the diagnostics were logged), then run them again")
    ap.add_argument("--extra_from_suite", type=int, default=0,
                    help="add N more tasks per category from the full suite, outside every evaluation subset (seed 1)")
    ap.add_argument("--shard", default="auto",
                    help="per-session results shard (wrap_<family>.<shard>.jsonl) so several runtimes can share one "
                         "--out_dir; auto = hostname suffix, empty = the one shared file (never with two sessions)")
    ap.add_argument("--reverse", action="store_true",
                    help="walk the task list from the end: give the SECOND runtime on a shared --out_dir this flag")
    ap.add_argument("--rotate", type=float, default=0.0,
                    help="start this fraction of the way into the task list (a THIRD runtime: --rotate 0.5)")
    a = ap.parse_args()

    from libero.libero import benchmark
    from policy_adapters import make_adapter
    suite = benchmark.get_benchmark_dict()[R.SUITE]()
    print(f"suite {R.SUITE}, horizon {R.MAX_STEPS} steps", flush=True)
    from plus_common import load_or_build
    sub = load_or_build(a.subset)                      # built for the suite in LIBERO_SUITE when the file does not exist
    nominals = W.read_nominals(suite, lambda i: make_env(suite, i), cache=a.nominals)
    adapter = make_adapter(a.family, a.octo_calib)
    sensings, by_mode = {}, {}
    for arm in a.arms.split(","):
        mode = {"full_learned": "mono", "full_plane": "plane", "full_hybrid": "hybrid", "full_metric": "metric"}.get(
            arm.replace("_truepose", "").replace("_refine2", ""))
        if mode:
            assert a.seg_ckpt and os.path.exists(a.seg_ckpt), f"{arm} needs --seg_ckpt (train_seg.py output)"
            if mode not in by_mode:
                by_mode[mode] = W.LearnedSensing(a.seg_ckpt, depth=mode)
            sensings[arm] = by_mode[mode]
    if "hybrid" in by_mode:
        W.read_static_clouds(suite, lambda i: make_env(suite, i), nominals, cache=a.nominals)
    only = {int(x) for x in a.tasks.split(",") if x.strip()}
    os.makedirs(a.out_dir, exist_ok=True)
    base = os.path.join(a.out_dir, f"wrap_{adapter.name}.jsonl")
    import socket
    from plus_common import load_shards
    shard = socket.gethostname()[-6:] if a.shard == "auto" else a.shard
    path = base[:-len(".jsonl")] + f".{shard}.jsonl" if shard else base      # this session appends ONLY to its own shard
    claims = os.path.join(a.out_dir, "claims")
    os.makedirs(claims, exist_ok=True)
    print(f"results shard: {os.path.basename(path)} (rows and claims of every other session count as taken)", flush=True)
    rows = load_shards(base)
    stale = [r for r in rows if r.get("photo_applied") and "photo_shading" not in r]
    if stale:                                             # photometric map applied before the shading gate: redo
        with open(path + ".before_shading_gate", "w") as f:
            f.writelines(json.dumps(r) + "\n" for r in rows)
        rows = [r for r in rows if not (r.get("photo_applied") and "photo_shading" not in r)]
        with open(path, "w") as f:
            f.writelines(json.dumps(r) + "\n" for r in rows)
        print(f"re-queued {len(stale)} episodes whose photometric map predates the shading gate", flush=True)
    if a.redo and only:
        old = [r for r in rows if r["task_id"] in only and r["arm"] in a.arms.split(",") and r["arm"] != "raw"
               and "cam_est_err_cm_deg" not in r]
        if old:
            with open(path + ".before_redo", "a") as f:
                f.writelines(json.dumps(r) + "\n" for r in old)
            rows = [r for r in rows if r not in old]
            with open(path, "w") as f:
                f.writelines(json.dumps(r) + "\n" for r in rows)
            print(f"redo: moved {len(old)} rows without calibration diagnostics to {os.path.basename(path)}.before_redo", flush=True)
    done = {(r["task_id"], r["arm"]) for r in rows}
    if a.only_failed_in:
        prev = [r for r in load_rows(a.only_failed_in) if r.get("rep", 0) == a.rep]
        assert prev, f"--only_failed_in: no rows in {a.only_failed_in}"
        ok = lambda arm: {r["task_id"] for r in prev if r["arm"] == arm and r["success"]}
        ran = lambda arm: {r["task_id"] for r in prev if r["arm"] == arm}
        only = ((ran("raw") - ok("raw")) & (ran("full") - ok("full"))) & (only or ran("raw"))
        done |= {(r["task_id"], r["arm"]) for r in prev}
        print(f"--only_failed_in: {len(only)} tasks fail under both raw and full in {os.path.basename(a.only_failed_in)}", flush=True)
        if not only:
            return
    full_rows = {r["task_id"]: r for r in rows if r["arm"] == "full"}
    nominal_cond = S.BY_NAME["nominal"]
    extra = {}
    if a.extra_from_suite:
        from plus_common import categorize
        taken = {t["task_id"] for t in sub["tasks"]}
        small = a.subset.replace("subset_libero10.json", "subset_libero10_small.json")
        if os.path.exists(small):
            taken |= {t["task_id"] for t in json.load(open(small))["tasks"]}
        rng = np.random.default_rng(1)
        for cat in a.categories.split(","):
            pool = [i for i in range(suite.n_tasks) if i not in taken and categorize(suite.get_task(i).name) == cat]
            extra[cat] = [dict(task_id=int(i), name=suite.get_task(int(i)).name, language=suite.get_task(int(i)).language,
                               category=cat) for i in sorted(rng.permutation(pool)[:a.extra_from_suite].tolist())]
    todo = []
    for cat in a.categories.split(","):
        k = int(round(PER_CAT[cat] * a.per_cat_scale)) if a.per_cat_scale > 0 else 0
        for t in [x for x in sub["tasks"] if x["category"] == cat][:k] + extra.get(cat, []):
            for arm in a.arms.split(","):
                if (t["task_id"], arm) not in done and (not only or t["task_id"] in only):
                    todo.append((t, arm))
    if a.reverse:
        todo = todo[::-1]
    if a.rotate:                        # a THIRD runtime on the same folder starts part-way in (e.g. 0.5 = the middle)
        k = int(len(todo) * a.rotate) % max(len(todo), 1)
        todo = todo[k:] + todo[:k]
    print(f"{adapter.name}: {len(todo)} episodes to run", flush=True)
    t0 = time.time()

    def taken(t, arm):
        """Done in ANY session's shard, or claimed by another session (claims/<task>_<arm>_<rep>.<shard>). Drive syncs
        with a lag, so two sessions can rarely both run one episode: harmless (separate shards, readers keep one row)."""
        key = f"{t['task_id']}_{arm}_{a.rep}"
        if any(r["task_id"] == t["task_id"] and r["arm"] == arm and r.get("rep", 0) == a.rep for r in load_shards(base)):
            return True
        now = time.time()                    # a claim older than 30 min is a dead session's (one episode takes minutes)
        return any(f.startswith(key + ".") and not f.endswith("." + (shard or "main"))
                   and now - os.path.getmtime(os.path.join(claims, f)) < 1800 for f in os.listdir(claims))

    for n, (t, arm) in enumerate(todo):
        if taken(t, arm):
            print(f"{adapter.name} {t['task_id']:5d} {arm:<12} taken by another session, skipped", flush=True)
            continue
        open(os.path.join(claims, f"{t['task_id']}_{arm}_{a.rep}.{shard or 'main'}"), "w").close()
        task = dict(task_id=t["task_id"], instruction=t["language"], scene=W.scene_of(t["name"]))
        env = make_env(suite, t["task_id"])
        try:
            fr = full_rows.get(t["task_id"]) if arm == "full_gated" else None
            ok, steps, chunk, info = R.run_episode(adapter, env, suite, task, nominal_cond,
                                                   {"full_gated": "gated", "full_lazy": "lazy", "full_oracle": "oracle", "full_homog": "homog", "full_nofill": "nofill", "full_settle": "settle", "full_verify": "verify", "full_tta": "full_tta",
                                                    "tta": "tta", "generic": "generic", "nominal_start": "nominal_start", "nominal_view": "nominal_view"}.get(
                                                       arm, "full" if arm.startswith("full") else "raw"),
                                                   nominals[task["scene"]],
                                                   sensing=sensings.get(arm),
                                                   pose="true" if arm.endswith("_truepose") else
                                                   ("refine2" if arm.endswith("_refine2") else POSE_SOURCE.get(arm)),
                                                   gate_skip=fr is not None, seed_offset=1000 * a.rep)
            if ok is None:                                  # the gate vetoed nothing: same episode as `full`
                ok, steps = fr["success"], fr["steps"]
                info["copied_from_full"] = True
        finally:
            env.close()
        row = dict(family=adapter.name, task_id=t["task_id"], name=t["name"], category=t["category"], arm=arm,
                   rep=a.rep, success=bool(ok), steps=steps, **{k: v for k, v in info.items() if k != "first_chunk"})
        append_row(path, row)
        if arm == "full":
            full_rows[t["task_id"]] = row
        el = (time.time() - t0) / 60
        print(f"{adapter.name} {t['task_id']:5d} {t['category']:<12} {arm:<12} success={ok!s:<5} "
              f"state={info.get('state_applied', False)!s:<5} cam={info.get('cam_applied')} "
              f"photo={info.get('photo_applied')} veto={info.get('gate_vetoes', '-')} | {n + 1}/{len(todo)} eta {el / (n + 1) * (len(todo) - n - 1):.0f} min",
              flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/scripts/train_seg.py
"""Train the robot segmenter used by the learned-sensing wrapper (replaces the renderer's perfect robot mask).

Data: collect_canon.py shards (held-out LIBERO-Plus tasks, many camera poses and arm configurations), which store the
observed frame `x` and the robot mask `r` in observation orientation. As with any deployed robot segmenter
(e.g. CtRNet), labels come from rendering the robot's known geometry in simulation at training time; nothing about
the evaluation tasks is used. Reports IoU on held-out tasks.
"""
import argparse
import glob
import json
import os
import sys

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from canon_net import block  # noqa: E402


class SegNet(nn.Module):
    def __init__(self, width=24):
        super().__init__()
        ch = [width, 2 * width, 4 * width, 8 * width]
        self.enc = nn.ModuleList()
        prev = 3
        for c in ch:
            self.enc.append(block(prev, c))
            prev = c
        self.dec = nn.ModuleList()
        for c in reversed(ch[:-1]):
            self.dec.append(block(prev + c, c))
            prev = c
        self.head = nn.Conv2d(prev, 1, 1)

    def forward(self, x):
        skips, h = [], x
        for i, b in enumerate(self.enc):
            h = b(h)
            if i < len(self.enc) - 1:
                skips.append(h)
                h = F.max_pool2d(h, 2)
        for b, s in zip(self.dec, reversed(skips)):
            h = F.interpolate(h, size=s.shape[-2:], mode="bilinear", align_corners=False)
            h = b(torch.cat([h, s], 1))
        return self.head(h)


def load(path, device="cuda"):
    ck = torch.load(path, map_location="cpu")
    net = SegNet(**ck.get("net_args", {}))
    net.load_state_dict(ck["state_dict"])
    return net.to(device).eval()


@torch.no_grad()
def predict(net, img_uint8, device="cuda", thr=0.5):
    x = torch.from_numpy(np.ascontiguousarray(img_uint8)).to(device).permute(2, 0, 1)[None].float() / 255.0
    return (torch.sigmoid(net(x))[0, 0] > thr).cpu().numpy().astype(np.uint8)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--epochs", type=int, default=20)
    ap.add_argument("--bs", type=int, default=16)
    ap.add_argument("--lr", type=float, default=1e-3)
    a = ap.parse_args()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0)
    X, Rm, T = [], [], []
    for p in sorted(glob.glob(os.path.join(a.data_dir, "task_*.npz"))):
        d = np.load(p)
        if "r" not in d:
            continue
        X.append(d["x"]); Rm.append(d["r"]); T += [json.loads(str(d["meta"]))["task_id"]] * len(d["x"])
    assert X, "no shards with robot masks (`r`): re-run collect_canon.py (it now stores them)"
    X, Rm, T = np.concatenate(X), np.concatenate(Rm), np.array(T)
    rng = np.random.default_rng(0)
    tasks = np.unique(T)
    val = set(rng.permutation(tasks)[:max(1, len(tasks) // 7)].tolist())
    va = np.array([t in val for t in T])
    print(f"{len(X)} frames, {len(tasks)} tasks, val tasks {len(val)}, robot pixel fraction {Rm.mean():.3f}", flush=True)
    net = SegNet().to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=a.lr, weight_decay=1e-4)
    pos_w = torch.tensor([(1 - Rm.mean()) / max(Rm.mean(), 1e-3)], device=dev).clamp(max=10.0)
    tr = np.where(~va)[0]
    assert len(tr) >= a.bs, f"only {len(tr)} training frames for batch size {a.bs}: collect more shards"
    best, loss = -1.0, torch.tensor(float("nan"))
    for ep in range(a.epochs):
        net.train()
        rng.shuffle(tr)
        for s in range(0, len(tr) - a.bs + 1, a.bs):
            b = tr[s:s + a.bs]
            x = torch.from_numpy(X[b]).to(dev).permute(0, 3, 1, 2).float() / 255.0
            y = torch.from_numpy(Rm[b]).to(dev).float()[:, None]
            if rng.random() < 0.5:                                   # photometric augmentation: gain and noise
                x = (x * torch.empty(len(b), 3, 1, 1, device=dev).uniform_(0.6, 1.4)
                     + torch.randn_like(x) * float(rng.uniform(0, 0.08))).clamp(0, 1)
            loss = F.binary_cross_entropy_with_logits(net(x), y, pos_weight=pos_w)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        net.eval()
        inter = union = 0.0
        with torch.no_grad():
            for i in np.where(va)[0]:
                x = torch.from_numpy(X[i:i + 1]).to(dev).permute(0, 3, 1, 2).float() / 255.0
                pr = torch.sigmoid(net(x))[0, 0].cpu().numpy() > 0.5
                gt = Rm[i] > 0
                inter += (pr & gt).sum(); union += (pr | gt).sum()
        iou = inter / max(union, 1)
        print(f"epoch {ep + 1:3d} loss {float(loss):.4f} val IoU {iou:.3f}", flush=True)
        if iou > best:
            best = iou
            os.makedirs(os.path.dirname(a.out) or ".", exist_ok=True)
            torch.save(dict(state_dict=net.state_dict(), net_args={}, val_iou=float(iou), epoch=ep + 1), a.out)
    print(f"best val IoU {best:.3f} -> {a.out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/scripts/camera_pooled.py
"""Pooled paired test for the camera claim: does the DEPLOYABLE wrapper beat the unwrapped policy on camera tasks?

Pairs come from two sources, never mixed within a pair:
  subset tasks     baseline rows_oft.jsonl (unwrapped) vs the wrapper run rows.jsonl (arm, default `full`)
  extension tasks  wrap_run rows (wrap_openvla_oft.jsonl): raw vs full on camera tasks outside the subset
Reports rates, Wilson CIs and an exact McNemar test on the pooled discordant pairs, plus the upper-bound arms.
"""
import argparse
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from invertibility_report import mcnemar_p  # noqa: E402
from plus_common import load_rows, wilson  # noqa: E402


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--baseline", required=True)
    ap.add_argument("--wrapper", required=True)
    ap.add_argument("--ext", default="")
    ap.add_argument("--category", default="camera")
    a = ap.parse_args()
    base = {r["task_id"]: bool(r["success"]) for r in load_rows(a.baseline)}
    wrap = {}
    for r in load_rows(a.wrapper):
        if r["category"] == a.category:
            wrap.setdefault(r["arm"], {})[r["task_id"]] = bool(r["success"])
    ext = {}
    for r in (load_rows(a.ext) if a.ext and os.path.exists(a.ext) else []):
        if r["category"] == a.category:
            ext.setdefault(r["arm"], {})[r["task_id"]] = bool(r["success"])

    def report(name, pairs):
        if not pairs:
            return
        n = len(pairs)
        b0, b1 = sum(x for x, _ in pairs), sum(y for _, y in pairs)
        rec = sum(1 for x, y in pairs if not x and y)
        harm = sum(1 for x, y in pairs if x and not y)
        lo0, hi0 = wilson(b0, n)
        lo1, hi1 = wilson(b1, n)
        print(f"{name:<34} n={n:>3}  unwrapped {b0 / n:.2f} [{lo0:.2f},{hi0:.2f}]  arm {b1 / n:.2f} [{lo1:.2f},{hi1:.2f}]  "
              f"recovered {rec} harmed {harm}  McNemar p={mcnemar_p(rec, harm):.4f}")

    print(f"category {a.category}: paired against the unwrapped policy on the same tasks")
    for arm in ("full", "full2", "cam_true_ref", "cam_true", "oracle", "learned", "learned_plane"):
        if arm in wrap:
            report(f"subset: {arm}", [(base[t], ok) for t, ok in wrap[arm].items() if t in base])
    ext_pairs = [(ext["raw"][t], ok) for t, ok in ext.get("full", {}).items() if t in ext.get("raw", {})]
    report("extension: full (raw paired)", ext_pairs)
    sub_full = [(base[t], ok) for t, ok in wrap.get("full", {}).items() if t in base]
    report("POOLED deployable wrapper", sub_full + ext_pairs)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/scripts/diag_learned.py
"""Learned-sensing diagnostics from wrap_run rows: per task and arm, success, TRUE calibration error, learned-mask
quality and reprojection coverage; then the 2x2 {calibrated pose, true pose} x {depth mode} success table.

    python diag_learned.py --rows <wraps>/wrap_openvla_oft.jsonl [--tasks 768,800,...]

Reading guide (2026-09-18 root-cause analysis):
  cam_est_err above ~5 cm on a lost task      -> calibration under the learned mask is the cause there
  mask_area_ratio away from 1.00 (uniformly)  -> the segmenter's boundary bias moves the camera along its axis
  *_truepose arm succeeds, deployable fails    -> calibration; both fail but sim-depth truepose succeeds -> depth
"""
import argparse
import json
from collections import defaultdict

ap = argparse.ArgumentParser()
ap.add_argument("--rows", required=True)
ap.add_argument("--tasks", default="")
ap.add_argument("--category", default="camera")
a = ap.parse_args()
only = {int(x) for x in a.tasks.split(",") if x.strip()}
rows = [json.loads(l) for l in open(a.rows) if l.strip()]
rows = [r for r in rows if r.get("category") == a.category and r["arm"].startswith("full") and (not only or r["task_id"] in only)]
last = {}
for r in rows:
    last[(r["task_id"], r["arm"])] = r
arms = sorted({k[1] for k in last})
print(f"{'task':>5} {'arm':<22} ok   est err cm/deg   true shift cm/deg  mask IoU  area  coverage  cal method")
for (tid, arm), r in sorted(last.items()):
    e, s = r.get("cam_est_err_cm_deg"), r.get("cam_true_shift_cm_deg")
    f = lambda v, n=2: "    -" if v is None else f"{v:5.{n}f}"
    print(f"{tid:>5} {arm:<22} {'Y' if r['success'] else '.'}   "
          f"{f(e and e[0], 1)} /{f(e and e[1], 1)}     {f(s and s[0], 1)} /{f(s and s[1], 1)}     {f(r.get('mask_iou'))} {f(r.get('mask_area_ratio'))}"
          f"    {f(r.get('reproj_coverage'))}  {r.get('cal_method', '')}")
print()
by = defaultdict(list)
for (tid, arm), r in last.items():
    by[arm].append(r)
common = set.intersection(*[{r["task_id"] for r in v} for v in by.values()]) if by else set()
print(f"success per arm (all rows | the {len(common)} tasks every arm has run)")
for arm in arms:
    v = by[arm]
    c = [r for r in v if r["task_id"] in common]
    errs = sorted(r["cam_est_err_cm_deg"][0] for r in v if r.get("cam_est_err_cm_deg"))
    med = f", median est err {errs[len(errs) // 2]:.1f} cm, above 5 cm on {sum(x > 5 for x in errs)}/{len(errs)}" if errs else ""
    print(f"  {arm:<22} {sum(r['success'] for r in v):>2}/{len(v):<3} | {sum(r['success'] for r in c):>2}/{len(c):<3}{med}")

In [ ]:
%%writefile /content/scripts/gate_report.py
"""The do-no-harm gate against the ungated wrapper, from wrap_run rows (arms raw / full / full_gated) or sweep episodes
(arms raw / full / gated). Pre-registered 2026-09-18 (RESULTS.md):

  H1 (harm)       among units the raw policy solves, the gated wrapper breaks at most HALF as many as the ungated one
                  (testable only if the ungated wrapper breaks at least 3)
  H2 (retention)  among units the raw policy fails, the gated wrapper recovers at least 80 percent of what the ungated
                  one recovers

    python gate_report.py --rows <file.jsonl> [--rows <file2.jsonl> ...]
"""
import argparse
import json
import os
import sys
from collections import defaultdict

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from plus_common import load_shards  # noqa: E402

ap = argparse.ArgumentParser()
ap.add_argument("--rows", action="append", required=True)
a = ap.parse_args()

for path in a.rows:
    rows = load_shards(path)                              # the file plus its per-session shards (plus_common)
    units = defaultdict(dict)
    for r in rows:
        arm = {"full_gated": "gated"}.get(r["arm"], r["arm"])
        if arm in ("raw", "full", "gated"):
            units[(r["task_id"], r.get("cond", ""), r.get("rep", 0))][arm] = r
    full3 = {k: v for k, v in units.items() if {"raw", "full", "gated"} <= set(v)}
    print(f"\n===== {path} =====  {len(full3)} units with raw, full and gated ({len(units)} units in the file)")
    if not full3:
        continue
    ok = lambda v, arm: bool(v[arm]["success"])
    fail_raw = [v for v in full3.values() if not ok(v, "raw")]
    succ_raw = [v for v in full3.values() if ok(v, "raw")]
    rec_f, rec_g = sum(ok(v, "full") for v in fail_raw), sum(ok(v, "gated") for v in fail_raw)
    brk_f, brk_g = sum(not ok(v, "full") for v in succ_raw), sum(not ok(v, "gated") for v in succ_raw)
    print(f"raw fails     {len(fail_raw):>3}: recovered by full {rec_f}, by gated {rec_g}")
    print(f"raw succeeds  {len(succ_raw):>3}: broken by full {brk_f}, by gated {brk_g}")
    h1 = "untestable (the ungated wrapper breaks fewer than 3)" if brk_f < 3 else ("HOLDS" if brk_g <= brk_f / 2 else "FAILS")
    h2 = "untestable (nothing recovered by full)" if rec_f == 0 else ("HOLDS" if rec_g >= 0.8 * rec_f else "FAILS")
    print(f"H1 harm: {h1} | H2 retention: {h2}")
    vetoed = [(k, v) for k, v in full3.items() if v["gated"].get("gate_vetoes")]
    by_ch = defaultdict(int)
    for _, v in vetoed:
        for c in v["gated"]["gate_vetoes"]:
            by_ch[c] += 1
    wanted = sum(bool(v["gated"].get("wanted_cam") or v["gated"].get("wanted_photo") or v["gated"].get("wanted_denoise"))
                 for v in full3.values())
    print(f"gate: a channel fired in {wanted} units, vetoed in {len(vetoed)} {dict(by_ch)}")
    print(f"{'task':>5} {'cond':<14} raw full gated  vetoed        s_raw -> s_cor (worse of 6)")
    for (tid, cond, rep), v in sorted(vetoed):
        g = v["gated"]
        det = "  ".join(f"{c}: {g['gate_' + c]['s_raw']:.3f} -> {g['gate_' + c]['s_cor']:.3f} ({g['gate_' + c]['worse']})"
                        for c in g["gate_vetoes"])
        print(f"{tid:>5} {cond:<14} {'Y' if ok(v, 'raw') else '.':>3} {'Y' if ok(v, 'full') else '.':>4} "
              f"{'Y' if ok(v, 'gated') else '.':>5}  {','.join(g['gate_vetoes']):<12}  {det}")
    kept_bad = [(k, v) for k, v in full3.items() if ok(v, "raw") and not ok(v, "gated") and not v["gated"].get("gate_vetoes")]
    if kept_bad:
        print(f"harm the gate did NOT catch ({len(kept_bad)}):", [f"{k[0]}:{k[1]}" for k, _ in kept_bad])

In [ ]:
%%writefile /content/scripts/smc.py
"""Self-motion consistency (SMC): a label-free, physically grounded test of whether a frozen visuomotor policy is
perceiving the scene correctly.

A goal-directed policy that commands end-effector displacements defines an action field over end-effector positions.
If the policy perceives the scene correctly, the rays `p_k + t * u_k` (p_k: end-effector positions the robot visits
and KNOWS from its encoders, u_k: the direction of the policy's commanded translation there) must meet at one point,
the policy's current sub-goal. If the observation is wrong in a way the policy cannot absorb (a moved camera, a bad
correction), the perceived displacement is a distorted function of the true one and the rays stop meeting. The
robot's own proprioception is the metric ground truth; no label, no simulator state and no model of the policy is used.

    residual_deg   RMS angle between each commanded direction and the direction to the triangulated sub-goal
    forward        share of probes whose sub-goal lies in FRONT of them along their own ray
    min_eig        smallest eigenvalue of the normal matrix / K: near 0 when the rays are parallel (no information)

Uses: (1) a policy-side, label-free harm test for any input correction (lower residual = the policy agrees with
itself more), (2) selection among candidate corrections, including "none", (3) calibration of nuisance parameters
with the frozen policy as the sensor.
"""
import numpy as np

# probe offsets from the nominal end-effector position (metres): a centred cross, 10 cm arms, 8 cm vertically
PROBES = [np.zeros(3), np.array([0.10, 0, 0]), np.array([-0.10, 0, 0]), np.array([0, 0.10, 0]), np.array([0, -0.10, 0]),
          np.array([0, 0, 0.08]), np.array([0, 0, -0.06])]
MIN_ACTION = 1e-3


def triangulate(P, A):
    """P (K,3) end-effector positions, A (K,3) commanded translations. Returns dict(goal, residual_deg, forward,
    min_eig, n). Probes with a vanishing command carry no direction and are dropped."""
    P, A = np.asarray(P, float), np.asarray(A, float)
    n = np.linalg.norm(A, axis=1)
    keep = n > MIN_ACTION
    P, U = P[keep], A[keep] / n[keep, None]
    K = len(P)
    if K < 3:
        return dict(goal=None, residual_deg=float("nan"), forward=float("nan"), min_eig=0.0, n=int(K))
    M = np.eye(3)[None] - U[:, :, None] * U[:, None, :]                  # projectors orthogonal to each ray
    N = M.sum(0)
    ev = np.linalg.eigvalsh(N)
    g = np.linalg.lstsq(N, (M @ P[:, :, None]).sum(0)[:, 0], rcond=None)[0]
    d = g[None] - P
    dist = np.linalg.norm(d, axis=1)
    cosang = np.clip((d * U).sum(1) / np.maximum(dist, 1e-9), -1, 1)
    ang = np.degrees(np.arccos(cosang))
    return dict(goal=g.tolist(), residual_deg=float(np.sqrt(np.mean(ang ** 2))), forward=float(np.mean(cosang > 0)),
                min_eig=float(ev[0] / K), n=int(K))


def compare(smc_raw, smc_cor, margin_deg=0.0):
    """Which frame does the policy agree with itself on? Returns 'raw', 'cor' or 'tie'. A variant whose rays are
    parallel (min_eig ~ 0) or point away from their own sub-goal is uninformative and never wins on its own."""
    def usable(s):
        return s["n"] >= 3 and np.isfinite(s["residual_deg"]) and s["min_eig"] > 1e-3 and s["forward"] >= 0.5
    ur, uc = usable(smc_raw), usable(smc_cor)
    if not ur and not uc:
        return "tie"
    if ur != uc:
        return "raw" if ur else "cor"
    diff = smc_raw["residual_deg"] - smc_cor["residual_deg"]
    return "tie" if abs(diff) <= margin_deg else ("cor" if diff > 0 else "raw")


# ---- beyond the blind spot: the measured nuisance PREDICTS how the policy's perceived sub-goal should move ------------
# Proposition 1: a policy that takes its own position from proprioception perceives a TRANSLATED goal under a moved
# camera, and its rays still meet. The wrapper, however, has measured the camera (policy-free). Geometry then predicts
# where such a policy must see a point x that really sits at height h: project x into the ACTUAL camera, back-project
# that pixel through the ASSUMED training camera onto the plane z = h. If the sub-goal triangulated on the raw frame
# sits where this map sends the sub-goal triangulated on the corrected frame, the policy was misperceiving exactly as
# geometry says and the correction is doing its job. If the sub-goal does not move at all, the policy is invariant to
# this nuisance and the correction is unnecessary: do nothing.
GOAL_MOVE_CM = 2.0            # below this the two sub-goals are the same point (triangulation noise on the organism: ~1 cm)


def predicted_raw_goal(g_cor, C_act, R_act, C_train, R_train):
    """Where a policy that assumes the training camera sees the world point g_cor when the actual camera is
    (C_act, R_act). MuJoCo camera convention (looks along -z); columns of R are camera axes in the world."""
    g = np.asarray(g_cor, float)
    pc = (g - C_act) @ R_act                              # point in the actual camera's frame
    if pc[2] > -1e-6:
        return None
    ray = np.array([pc[0] / -pc[2], pc[1] / -pc[2], -1.0])  # the pixel's ray, read as if it came from the training camera
    dw = R_train @ ray
    if abs(dw[2]) < 1e-9:
        return None
    t = (g[2] - C_train[2]) / dw[2]
    return None if t <= 0 else C_train + t * dw


def decide_v2(smc_raw, smc_cor, floor_deg, pred_raw_goal=None, margin_deg=1.0):
    """Minimal intervention. Returns (choice, reason), choice in {'raw', 'cor'}.
      A  raw residual above the policy's nominal floor and the correction lowers it      -> cor  (distortion removed)
      D  the correction raises the residual above the floor                               -> raw  (correction harmful)
      B  raw at the floor and the sub-goal does not move under the correction             -> raw  (policy invariant)
      C  raw at the floor, sub-goal moves, and it moves the way the MEASURED shift says    -> cor  (translated goal)
         raw at the floor, sub-goal moves some other way                                   -> raw  (unexplained: do nothing)
    floor_deg: the policy's residual in the training setup (the 'nominal' probe of the same task)."""
    rr, rc = smc_raw["residual_deg"], smc_cor["residual_deg"]
    if not (np.isfinite(rr) and np.isfinite(rc)) or smc_raw["goal"] is None or smc_cor["goal"] is None:
        return "cor", "uninformative: the wrapper's own decision stands"
    hi = floor_deg + margin_deg
    if rc > hi and rc > rr:
        return "raw", "D: the correction makes the policy less self-consistent"
    if rr > hi and rc < rr:
        return "cor", "A: the correction removes a distortion the raw frame shows"
    move = float(np.linalg.norm(np.array(smc_cor["goal"]) - np.array(smc_raw["goal"]))) * 100
    if move < GOAL_MOVE_CM:
        return "raw", f"B: the perceived sub-goal does not move ({move:.1f} cm): the policy is invariant here"
    if pred_raw_goal is not None:
        miss = float(np.linalg.norm(np.array(smc_raw["goal"]) - np.asarray(pred_raw_goal))) * 100
        if miss < 0.5 * move:
            return "cor", f"C: the sub-goal moves {move:.1f} cm, as the measured shift predicts (within {miss:.1f} cm)"
        return "raw", f"the sub-goal moves {move:.1f} cm but not as the measured shift predicts ({miss:.1f} cm off)"
    return "raw", f"the sub-goal moves {move:.1f} cm and no measured shift explains it"


# ---- action-field fingerprint (2026-09-19, after the OFT probe) ------------------------------------------------------
# The concurrency premise above is FALSE for OpenVLA-OFT: its opening action field DIVERGES (the commanded speed grows
# along the reach, as in the accelerating phase of its demonstrations), the triangulated point sits a few cm BEHIND the
# gripper in every task, and the residual is ~132 deg even at the training camera. What the probe data do carry is a
# FINGERPRINT: the actions of the policy over the probe cross, recorded once in the reference configuration of a task
# (when the system is commissioned and known to work), against the same actions under a candidate frame at deployment.
# No assumption about the shape of the field is needed. On OFT (development family, 72 units, stored rows): AUROC of a
# small corrected-frame distance for success of the wrapped policy 0.947, of a small raw-frame distance for raw success
# 0.786; with ONE probe 0.817 / 0.685, with three 0.874 / 0.739: sampling the FIELD through self-motion is what makes
# the fingerprint discriminative. Corrected distance <= 0.15: success 0.96 (n = 54); above 0.25: 0.00 (n = 6).
FP_UNAFFECTED = 0.10          # raw field within this of the fingerprint: the policy is unaffected, do nothing
FP_LOST = 0.25                # neither field within this: expect failure whatever is done


def field_distance(A, A_ref):
    """Relative distance between two sets of probe actions (K x 3, same probe order): ||A - A_ref|| / ||A_ref||."""
    A, A_ref = np.asarray(A, float), np.asarray(A_ref, float)
    return float(np.linalg.norm(A - A_ref) / max(np.linalg.norm(A_ref), 1e-9))


def decide_fingerprint(d_raw, d_cor):
    """(choice, reason) for the IMAGE channels of the wrapper; the state servo is never gated.
    1. raw field within FP_UNAFFECTED of the fingerprint  -> raw: the policy behaves as in training, leave it alone
    2. otherwise the frame whose field is closer          -> cor if the correction brings the policy back, else raw
    The reason flags d > FP_LOST on the chosen frame: the policy does not behave as in training under either frame."""
    if d_raw <= FP_UNAFFECTED:
        return "raw", f"unaffected: raw field {d_raw:.3f} from the fingerprint"
    choice, d = ("cor", d_cor) if d_cor < d_raw else ("raw", d_raw)
    lost = "; EXPECT FAILURE, neither frame looks like training" if d > FP_LOST else ""
    return choice, f"{choice} field closer ({d_raw:.3f} raw, {d_cor:.3f} corrected){lost}"

In [ ]:
%%writefile /content/scripts/smc_probe.py
"""Self-motion consistency probe: NO rollouts. For every (base task, condition) of the graded sweep, set the scene up
exactly as sweep_run does, let the wrapper prepare its corrections, move the end-effector through the probe cross
(smc.PROBES), and at each probe position ask the frozen policy for its action on the RAW frame and on the CORRECTED
frame. Rows store positions and actions, so every statistic can be recomputed offline.

    python smc_probe.py run    --family oft|openvla|pi05 --out_dir <sweep dir> --nominals <json> [--n_tasks 3] [--extended]
    python smc_probe.py report --out_dir <sweep dir>          # joins with episodes_<policy>.jsonl (raw / full outcomes)

About 1 to 1.5 minutes per unit: a few hundred simulator steps and 14 policy forward passes.
"""
import argparse
import glob
import json
import os
import sys
import time

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import smc  # noqa: E402


def probe_unit(adapter, env, suite, task, cond, nominal, R, S, W):
    seed = task["task_id"]
    obs, info = R.start(env, suite, task, cond, seed)
    rng = np.random.default_rng(seed)
    sigma = cond["params"].get("sigma", 0.0)
    noise = lambda o: S.add_noise(o, sigma, rng)
    adapter.new_episode(task["instruction"], seed)
    obs, transform, winfo = R.prepare_full(env, obs, nominal, noise)         # obs is already noised here
    can = W.Canonicalizer(env, nominal)
    p0, q0 = can.nominal_eef()
    copy = lambda o: {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in o.items()}

    def act(o):
        adapter.new_episode(task["instruction"], seed)                       # same seed: only the frame differs
        return np.mean([np.asarray(q, float)[:3] for q in adapter.chunk(dict(o))], 0)

    P, A_raw, A_cor = [], [], []
    for k, off in enumerate(smc.PROBES):
        if k:
            obs, _ = can.servo(obs, p0 + off, q0)
            obs = noise(obs)
        P.append(np.asarray(obs["robot0_eef_pos"], float).tolist())
        A_raw.append(act(copy(obs)).tolist())
        A_cor.append(act(transform(copy(obs))).tolist())
    s_raw, s_cor = smc.triangulate(P, A_raw), smc.triangulate(P, A_cor)
    keep = ("cam_applied", "photo_applied", "denoise_applied", "state_applied", "cam_est_err_cm_deg", "cam_true_shift_cm_deg",
            "cam_est_pose", "cam_nom_pose")
    return dict(P=P, A_raw=A_raw, A_cor=A_cor, smc_raw=s_raw, smc_cor=s_cor, pick=smc.compare(s_raw, s_cor),
                **{k: winfo[k] for k in keep if k in winfo})


def run(a):
    import sweep_conditions as S
    import sweep_run as R
    import wrapper_libero as W
    from calib_check import make_env
    from libero.libero import benchmark
    from plus_common import append_row, load_rows
    from policy_adapters import make_adapter
    suite = benchmark.get_benchmark_dict()[R.SUITE]()
    tasks = S.base_tasks(suite, a.n_tasks)
    nominals = W.read_nominals(suite, lambda i: make_env(suite, i), cache=a.nominals)
    adapter = make_adapter(a.family, a.octo_calib)
    path = os.path.join(a.out_dir, f"smc_{adapter.name}.jsonl")
    done = {(r["task_id"], r["cond"]) for r in load_rows(path)}
    conds = S.CONDITIONS + (S.EXTENDED if a.extended else [])
    todo = [(c, t) for c in conds for t in tasks if (t["task_id"], c["name"]) not in done]
    print(f"{adapter.name}: {len(todo)} units to probe", flush=True)
    t0 = time.time()
    if True:        # rows go through append_row (open, write, CLOSE): Drive uploads a file only when it is closed
        for n, (cond, t) in enumerate(todo):
            env = make_env(suite, t["task_id"])
            try:
                row = probe_unit(adapter, env, suite, t, cond, nominals[t["scene"]], R, S, W)
            finally:
                env.close()
            row = dict(policy=adapter.name, task_id=t["task_id"], cond=cond["name"], family=cond["family"], **row)
            append_row(path, row)
            el = (time.time() - t0) / 60
            print(f"{adapter.name} {t['task_id']:5d} {cond['name']:<14} residual raw {row['smc_raw']['residual_deg']:5.1f} "
                  f"cor {row['smc_cor']['residual_deg']:5.1f} deg -> pick {row['pick']:<3} | {n + 1}/{len(todo)} "
                  f"eta {el / (n + 1) * (len(todo) - n - 1):.0f} min", flush=True)


def auroc(score, label):
    score, label = np.asarray(score, float), np.asarray(label, bool)
    ok = np.isfinite(score)
    score, label = score[ok], label[ok]
    if label.all() or (~label).all() or len(label) < 4:
        return float("nan")
    order = np.argsort(score)
    ranks = np.empty(len(score))
    ranks[order] = np.arange(1, len(score) + 1)
    for v in np.unique(score):                                                # average ranks over ties
        m = score == v
        ranks[m] = ranks[m].mean()
    return float((ranks[label].sum() - label.sum() * (label.sum() + 1) / 2) / (label.sum() * (~label).sum()))


def report(a):
    for path in sorted(glob.glob(os.path.join(a.out_dir, "smc_*.jsonl"))):
        pol = os.path.basename(path)[4:-6]
        ep_path = os.path.join(a.out_dir, f"episodes_{pol}.jsonl")
        if not os.path.exists(ep_path):
            print(f"{pol}: no episodes file next to the probe rows, skipping")
            continue
        ep = {}
        for l in open(ep_path):
            if l.strip():
                r = json.loads(l)
                if r.get("rep", 0) == 0:
                    ep[(r["task_id"], r["cond"], r["arm"])] = bool(r["success"])
        rows = [json.loads(l) for l in open(path) if l.strip()]
        U = [r for r in rows if (r["task_id"], r["cond"], "raw") in ep and (r["task_id"], r["cond"], "full") in ep
             and ep.get((r["task_id"], "nominal", "raw"), False)]
        raw = np.array([ep[(r["task_id"], r["cond"], "raw")] for r in U])
        full = np.array([ep[(r["task_id"], r["cond"], "full")] for r in U])
        pick = np.array([r["pick"] for r in U])
        chosen = np.where(pick == "raw", raw, full)                           # 'cor' and 'tie' keep the wrapper's frame
        print(f"\n===== {pol} =====  {len(U)} units on base tasks the policy solves unperturbed")
        print(f"success: always raw {raw.mean():.3f} | always full {full.mean():.3f} | SMC-selected {chosen.mean():.3f} "
              f"| oracle {np.maximum(raw, full).mean():.3f}")
        print(f"broken by the wrapper (raw ok, full fails): {int((raw & ~full).sum())}, of which SMC picks raw: "
              f"{int((raw & ~full & (pick == 'raw')).sum())}")
        print(f"recovered by the wrapper (raw fails, full ok): {int((~raw & full).sum())}, of which SMC keeps the wrapper: "
              f"{int((~raw & full & (pick != 'raw')).sum())}")
        r_raw = np.array([r["smc_raw"]["residual_deg"] for r in U])
        r_cor = np.array([r["smc_cor"]["residual_deg"] for r in U])
        print(f"AUROC, low residual on the RAW frame predicts raw success: {auroc(-r_raw, raw):.3f}")
        print(f"AUROC, low residual on the CORRECTED frame predicts full success: {auroc(-r_cor, full):.3f}")
        print(f"median residual deg: raw frame, raw succeeds {np.nanmedian(r_raw[raw]):.1f} / fails "
              f"{np.nanmedian(r_raw[~raw]) if (~raw).any() else float('nan'):.1f}")
        print(f"picks: {dict(zip(*np.unique(pick, return_counts=True)))}")
        # action-field fingerprint: distance of the probe actions to the SAME task's actions in the training setup
        ref = {r["task_id"]: r["A_raw"] for r in rows if r["cond"] == "nominal"}
        F = [r for r in U if r["task_id"] in ref and r["cond"] != "nominal"]      # the reference units are not test units
        if F:
            fr = np.array([ep[(r["task_id"], r["cond"], "raw")] for r in F])
            ff = np.array([ep[(r["task_id"], r["cond"], "full")] for r in F])
            d_r = np.array([smc.field_distance(r["A_raw"], ref[r["task_id"]]) for r in F])
            d_c = np.array([smc.field_distance(r["A_cor"], ref[r["task_id"]]) for r in F])
            img = np.array([bool(r.get("cam_applied") or r.get("photo_applied") or r.get("denoise_applied")) for r in F])
            keep_raw = np.array([bool(i) and smc.decide_fingerprint(a_, b_)[0] == "raw" for i, a_, b_ in zip(img, d_r, d_c)])
            sel = np.where(keep_raw, fr, ff)
            mean = lambda v, m: float(v[m].mean()) if m.any() else float("nan")
            print(f"\nFINGERPRINT ({len(F)} units, image channel fired on {int(img.sum())})")
            print(f"  AUROC, small raw-frame distance predicts raw success: {auroc(-d_r, fr):.3f} | small corrected-frame "
                  f"distance predicts full success: {auroc(-d_c, ff):.3f}")
            for k, name in ((1, "centre probe only"), (3, "centre + x pair"), (7, "all 7 probes")):
                dk = np.array([smc.field_distance(np.array(r["A_cor"])[:k], np.array(ref[r["task_id"]])[:k]) for r in F])
                print(f"    {name:<18} corrected-frame AUROC {auroc(-dk, ff):.3f}")
            print(f"  success: always raw {fr.mean():.3f} | always full {ff.mean():.3f} | fingerprint-selected {sel.mean():.3f} "
                  f"| oracle {np.maximum(fr, ff).mean():.3f}")
            print(f"  where an image channel fired: raw {mean(fr, img):.3f} | full {mean(ff, img):.3f} | selected {mean(sel, img):.3f}")
            print(f"  broken by an image channel (raw ok, full fails): {int((fr & ~ff & img).sum())}, kept raw: "
                  f"{int((fr & ~ff & keep_raw).sum())} | recovered by an image channel: {int((~fr & ff & img).sum())}, "
                  f"kept corrected: {int((~fr & ff & img & ~keep_raw).sum())}")
            calm = img & (d_r <= smc.FP_UNAFFECTED)
            print(f"  left alone (raw field within {smc.FP_UNAFFECTED} of the fingerprint): {int(calm.sum())} units, raw success there {mean(fr, calm):.3f}")
            for lo, hi in ((0, 0.15), (0.15, smc.FP_LOST), (smc.FP_LOST, 9)):
                m = (d_c > lo) & (d_c <= hi)
                print(f"  corrected distance in ({lo}, {hi}]: full success {mean(ff, m):.2f} (n={int(m.sum())})")
        # minimal intervention (smc.decide_v2): floor = the same task's residual in the training setup; the MEASURED
        # camera predicts how a translated-goal policy's sub-goal must move (needs cam_est_pose in the rows)
        import wrapper_libero as W
        floor = {r["task_id"]: r["smc_raw"]["residual_deg"] for r in rows if r["cond"] == "nominal"}
        v2, why = [], {}
        for r in U:
            pred = None
            if r.get("cam_applied") and r.get("cam_est_pose") and r["smc_cor"]["goal"] is not None:
                e, n = np.array(r["cam_est_pose"]), np.array(r["cam_nom_pose"])
                pred = smc.predicted_raw_goal(r["smc_cor"]["goal"], e[:3], W.rotvec_to_mat(e[3:]), n[:3], W.rotvec_to_mat(n[3:]))
            c, reason = smc.decide_v2(r["smc_raw"], r["smc_cor"], floor.get(r["task_id"], float("nan")), pred)
            v2.append(c)
            why[reason[:1]] = why.get(reason[:1], 0) + 1
        v2 = np.array(v2)
        chosen2 = np.where(v2 == "raw", raw, full)
        print(f"minimal intervention (decide_v2): success {chosen2.mean():.3f} | broken units kept raw "
              f"{int((raw & ~full & (v2 == 'raw')).sum())}/{int((raw & ~full).sum())} | recovered units kept corrected "
              f"{int((~raw & full & (v2 == 'cor')).sum())}/{int((~raw & full).sum())} | cases {why}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("mode", choices=["run", "report"])
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--nominals", default="")
    ap.add_argument("--family", default="oft", choices=["oft", "openvla", "octo", "pi05", "pi0", "smolvla"])
    ap.add_argument("--octo_calib", default="")
    ap.add_argument("--n_tasks", type=int, default=3)
    ap.add_argument("--extended", action="store_true")
    a = ap.parse_args()
    (run if a.mode == "run" else report)(a)

In [ ]:
%%writefile /content/scripts/arms_report.py
"""Paired comparison of every arm in a wrap_run file, per category, on the tasks ALL the listed arms have run.

    python arms_report.py --rows <wrap_*.jsonl> [--arms raw,full,tta,generic,full_homog,full_nofill] [--ref raw]

For each arm: success, and against the reference arm the tasks recovered (ref fails, arm succeeds), the tasks broken
(ref succeeds, arm fails) and the exact two-sided McNemar p value.
"""
import argparse
import json
from collections import defaultdict
from math import comb

ap = argparse.ArgumentParser()
ap.add_argument("--rows", required=True)
ap.add_argument("--arms", default="")
ap.add_argument("--ref", default="raw")
a = ap.parse_args()

rows = [json.loads(l) for l in open(a.rows) if l.strip()]
by = defaultdict(dict)
for r in rows:
    by[(r["category"], r["arm"])][r["task_id"]] = bool(r["success"])          # the last row of a (task, arm) wins
cats = sorted({c for c, _ in by})
want = [x for x in a.arms.split(",") if x]


def mcnemar(b, c):
    n = b + c
    return 1.0 if n == 0 else min(1.0, 2 * sum(comb(n, k) for k in range(0, min(b, c) + 1)) / 2 ** n)


for cat in cats:
    arms = [arm for (c, arm) in by if c == cat and (not want or arm in want)]
    if a.ref not in arms or len(arms) < 2:
        continue
    arms = [a.ref] + sorted(x for x in arms if x != a.ref)
    common = set.intersection(*[set(by[(cat, arm)]) for arm in arms])
    if not common:
        print(f"\n{cat}: the arms {arms} share no task yet")
        continue
    print(f"\n{cat}: {len(common)} tasks run by every arm")
    print(f"  {'arm':<22}{'success':>9}{'recovered':>11}{'broken':>8}{'p vs ' + a.ref:>12}")
    ref = by[(cat, a.ref)]
    for arm in arms:
        v = by[(cat, arm)]
        s = sum(v[t] for t in common) / len(common)
        rec = sum((not ref[t]) and v[t] for t in common)
        brk = sum(ref[t] and (not v[t]) for t in common)
        tail = "" if arm == a.ref else f"{rec:>8}/{sum(not ref[t] for t in common):<3}{brk:>5}/{sum(ref[t] for t in common):<3}{mcnemar(rec, brk):>10.3f}"
        print(f"  {arm:<22}{s:>9.3f}{tail}")

In [ ]:
%%writefile /content/scripts/attribution_ladder.py
"""WHERE THE DEFICIT LIVES: a counterfactual ladder on camera tasks (the title claim as a table).

The same task and seed under nested arms, each removing one more cause:
  raw      the frozen policy on the shifted camera
  full     + the wrapper with the pose the robot MEASURED on its own body
  true     + the simulator's TRUE camera pose in the same reprojection      (cam_true_ref / full_truepose): removes measurement error
  oracle   + a perfect render from the training camera                        (oracle / full_oracle): removes the inversion loss
A raw failure is attributed to the FIRST rung that repairs it:
  recovered by measurement | not identified (only the true pose repairs it) | not invertible (only the perfect render does:
  the view is lost) | policy (fails even on a perfect training-view frame: not the camera at all).
The upper rungs are usually run only where `full` failed, so a missing rung above a success is not needed.
Single rollouts are noisy (about 15% flip with nothing applied): units that succeed on a lower rung and fail on a higher one
are COUNTED and printed as that noise rate, never hidden.

    python attribution_ladder.py --set libero_10 rows.jsonl rows_oft.jsonl --set object wraps_libero_object/wrap_openvla_oft.jsonl wraps_ladder_libero_object/wrap_openvla_oft.jsonl
"""
import argparse
import os
import sys
from collections import Counter, defaultdict
from math import comb

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from plus_common import load_rows  # noqa: E402
from sweep_analysis import auroc, boot_ci  # noqa: E402

RUNG = {"raw": "raw", "full": "full", "cam_true_ref": "true", "full_truepose": "true", "oracle": "oracle", "full_oracle": "oracle"}
ORDER = ["raw", "full", "true", "oracle"]


def mcnemar(b, c):
    n = b + c
    return 1.0 if n == 0 else min(1.0, 2 * sum(comb(n, k) for k in range(min(b, c) + 1)) / 2 ** n)


def units(files):
    u = defaultdict(dict)
    for f in files:
        for r in load_rows(f):
            if r.get("category") != "camera" or r.get("rep", 0) != 0:
                continue
            rung = RUNG.get(r.get("arm", "raw"))                 # baseline files carry no arm: they are the raw policy
            if rung and rung not in u[r["task_id"]]:
                u[r["task_id"]][rung] = r
    return u


def bucket(v):
    ok = lambda k: bool(v[k]["success"])
    if ok("full"):
        return "recovered by measurement"
    if "true" in v and ok("true"):
        return "not identified"
    if "oracle" not in v:
        return "upper rungs not run"
    return "not invertible" if ok("oracle") else "policy"


def report(name, u):
    base = {t: v for t, v in u.items() if "raw" in v and "full" in v}
    print(f"\n===== {name} =====  {len(base)} camera tasks with raw and full")
    if not base:
        return Counter(), []
    for k in ORDER:
        have = [v for v in base.values() if k in v]
        print(f"  {k:<7} success {sum(bool(v[k]['success']) for v in have):>3} / {len(have)}")
    fails = {t: v for t, v in base.items() if not v["raw"]["success"]}
    b = Counter(bucket(v) for v in fails.values())
    print(f"  raw failures {len(fails)}: " + ", ".join(f"{k} {b[k]}" for k in
          ("recovered by measurement", "not identified", "not invertible", "policy", "upper rungs not run") if b[k]))
    for lo, hi in (("full", "true"), ("true", "oracle"), ("full", "oracle")):
        both = [v for v in base.values() if lo in v and hi in v]
        up = sum(bool(v[hi]["success"]) and not v[lo]["success"] for v in both)
        down = sum(bool(v[lo]["success"]) and not v[hi]["success"] for v in both)
        if both:
            print(f"  {hi} vs {lo}: fixes {up}, breaks {down} on {len(both)} paired tasks, exact McNemar p = {mcnemar(up, down):.3f}")
    cov = lambda v: v["full"].get("reproj_coverage")
    for k in ("recovered by measurement", "not identified", "not invertible", "policy"):
        c = [cov(v) for v in fails.values() if bucket(v) == k and cov(v) is not None]
        e = [v["full"]["cam_est_err_cm_deg"][0] for v in fails.values() if bucket(v) == k and v["full"].get("cam_est_err_cm_deg")]
        if c or e:
            print(f"    {k:<26} coverage median {np.median(c):.2f} [{min(c):.2f}, {max(c):.2f}] (n {len(c)})" if c else f"    {k:<26}",
                  f"| calibration error median {np.median(e):.1f} cm (n {len(e)})" if e else "")
        ids = sorted(t for t, v in fails.items() if bucket(v) == k)
        if ids and k != "recovered by measurement":
            print(f"      tasks: {ids}")
    return b, list(fails.values())


def baseline_row(fails):
    """Which single number, known BEFORE the rollout, says whether the wrapper repairs a raw camera failure?"""
    y = [bool(v["full"]["success"]) for v in fails]
    feats = {
        "shift distance (cm), negated": lambda v: -v["full"]["cam_true_shift_cm_deg"][0] if v["full"].get("cam_true_shift_cm_deg") else None,
        "shift rotation (deg), negated": lambda v: -v["full"]["cam_true_shift_cm_deg"][1] if v["full"].get("cam_true_shift_cm_deg") else None,
        "calibration error (cm), negated": lambda v: -v["full"]["cam_est_err_cm_deg"][0] if v["full"].get("cam_est_err_cm_deg") else None,
        "self-measured coverage": lambda v: v["full"].get("reproj_coverage"),
    }
    print(f"\n===== pooled: what predicts repair of a raw camera failure ({len(y)} units, {sum(y)} repaired) =====")
    for k, fn in feats.items():
        x = [(fn(v), yy) for v, yy in zip(fails, y) if fn(v) is not None]
        if len(x) >= 10 and 0 < sum(b for _, b in x) < len(x):
            s, l = [a for a, _ in x], [b for _, b in x]
            lo, hi = boot_ci(s, l)
            print(f"  {k:<34} AUROC {auroc(s, l):.3f} [{lo:.2f}, {hi:.2f}]  (n {len(x)})")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--set", nargs="+", action="append", required=True, metavar=("NAME", "FILE"))
    a = ap.parse_args()
    total, every = Counter(), []
    for s in a.set:
        b, f = report(s[0], units(s[1:]))
        total += b
        every += f
    print(f"\n===== pooled attribution of {sum(total.values())} raw camera failures =====")
    for k, n in total.most_common():
        print(f"  {k:<26} {n:>3}  ({100 * n / max(sum(total.values()), 1):.0f}%)")
    baseline_row(every)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/scripts/tolerance_probe.py
"""THE POLICY'S OWN CALIBRATION TOLERANCE, stage 1: a rollout-free dose-response (POST HOC, exploratory; touches no method code).

How wrong may the camera pose be before a frozen policy's OPENING ACTION leaves what it does under a perfect calibration?
For a base task under a camera shift the correction can undo (orbit 10 and 20 deg, coverage above 0.6), at the rest pose:
  reference  the first action chunk on the frame corrected with the TRUE camera pose
  dose       the same frame corrected with the true pose PLUS a known residual: 0.5 to 16 cm along each camera axis, 0.5 to
             8 deg about each camera axis; same instruction, same sampler seed, so only the residual differs
  distance   smc.field_distance(chunk, reference), the quantity whose frozen threshold FP_UNAFFECTED = 0.10 means
             "the policy behaves as in training"
  floor      the reference queried again (same seed): what the distance reads when nothing changed
The knee (smallest dose whose median distance exceeds 0.10) is the policy's tolerance in centimetres and degrees. The
robot's own calibration error (0.07 to 0.54 cm from 7 to 1 poses, 2.8 cm under learned masks, 19.6 cm on DROID) is then read
off the same axis. No episode is rolled out. Single-pose distance is a weaker outcome proxy than the 7-probe fingerprint.

    python tolerance_probe.py run --family oft --n_tasks 10 --out_dir .../tolerance --nominals ...
    python tolerance_probe.py report --out_dir .../tolerance
"""
import argparse
import glob
import json
import os
import socket
import sys
import time

import numpy as np

sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

CONDS = ("orbit_az10", "orbit_az20")
CM = (0.5, 1, 2, 4, 8, 16)
DEG = (0.5, 1, 2, 4, 8)
MARKS_CM = (("7-pose self-calibration", 0.07), ("3-pose (the method)", 0.28), ("1 pose", 0.54), ("learned sensing, 30 tasks", 12.6), ("DROID real images", 19.6))


def probe_unit(adapter, env, suite, task, cond, nominal, R, W, smc):
    seed = task["task_id"]
    obs, _ = R.start(env, suite, task, cond, seed)
    can = W.Canonicalizer(env, nominal)
    Ct, Rt = can.true_camera()
    copy = lambda o: {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in o.items()}

    def act(C, Rm):
        o = copy(obs)
        o["agentview_image"], cov = can.canonical_image(o, C=C, R=Rm)
        adapter.new_episode(task["instruction"], seed)                       # same seed: only the frame differs
        return np.mean([np.asarray(q, float)[:3] for q in adapter.chunk(o)], 0), float(cov)

    ref, cov = act(Ct, Rt)
    again, _ = act(Ct, Rt)
    adapter.new_episode(task["instruction"], seed)
    raw = np.mean([np.asarray(q, float)[:3] for q in adapter.chunk(copy(obs))], 0)
    doses = []
    for axis in range(3):
        for d in CM:
            a, _ = act(Ct + Rt[:, axis] * d / 100.0, Rt)
            doses.append(dict(kind="cm", axis=axis, dose=d, distance=smc.field_distance([a], [ref])))
        for d in DEG:
            r = np.zeros(3)
            r[axis] = np.deg2rad(d)
            a, _ = act(Ct, Rt @ W.rotvec_to_mat(r))
            doses.append(dict(kind="deg", axis=axis, dose=d, distance=smc.field_distance([a], [ref])))
    return dict(coverage=cov, floor=smc.field_distance([again], [ref]), uncorrected=smc.field_distance([raw], [ref]),
                shift_cm_deg=list(W.pose_error(Ct, Rt, can.Cn, can.Rn)), doses=doses)


def run(a):
    import smc
    import sweep_conditions as S
    import sweep_run as R
    import wrapper_libero as W
    from calib_check import make_env
    from libero.libero import benchmark
    from plus_common import append_row, load_rows
    from policy_adapters import make_adapter
    suite = benchmark.get_benchmark_dict()[R.SUITE]()
    tasks = S.base_tasks(suite, a.n_tasks)
    nominals = W.read_nominals(suite, lambda i: make_env(suite, i), cache=a.nominals)
    adapter = make_adapter(a.family, a.octo_calib)
    os.makedirs(a.out_dir, exist_ok=True)
    done = {(r["task_id"], r["cond"]) for p in glob.glob(os.path.join(a.out_dir, f"tolerance_{adapter.name}*.jsonl")) for r in load_rows(p)}
    path = os.path.join(a.out_dir, f"tolerance_{adapter.name}.{socket.gethostname()[-6:]}.jsonl")     # one file per runtime
    todo = [(S.BY_NAME[c], t) for c in CONDS for t in tasks if (t["task_id"], c) not in done]
    print(f"{adapter.name}: {len(todo)} units to probe, {3 * (len(CM) + len(DEG)) + 3} policy queries each", flush=True)
    t0 = time.time()
    for n, (cond, t) in enumerate(todo):
        env = make_env(suite, t["task_id"])
        try:
            row = probe_unit(adapter, env, suite, t, cond, nominals[t["scene"]], R, W, smc)
        finally:
            env.close()
        append_row(path, dict(policy=adapter.name, task_id=t["task_id"], cond=cond["name"], **row))
        d = {(x["kind"], x["dose"]): [] for x in row["doses"]}
        for x in row["doses"]:
            d[(x["kind"], x["dose"])].append(x["distance"])
        el = (time.time() - t0) / 60
        print(f"{adapter.name} {t['task_id']:5d} {cond['name']:<11} floor {row['floor']:.3f} uncorrected {row['uncorrected']:.2f} | "
              f"1 cm {np.median(d[('cm', 1)]):.3f}  4 cm {np.median(d[('cm', 4)]):.3f}  16 cm {np.median(d[('cm', 16)]):.3f} | "
              f"1 deg {np.median(d[('deg', 1)]):.3f}  8 deg {np.median(d[('deg', 8)]):.3f} | {n + 1}/{len(todo)} "
              f"eta {el / (n + 1) * (len(todo) - n - 1):.0f} min", flush=True)


def report(a):
    import smc
    from plus_common import load_rows
    by = {}
    for p in sorted(glob.glob(os.path.join(a.out_dir, "tolerance_*.jsonl"))):
        for r in load_rows(p):
            by.setdefault(r["policy"], {})[(r["task_id"], r["cond"])] = r
    for pol, rows in by.items():
        rows = list(rows.values())
        print(f"\n===== {pol} =====  {len(rows)} units | noise floor (same frame, same seed) median {np.median([r['floor'] for r in rows]):.3f} | "
              f"uncorrected frame median {np.median([r['uncorrected'] for r in rows]):.2f} | coverage median {np.median([r['coverage'] for r in rows]):.2f}")
        for kind, grid, unit in (("cm", CM, "cm"), ("deg", DEG, "deg")):
            knee = None
            print(f"  injected residual ({unit})   median distance   [25%, 75%]   share of units over {smc.FP_UNAFFECTED}")
            for d in grid:
                per_unit = [np.median([x["distance"] for x in r["doses"] if x["kind"] == kind and x["dose"] == d]) for r in rows]
                q = np.quantile(per_unit, [0.25, 0.5, 0.75])
                over = float(np.mean(np.asarray(per_unit) > smc.FP_UNAFFECTED))
                print(f"    {d:>5}                      {q[1]:.3f}          [{q[0]:.3f}, {q[2]:.3f}]        {over:.2f}")
                if knee is None and q[1] > smc.FP_UNAFFECTED:
                    knee = d
            print(f"  TOLERANCE ({unit}): " + (f"the median distance first exceeds {smc.FP_UNAFFECTED} at {knee} {unit}" if knee
                  else f"never exceeds {smc.FP_UNAFFECTED} up to {grid[-1]} {unit}"))
            if kind == "cm":
                for name, e in MARKS_CM:
                    side = "no knee in range" if knee is None else ("INSIDE the tolerance" if e < knee else "OUTSIDE the tolerance")
                    print(f"      {name:<26} {e:>5} cm  -> {side}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("mode", choices=["run", "report"])
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--nominals", default="")
    ap.add_argument("--family", default="oft", choices=["oft", "openvla", "octo", "pi05", "pi0", "smolvla"])
    ap.add_argument("--octo_calib", default="")
    ap.add_argument("--n_tasks", type=int, default=10)
    a = ap.parse_args()
    (run if a.mode == "run" else report)(a)

In [ ]:
%%writefile /content/scripts/depth_server.py
"""Monocular METRIC depth served from a python whose transformers can read the model (Colab's system python), for a
venv that cannot (OFT pins transformers 4.40). Started by wrapper_libero.MetricDepth; not run by hand.

Protocol on stdin / stdout, one line each: prints `ready <model name>` once, then for every line `<in.npy> <out.npy>`
reads an HxWx3 uint8 image, writes HxW float32 depth in metres at the same size, and prints `ok` (or `error ...`).
"""
import sys

import numpy as np

MODELS = ("depth-anything/Depth-Anything-V2-Metric-Indoor-Small-hf", "Intel/zoedepth-nyu-kitti")


def main():
    import torch
    import torch.nn.functional as F
    from transformers import AutoImageProcessor, AutoModelForDepthEstimation
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    last = None
    for name in MODELS:
        try:
            proc = AutoImageProcessor.from_pretrained(name)
            model = AutoModelForDepthEstimation.from_pretrained(name).to(dev).eval()
            break
        except Exception as e:
            last = e
    else:
        print(f"error no metric depth model loads: {last!r}".replace("\n", " "), flush=True)
        return
    print(f"ready {name}", flush=True)
    for line in sys.stdin:
        try:
            src, dst = line.split()
            img = np.load(src)
            with torch.no_grad():
                pred = model(**proc(images=img, return_tensors="pt").to(dev)).predicted_depth
                pred = F.interpolate(pred[:, None].float(), size=img.shape[:2], mode="bicubic", align_corners=False)[0, 0]
            np.save(dst, pred.cpu().numpy().astype(np.float32))
            print("ok", flush=True)
        except Exception as e:
            print(f"error {e!r}".replace("\n", " "), flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/scripts/render_paper_views.py
"""POLICY-FREE renders for the paper's new 3D figures (Colab, MuJoCo offscreen under EGL, 512 x 512, nothing drawn on
any frame). No policy is loaded and nothing is scored; every number stored is read from the simulator.

    python render_paper_views.py --nominals <nominals_v3.json> --out_dir <dir> [--jobs scene_frusta,joint_offset_ghost,probe_ghosts]
    python render_paper_views.py --smoke --nominals <json> --out_dir <dir>      # one job at 256 px, prints the anchor check

Jobs (one .npz each in --out_dir; copy them to wrapper/paper/iclr/figs/renders/, make_image_figures.py reads them):
  scene_frusta   god's-eye view of one LIBERO-10 scene (task 1167, walls hidden), the TRAINING camera and the four Fig. 1
                 deployed cameras (orbit 20, pulled back 30%, orbit 45, turned away 35) as frusta: 3D centres, look-at
                 points and frustum corners in world metres AND projected into the overview image; plus each camera's own
                 clean 512 px view of the unperturbed scene.
  joint_offset_ghost  one joint-offset condition (sweep condition offset0.20, task 1167): the arm at the OFFSET start and at the
                 TRAINING start (q_nom), from the training camera and from the overview, with robot masks for ghosting.
  probe_ghosts   the 7 fingerprint probe poses (smc.PROBES, the cross the probe servo drives through) from the training
                 camera and from the overview, with robot masks and the end-effector position of each probe (world and
                 pixels). The per-probe ACTION vectors are never drawn on a frame: make_image_figures.py joins them from
                 the logged fingerprint rows (smc_<policy>.jsonl) and writes them as a SEPARATE transparent arrow layer
                 (one PNG per logged unit and frame, anchored at eef_px_training), so the tiles stay clean.

Frames are stored in DISPLAY orientation (row 0 = top; mujoco.Renderer output, NOT the upside-down robosuite buffer), key
`orientation` = "display", so the figure script must not turn them. Every anchor is a display pixel (u right, v down).
`anchor_check` = fraction of robot-geom centres whose projection lands on the rendered robot mask (dilated 3 px): below
0.9 means the projection convention is wrong and the anchors must not be used.
"""
import argparse
import json
import os
import sys

import numpy as np

sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

TASK = 1167                                                     # the Fig. 1 orbit task; a scene every figure already uses
FIG1_CAMERAS = ["orbit_az20", "pullback_1.3", "orbit_az45", "look_away_35"]
JOINT_COND = "offset0.20"
OVERVIEW = dict(az=55.0, el=28.0, scale=2.5)                    # same god's-eye pose as the design bundle's overviews
FRUSTUM_DEPTH = 0.30                                            # metres along the optical axis for the drawn frustum
HIDE = ("wall", "ceiling")                                      # geoms hidden in the god's-eye view (name contains)


# ---------------------------------------------------------------- pure geometry (tested locally, no simulator)
def project(X, C, R, fovy_deg, H, W=None):
    """World points X (N,3) -> display pixels (N,2) for a MuJoCo camera at C with axes R (columns: x right, y up, z back).
    Returns (uv, in_front). Square pixels; f = 0.5 H / tan(fovy / 2), fovy is the vertical field of view."""
    W = H if W is None else W
    X = np.atleast_2d(np.asarray(X, float))
    pc = (X - np.asarray(C, float)) @ np.asarray(R, float)      # = R^T (X - C) for each row
    z = -pc[:, 2]
    f = 0.5 * H / np.tan(np.deg2rad(fovy_deg) / 2)
    zs = np.where(z > 1e-9, z, np.nan)
    uv = np.stack([W / 2 + f * pc[:, 0] / zs, H / 2 - f * pc[:, 1] / zs], 1)
    return uv, z > 1e-9


def frustum(C, R, fovy_deg, depth=FRUSTUM_DEPTH):
    """World corners (4,3) of the image rectangle at `depth` in front of the camera: top-left, top-right, bottom-right,
    bottom-left as seen in the camera's own display image."""
    t = np.tan(np.deg2rad(fovy_deg) / 2) * depth
    pc = np.array([[-t, t, -depth], [t, t, -depth], [t, -t, -depth], [-t, -t, -depth]])
    return np.asarray(C, float) + pc @ np.asarray(R, float).T


# ---------------------------------------------------------------- simulator side (Colab only)
class Shot:
    """A 512 px renderer of its own (robosuite's offscreen buffer is too small for 512), sharing the env's model/data."""

    def __init__(self, env, res):
        import mujoco
        self.mj, self.res = mujoco, res
        self.sim = env.env.sim
        self.m, self.d = self.sim.model._model, self.sim.data._data
        self.m.vis.global_.offwidth = max(int(self.m.vis.global_.offwidth), res)
        self.m.vis.global_.offheight = max(int(self.m.vis.global_.offheight), res)
        self.r = mujoco.Renderer(self.m, res, res)
        self.opt = mujoco.MjvOption()
        self.hidden = []

    def hide_walls(self):
        mj = self.mj
        for g in range(self.m.ngeom):
            name = mj.mj_id2name(self.m, mj.mjtObj.mjOBJ_GEOM, g) or ""
            if any(h in name.lower() for h in HIDE):
                self.hidden.append((g, int(self.m.geom_group[g])))
                self.m.geom_group[g] = 5
        self.opt.geomgroup[5] = 0
        return [self.mj.mj_id2name(self.m, self.mj.mjtObj.mjOBJ_GEOM, g) for g, _ in self.hidden]

    def show_walls(self):
        for g, grp in self.hidden:
            self.m.geom_group[g] = grp
        self.hidden = []
        self.opt.geomgroup[5] = 1

    def _scene(self, cam):
        try:
            self.r.update_scene(self.d, camera=cam, scene_option=self.opt)
        except TypeError:                   # an older mujoco without scene_option: walls stay visible, said once
            if not getattr(self, "_warned", False):
                print("note: this mujoco's Renderer has no scene_option; walls are NOT hidden in the overview", flush=True)
                self._warned = True
            self.r.update_scene(self.d, camera=cam)

    def rgb(self, cam="agentview"):
        self.mj.mj_forward(self.m, self.d)
        self._scene(cam)
        return self.r.render().copy()

    def robot_mask(self, geoms, cam="agentview"):
        self.mj.mj_forward(self.m, self.d)
        self.r.enable_segmentation_rendering()
        try:
            self._scene(cam)
            seg = self.r.render().copy()
        finally:
            self.r.disable_segmentation_rendering()
        return (seg[..., 1] == int(self.mj.mjtObj.mjOBJ_GEOM)) & np.isin(seg[..., 0], geoms)   # Renderer: (id, type)


def anchor_check(shot, can, C, R, mask):
    import cv2
    pts = shot.d.geom_xpos[can.robot_geoms]
    uv, ok = project(pts, C, R, can.fovy, shot.res)
    m = cv2.dilate(mask.astype(np.uint8), np.ones((7, 7), np.uint8)) > 0
    hit = [m[int(v), int(u)] for (u, v), k in zip(uv, ok) if k and 0 <= u < shot.res and 0 <= v < shot.res]
    return float(np.mean(hit)) if hit else 0.0


def overview_pose(can):
    import wrapper_libero as W
    pivot = W.optical_pivot(can.Cn, can.Rn, can.table_z if can.table_z is not None else W.TABLE_Z)
    C, R = W.orbit(can.Cn, can.Rn, pivot, OVERVIEW["az"], OVERVIEW["el"], OVERVIEW["scale"])
    return C, R, pivot


def at_camera(shot, can, C, R, fn):
    s = can._save()
    try:
        can._set_camera(C, R)
        return fn()
    finally:
        can._restore(s)


def eef(env):
    return np.asarray(env.env._get_observations(force_update=True)["robot0_eef_pos"], float)


def job_scene_frusta(ctx):
    import wrapper_libero as W
    env, t = ctx["env"](), ctx["task"]
    out = {}
    try:
        ctx["R"].start(env, ctx["suite"], t, ctx["S"].BY_NAME["nominal"], seed=t["task_id"])
        can = W.Canonicalizer(env, ctx["nominal"])
        shot = Shot(env, ctx["res"])
        Cov, Rov, pivot = overview_pose(can)
        out["hidden_geoms"] = np.array(shot.hide_walls(), dtype=object)
        out["overview"] = at_camera(shot, can, Cov, Rov, shot.rgb)
        out["overview_robot"] = at_camera(shot, can, Cov, Rov, lambda: shot.robot_mask(can.robot_geoms))
        out["anchor_check"] = at_camera(shot, can, Cov, Rov, lambda: anchor_check(shot, can, Cov, Rov, out["overview_robot"]))
        shot.show_walls()
        out["training_view"] = at_camera(shot, can, can.Cn, can.Rn, shot.rgb)
        cams = {"training": (can.Cn, can.Rn)}
        env.close()
        for c in FIG1_CAMERAS:                      # each deployed camera from a fresh env (apply_scene edits the model)
            env = ctx["env"]()
            ctx["R"].start(env, ctx["suite"], t, ctx["S"].BY_NAME[c], seed=t["task_id"])
            c2 = W.Canonicalizer(env, ctx["nominal"])
            cams[c] = c2.true_camera()
            out[f"view_{c}"] = Shot(env, ctx["res"]).rgb()
            out[f"shift_{c}"] = np.array(W.pose_error(*cams[c], can.Cn, can.Rn), float)       # (cm, deg) vs training
            env.close()
        env = None
        for k, (C, R) in cams.items():
            corners = frustum(C, R, can.fovy)
            look = W.optical_pivot(C, R, can.table_z if can.table_z is not None else W.TABLE_Z)
            uv, _ = project(np.vstack([C[None], look[None], corners]), Cov, Rov, can.fovy, ctx["res"])
            out[f"cam_{k}_world"] = np.vstack([C[None], look[None], corners])                 # centre, look-at, 4 corners
            out[f"cam_{k}_px"] = uv
            out[f"cam_{k}_R"] = np.asarray(R, float)
        out.update(overview_C=Cov, overview_R=Rov, overview_pivot=pivot, fovy=can.fovy, cameras=np.array(list(cams), dtype=object))
    finally:
        if env is not None:
            env.close()
    return out


def job_joint_offset(ctx):
    import wrapper_libero as W
    env, t = ctx["env"](), ctx["task"]
    out = {}
    try:
        cond = ctx["S"].BY_NAME[JOINT_COND]
        ctx["R"].start(env, ctx["suite"], t, cond, seed=t["task_id"])
        can = W.Canonicalizer(env, ctx["nominal"])
        shot = Shot(env, ctx["res"])
        Cov, Rov, _ = overview_pose(can)

        def grab(tag):
            shot.show_walls()                               # training-camera tiles keep the walls, as the policy sees them
            out[f"{tag}_training_view"] = shot.rgb()
            out[f"{tag}_training_robot"] = shot.robot_mask(can.robot_geoms)
            shot.hide_walls()
            out[f"{tag}_overview"] = at_camera(shot, can, Cov, Rov, shot.rgb)
            out[f"{tag}_overview_robot"] = at_camera(shot, can, Cov, Rov, lambda: shot.robot_mask(can.robot_geoms))
            p = eef(env)
            out[f"{tag}_eef_world"] = p
            out[f"{tag}_eef_px_training"] = project(p, can.Cn, can.Rn, can.fovy, ctx["res"])[0][0]
            out[f"{tag}_eef_px_overview"] = project(p, Cov, Rov, can.fovy, ctx["res"])[0][0]
            out[f"{tag}_q"] = shot.d.qpos[can.arm_idx].copy()

        grab("offset")
        out["anchor_check"] = anchor_check(shot, can, can.Cn, can.Rn, out["offset_training_robot"])
        shot.d.qpos[can.arm_idx] = can.q_nom                # what nominal_start does: the arm PLACED at the training start
        shot.d.qvel[list(can.robot._ref_joint_vel_indexes)] = 0.0
        grab("nominal")
        out["joint_dev_rad"] = np.abs(out["offset_q"] - out["nominal_q"])
        out.update(condition=JOINT_COND, overview_C=Cov, overview_R=Rov, fovy=can.fovy)
    finally:
        env.close()
    return out


def job_probe_ghosts(ctx):
    import smc
    import wrapper_libero as W
    env, t = ctx["env"](), ctx["task"]
    out = {}
    try:
        obs, _ = ctx["R"].start(env, ctx["suite"], t, ctx["S"].BY_NAME["nominal"], seed=t["task_id"])
        can = W.Canonicalizer(env, ctx["nominal"])
        shot = Shot(env, ctx["res"])
        Cov, Rov, _ = overview_pose(can)
        p0, q0 = can.nominal_eef()
        imgs, masks, oimgs, omasks, P, uv_t, uv_o = [], [], [], [], [], [], []
        for k, off in enumerate(smc.PROBES):
            obs, _ = can.servo(obs, p0 + off, q0)           # the probe servo, exactly as smc_probe.probe_unit drives it
            shot.show_walls()
            imgs.append(shot.rgb()); masks.append(shot.robot_mask(can.robot_geoms))
            shot.hide_walls()
            oimgs.append(at_camera(shot, can, Cov, Rov, shot.rgb))
            omasks.append(at_camera(shot, can, Cov, Rov, lambda: shot.robot_mask(can.robot_geoms)))
            p = np.asarray(obs["robot0_eef_pos"], float)
            P.append(p)
            uv_t.append(project(p, can.Cn, can.Rn, can.fovy, ctx["res"])[0][0])
            uv_o.append(project(p, Cov, Rov, can.fovy, ctx["res"])[0][0])
        out["anchor_check"] = anchor_check(shot, can, can.Cn, can.Rn, masks[-1])
        out.update(training_views=np.stack(imgs), training_robot=np.stack(masks), overviews=np.stack(oimgs),
                   overview_robot=np.stack(omasks), P=np.stack(P), probe_offsets=np.stack(smc.PROBES), p0=p0,
                   eef_px_training=np.stack(uv_t), eef_px_overview=np.stack(uv_o), overview_C=Cov, overview_R=Rov,
                   camera_C=can.Cn, camera_R=can.Rn, fovy=can.fovy)
    finally:
        env.close()
    return out


JOBS = {"scene_frusta": job_scene_frusta, "joint_offset_ghost": job_joint_offset, "probe_ghosts": job_probe_ghosts}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--nominals", required=True)
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--jobs", default=",".join(JOBS))
    ap.add_argument("--res", type=int, default=512)
    ap.add_argument("--smoke", action="store_true", help="probe_ghosts only, 256 px, printed check, nothing kept")
    ap.add_argument("--redo", action="store_true")
    a = ap.parse_args()
    os.environ.setdefault("MUJOCO_GL", "egl")
    import sweep_conditions as S
    import sweep_run as R
    import wrapper_libero as W
    from calib_check import make_env
    from libero.libero import benchmark
    suite = benchmark.get_benchmark_dict()["libero_10"]()
    nominals = W.read_nominals(suite, lambda i: make_env(suite, i), cache=a.nominals)
    task = {x["task_id"]: x for x in S.base_tasks(suite, 10)}[TASK]
    ctx = dict(suite=suite, S=S, R=R, task=task, nominal=nominals[task["scene"]], res=256 if a.smoke else a.res,
               env=lambda: make_env(suite, TASK))
    os.makedirs(a.out_dir, exist_ok=True)
    for name in (["probe_ghosts"] if a.smoke else a.jobs.split(",")):
        path = os.path.join(a.out_dir, name + ("_smoke" if a.smoke else "") + ".npz")
        if os.path.exists(path) and not a.redo and not a.smoke:
            print(f"{name}: exists, skipped ({path})", flush=True)
            continue
        d = JOBS[name](ctx)
        d.update(task_id=TASK, orientation="display", res=ctx["res"])
        chk = float(d["anchor_check"])
        np.savez_compressed(path, **{k: (v if isinstance(v, np.ndarray) else np.asarray(v)) for k, v in d.items()})
        print(f"{name}: wrote {path} | anchor check {chk:.2f} (need >= 0.90) | "
              + json.dumps({k: np.round(np.asarray(v, float), 2).tolist() for k, v in d.items() if k.startswith("shift_")}), flush=True)
        if chk < 0.9:
            print(f"WARNING {name}: anchors do not land on the robot; do not use them, send this line to the agent", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/scripts/pose_sources.py
"""POST HOC (TASKS.md P47, added 2026-09-24; not a registered test, reported with a dagger): the POSE-SOURCE ABLATION.

The same reprojection (sweep_run.prepare_full, `full`) driven by three camera-pose sources on the same sweep, the same
state channel, the same hand-over and the same seed, so the ONLY difference between the arms is where the pose comes from:
  full           (i)   the paper's silhouette search (wrapper_libero.Canonicalizer.calibrate_v2), unchanged
  full_refine    (ii)  EasyHeC-style refinement (Chen et al., "EasyHeC", RA-L 2023): gradient descent on a DENSE mask loss
                       from the SAME initial pose (the silhouette estimate (i)), on the SAME sweep masks. EasyHeC
                       back-propagates through a differentiable rasteriser; MuJoCo's renderer is not differentiable, so the
                       gradient here is a central finite difference of a soft-silhouette loss (both masks blurred, the
                       role a soft rasteriser's blur plays), with Adam, coarse to fine in the blur. The loss, the
                       initialisation and the optimiser follow EasyHeC; the gradient estimator does not.
  full_fiducial  (iii) a fiducial pose, the AnyCamVLA pose source: known 3D corners of four square markers lying on the
                       training table, projected into the deployed camera with the simulator's TRUE pose, perturbed by
                       Gaussian corner-localisation noise (sigma = FID_SIGMA_PX px at the 256 px observation), then
                       Perspective-n-Point (OpenCV) with the known intrinsics. Corners are never occluded ("oracle
                       corners"), which favours this source; a marker counts only when all four corners are in the image.
Nothing here changes a registered rule or an existing arm: `full` rows are produced exactly as before, and the two new
arms only replace (C_est, R_est) after calibrate_v2 has run. Every row logs `cam_est_err_cm_deg` of the pose it USED and,
for (ii) and (iii), `cam_silhouette_err_cm_deg` of the silhouette pose it started from or replaced.

Pure numpy / OpenCV; `refine` needs only the Canonicalizer methods _save/_restore/_set_camera/_robot_mask and its
arm_idx/grip_idx/sim/mres, so it is tested locally against a mock renderer (_wrapper_tests.py).
"""
import cv2
import numpy as np

FID_SIGMA_PX = 0.5                     # corner localisation noise of a square fiducial detector at 256 px (typical 0.1-0.5)
FID_SIDE = 0.06                        # marker side, metres
FID_OFFSETS = [(0.15, 0.15), (0.15, -0.15), (-0.15, 0.15), (-0.15, -0.15)]   # marker centres around the table pivot, m
REFINE_SIGMAS = (4.0, 2.0, 1.0)        # blur (px at the mask resolution) per stage, coarse to fine
REFINE_ITERS = 30                      # Adam steps per stage
REFINE_LR = (0.01, 0.01)               # Adam step: metres, radians (first stage; halved each stage)
REFINE_H = (0.004, 0.004)              # finite-difference step: metres, radians
ACCEPT = 0.8                           # refined pose used iff its loss < ACCEPT x the loss at the training pose


def _rotvec_to_mat(r):
    return cv2.Rodrigues(np.asarray(r, np.float64).reshape(3, 1))[0]


def _pose_error(C1, R1, C2, R2):
    rv = cv2.Rodrigues(np.asarray(R1 @ R2.T, np.float64))[0].ravel()
    return float(np.linalg.norm(np.asarray(C1) - np.asarray(C2)) * 100), float(np.rad2deg(np.linalg.norm(rv)))


# ---------------------------------------------------------------- (ii) EasyHeC-style dense refinement
def _soft(mask, sigma):
    m = (np.asarray(mask) > 0).astype(np.float32)
    k = int(2 * round(3 * sigma) + 1)
    return cv2.GaussianBlur(m, (k, k), sigma)


def dense_loss(can, recs, C, R, sigma, soft_obs=None):
    """L(C, R) = (1 / V) sum_v mean_pixels (G_sigma * M_render_v(C, R) - G_sigma * M_obs_v)^2 over the V sweep poses,
    both masks at the sweep's mask resolution, G_sigma a Gaussian blur of sigma pixels. The caller restores the state."""
    soft_obs = soft_obs or [_soft(rec[2], sigma) for rec in recs]
    can._set_camera(C, R)
    total = 0.0
    for rec, so in zip(recs, soft_obs):
        can.sim.data.qpos[can.arm_idx] = rec[0]
        if rec[1] is not None:
            can.sim.data.qpos[can.grip_idx] = rec[1]
        can.sim.forward()
        ren = cv2.resize(can._robot_mask(can.mres).astype(np.uint8), (so.shape[1], so.shape[0]), interpolation=cv2.INTER_NEAREST)
        total += float(np.mean((_soft(ren, sigma) - so) ** 2))
    return total / len(recs)


def refine(can, recs, C0, R0, sigmas=REFINE_SIGMAS, iters=REFINE_ITERS, lr=REFINE_LR, h=REFINE_H):
    """Adam on x in R^6 (C = C0 + x[:3], R = exp([x[3:]]) R0) with a central-difference gradient of dense_loss; the best
    pose seen at the finest blur is returned. 2 x 6 + 1 renders of every sweep pose per step."""
    s = can._save()
    try:
        x = np.zeros(6)
        pose = lambda x: (C0 + x[:3], _rotvec_to_mat(x[3:]) @ R0)
        hh = np.array([h[0]] * 3 + [h[1]] * 3)
        for k, sg in enumerate(sigmas):
            so = [_soft(rec[2], sg) for rec in recs]
            f = lambda x: dense_loss(can, recs, *pose(x), sg, so)
            step = np.array([lr[0]] * 3 + [lr[1]] * 3) / 2 ** k
            m, v = np.zeros(6), np.zeros(6)
            best = (f(x), x.copy())
            for t in range(1, iters + 1):
                g = np.array([(f(x + hh[i] * e) - f(x - hh[i] * e)) / (2 * hh[i]) for i, e in enumerate(np.eye(6))])
                m = 0.9 * m + 0.1 * g
                v = 0.999 * v + 0.001 * g * g
                x = x - step * (m / (1 - 0.9 ** t)) / (np.sqrt(v / (1 - 0.999 ** t)) + 1e-12)
                fx = f(x)
                if fx < best[0]:
                    best = (fx, x.copy())
            x = best[1]
        C, R = pose(x)
        so = [_soft(rec[2], sigmas[-1]) for rec in recs]
        out = dict(refine_loss_init=dense_loss(can, recs, C0, R0, sigmas[-1], so),
                   refine_loss_final=dense_loss(can, recs, C, R, sigmas[-1], so),
                   refine_loss_nominal=dense_loss(can, recs, can.Cn, can.Rn, sigmas[-1], so))
    finally:
        can._restore(s)
    return C, R, out


# ---------------------------------------------------------------- (iii) fiducial (oracle-corner) pose, AnyCamVLA's source
def marker_corners(pivot, table_z, offsets=FID_OFFSETS, side=FID_SIDE):
    """World corners (4 per marker, (M, 4, 3)) of square markers lying flat on the table around `pivot`."""
    c = np.array([[-1, -1], [1, -1], [1, 1], [-1, 1]]) * side / 2
    return np.array([[[pivot[0] + dx + a, pivot[1] + dy + b, table_z] for a, b in c] for dx, dy in offsets], float)


def _cv_from_mj(R):
    """MuJoCo camera axes (columns x right, y up, z back) -> OpenCV rotation world->camera (x right, y down, z forward)."""
    return np.diag([1.0, -1.0, -1.0]) @ R.T


def project_cv(X, C, R, f, H, W):
    """World points -> OpenCV pixels (u right, v down from the top-left pixel centre at 0) and depth, for a MuJoCo camera.
    The same pinhole as wrapper_libero.reproject (principal point at the image centre, square pixels)."""
    pc = (np.asarray(X, float).reshape(-1, 3) - C) @ _cv_from_mj(R).T
    uv = f * pc[:, :2] / pc[:, 2:3] + np.array([W / 2 - 0.5, H / 2 - 0.5])
    return uv, pc[:, 2]


def fiducial_pose(Ct, Rt, corners, f, H, W, sigma=FID_SIGMA_PX, rng=None):
    """PnP from the markers whose four corners all project inside the image and in front of the TRUE camera. Returns
    (C, R, info); (None, None, info) when no marker is fully visible."""
    rng = rng or np.random.default_rng(0)
    obj, img = [], []
    for M in corners:
        uv, z = project_cv(M, Ct, Rt, f, H, W)
        if (z > 1e-3).all() and (uv >= 0).all() and (uv[:, 0] <= W - 1).all() and (uv[:, 1] <= H - 1).all():
            obj.append(M)
            img.append(uv + sigma * rng.standard_normal(uv.shape))
    info = dict(fid_markers_visible=len(obj), fid_sigma_px=sigma)
    if not obj:
        return None, None, info
    K = np.array([[f, 0, W / 2 - 0.5], [0, f, H / 2 - 0.5], [0, 0, 1]], float)
    P, Q = np.concatenate(obj).astype(np.float64), np.concatenate(img).astype(np.float64)
    flag = cv2.SOLVEPNP_IPPE if len(obj) == 1 else cv2.SOLVEPNP_ITERATIVE
    ok, rvec, tvec = cv2.solvePnP(P, Q, K, None, flags=flag)
    if not ok:
        return None, None, info
    if len(obj) > 1:
        rvec, tvec = cv2.solvePnPRefineLM(P, Q, K, None, rvec, tvec)
    Rcv = cv2.Rodrigues(rvec)[0]
    R = Rcv.T @ np.diag([1.0, -1.0, -1.0])                  # inverse of _cv_from_mj
    C = -Rcv.T @ tvec.ravel()
    uv, _ = project_cv(P, C, R, f, H, W)
    info["fid_reproj_px"] = float(np.sqrt(np.mean(np.sum((uv - Q) ** 2, 1))))
    return C, R, info


# ---------------------------------------------------------------- entry point used by sweep_run.prepare_full
def estimate(source, can, recs, obs_hw, table_z_default):
    """source 'refine' | 'fiducial'. Returns (C, R, applied, info). `can.C_est, can.R_est` hold the silhouette pose (i)."""
    import wrapper_libero as W
    Ct, Rt = can.true_camera()
    C0, R0 = can.C_est.copy(), can.R_est.copy()
    info = dict(pose_source=source, cam_silhouette_err_cm_deg=list(W.pose_error(C0, R0, Ct, Rt)))
    if source == "refine":
        C, R, o = refine(can, recs, C0, R0)
        info.update(o)
        moved = max(_pose_error(C, R, can.Cn, can.Rn)) > 0.5          # cm or deg away from the training pose
        applied = bool(moved and o["refine_loss_final"] < ACCEPT * o["refine_loss_nominal"])
    elif source == "refine2":
        # POST HOC 2026-09-25: two starts, the silhouette pose and the training pose; keep the refined pose whose OWN
        # dense loss is lower (self-measured, no simulator truth). Guards against a wrong basin of the silhouette search.
        C1, R1, o1 = refine(can, recs, C0, R0)
        C2, R2, o2 = refine(can, recs, can.Cn.copy(), can.Rn.copy())
        first = o1["refine_loss_final"] <= o2["refine_loss_final"]
        C, R, o = (C1, R1, o1) if first else (C2, R2, o2)
        info.update(o, refine_start="silhouette" if first else "training",
                    refine_loss_from_silhouette=o1["refine_loss_final"], refine_loss_from_training=o2["refine_loss_final"])
        moved = max(_pose_error(C, R, can.Cn, can.Rn)) > 0.5
        applied = bool(moved and o["refine_loss_final"] < ACCEPT * o["refine_loss_nominal"])
    elif source == "fiducial":
        H, Wd = obs_hw
        tz = can.table_z if getattr(can, "table_z", None) is not None else table_z_default
        corners = marker_corners(W.optical_pivot(can.Cn, can.Rn, tz), tz)
        C, R, o = fiducial_pose(Ct, Rt, corners, can._focal(H), H, Wd)
        info.update(o)
        applied = C is not None
    else:
        raise ValueError(source)
    if not applied:
        C, R = can.Cn.copy(), can.Rn.copy()
    return C, R, applied, info

In [ ]:
%%bash
set -e
pip -q install uv
[ -x /content/venv_oft/bin/python ] || uv venv -q /content/venv_oft -p 3.10
cd /content
[ -d openvla-oft ] || git clone -q https://github.com/moojink/openvla-oft.git
uv pip install -q --python /content/venv_oft/bin/python -e /content/openvla-oft

apt-get -qq update > /dev/null
apt-get -qq install -y libexpat1 libfontconfig1-dev libmagickwand-dev unzip > /dev/null
cd /content
[ -d LIBERO-plus ] || git clone -q --depth 1 https://github.com/sylvestf/LIBERO-plus.git
uv pip install -q --python /content/venv_oft/bin/python -e /content/LIBERO-plus 
uv pip install -q --python /content/venv_oft/bin/python wand scikit-image
# LIBERO's setup.py does not register the top-level package, so the editable install alone is not importable
SITE=$(/content/venv_oft/bin/python -c "import sysconfig; print(sysconfig.get_paths()['purelib'])")
echo /content/LIBERO-plus > "$SITE/libero_plus.pth"
/content/venv_oft/bin/python -c "import importlib.util as u; assert u.find_spec('libero') is not None, 'libero still not importable'; print('libero importable')"
if [ ! -f /content/LIBERO-plus/.assets_ok ]; then
  echo "downloading LIBERO-Plus assets (6.4 GB)..."
  # Hugging Face answers a rate-limited or interrupted download with an error page or a short file; plain `curl -s`
  # saved that as the zip and unzip then failed with no message (2026-09-19). Fail on HTTP errors, resume, retry, send
  # the HF_TOKEN when there is one (anonymous downloads are limited first), and test the archive before unpacking.
  ZIP=/content/lp_assets.zip
  URL=https://huggingface.co/datasets/sii-research/LIBERO_plus_assets/resolve/main/assets.zip
  ok=0
  for try in 1 2 3 4 5 6; do
    if [ -f $ZIP ] && unzip -tq $ZIP > /dev/null 2>&1; then ok=1; break; fi
    size=$(stat -c %s $ZIP 2>/dev/null || echo 0)
    if [ $try -gt 1 ]; then echo "asset download attempt $((try-1)) incomplete ($size bytes), retrying in 30 s"; sleep 30; fi
    if [ "$size" -lt 1000000 ]; then rm -f $ZIP; fi                 # an error page cannot be resumed
    curl -fL -C - --retry 3 --retry-delay 15 -sS ${HF_TOKEN:+-H "Authorization: Bearer $HF_TOKEN"} -o $ZIP $URL || true
  done
  if [ $ok != 1 ]; then echo "LIBERO-Plus assets could not be downloaded (Hugging Face rate limit or outage). Wait a few minutes and re-run this cell; it resumes."; exit 1; fi
  rm -rf /content/lp_assets && mkdir -p /content/lp_assets
  unzip -q /content/lp_assets.zip -d /content/lp_assets && rm -f /content/lp_assets.zip
  python3 - <<'EOF'
import os, shutil
root, dst = "/content/lp_assets", "/content/LIBERO-plus/libero/libero/assets"
cands = [os.path.join(d, n) for d, ns, _ in os.walk(root) for n in ns if n == "assets"]
src = min(cands, key=len) if cands else root
os.makedirs(dst, exist_ok=True)
shutil.copytree(src, dst, dirs_exist_ok=True)
print("assets:", src, "->", dst, "|", len(os.listdir(dst)), "entries")
EOF
  touch /content/LIBERO-plus/.assets_ok
fi

uv pip install -q --python /content/venv_oft/bin/python -r /content/openvla-oft/experiments/robot/libero/libero_requirements.txt
uv pip install -q --python /content/venv_oft/bin/python "numpy<2" "mujoco==2.3.7"
uv pip install -q --python /content/venv_oft/bin/python "protobuf==4.25.3"
uv pip install -q --python /content/venv_oft/bin/python --no-deps "tensorflow-metadata==1.16.1"
/content/venv_oft/bin/python -c "import torch, robosuite, mujoco, cv2; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'mujoco', mujoco.__version__)"

In [ ]:
# LIBERO-Plus asks for its paths on first import; write them up front
import os
root = '/content/LIBERO-plus/libero/libero'
os.makedirs(os.path.expanduser('~/.libero'), exist_ok=True)
open(os.path.expanduser('~/.libero/config.yaml'), 'w').write(
    f'benchmark_root: {root}\nbddl_files: {root}/bddl_files\ninit_states: {root}/init_files\n'
    f'datasets: {root}/../datasets\nassets: {root}/assets\n')
os.environ['MUJOCO_GL'] = 'egl'
os.environ['MPLBACKEND'] = 'Agg'   # Colab's inline backend is not valid inside the venvs
os.environ['PYTHONPATH'] = '/content/LIBERO-plus'   # Colab's system python ignores the .pth file
print(open(os.path.expanduser('~/.libero/config.yaml')).read())

In [ ]:
# the SAME task subset for every family (built once, then read from Drive)
!MUJOCO_GL=egl /content/venv_oft/bin/python -c "import sys; sys.path.insert(0, '/content/scripts'); import plus_common as pc; s = pc.load_or_build('/content/drive/MyDrive/restoring_organs/plus/subset_libero10.json'); print('suite tasks', s['n_tasks']); print('per category', s['counts']); print('name tokens', s['name_tokens']); print('other examples', s['other_examples']); print(len(s['tasks']), 'tasks in subset')"

In [ ]:
# validate the camera channel on 3 camera tasks (paste this output)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; mkdir -p /content/drive/MyDrive/restoring_organs/plus/wrapper_oft && cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/eval_oft_wrapper.py validate --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/rows.jsonl --n 3 2>&1 | tee -a /content/logs/eval_oft_wrapper.log

In [ ]:
# calibration-only check, no policy (~1 to 2 min per task): old vs new camera calibration (paste the table)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/calib_check.py --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/calib_check.jsonl 2>&1 | tee -a /content/logs/calib_check.log

In [ ]:
%%bash
# claim 3, step 1: self-labelled pairs for the learned canonicalizer, 4 parallel workers (~1 h, resumable, no policy)
mkdir -p /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/canon_data_v2
# v2 shards also store the true nominal render (training-time target only); older canon_data shards lack it
# build the per-scene nominal cache ONCE before the workers start (four workers rebuilding it at once would race on the file)
MUJOCO_GL=egl PYTHONPATH=/content/LIBERO-plus MPLBACKEND=Agg /content/venv_oft/bin/python -c "import sys; sys.path.insert(0, '/content/scripts'); import wrapper_libero as W, calib_check as C; from libero.libero import benchmark; s = benchmark.get_benchmark_dict()['libero_10'](); W.read_nominals(s, lambda i: C.make_env(s, i), cache='/content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json'); print('nominal cache ok')"
for i in 0 1 2 3; do
  MUJOCO_GL=egl PYTHONPATH=/content/LIBERO-plus MPLBACKEND=Agg /content/venv_oft/bin/python /content/scripts/collect_canon.py --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/canon_data_v2 --worker $i --workers 4 > /content/collect_$i.log 2>&1 &
  sleep 60
done
wait
tail -n 3 /content/collect_*.log
ls /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/canon_data_v2 | wc -l

In [ ]:
# claim 3, step 2: train the learned canonicalizer: input = the wrapper's own reprojection + hole mask + frame (measured geometry), target = the oracle nominal render (privileged at training time only; ~30 min on A100). Add --input rgb for the learning-alone ablation
# needs canon_data_v2 shards: older canon_data shards lack the oracle render and the trainer refuses them
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/train_canon.py --data_dir /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/canon_data_v2 --out /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/canon_net_oracle.pt --target oracle --input reproj 2>&1 | tee -a /content/logs/train_canon.log

In [ ]:
# no simulator shortcuts, step 1: train the robot segmenter on the same held-out shards (~15 min; skip if seg_net.pt already exists)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/train_seg.py --data_dir /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/canon_data_v2 --out /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/seg_net.pt 2>&1 | tee -a /content/logs/train_seg.log

In [ ]:
# wrapper run in priority order (resumable): camera full (wider calibration + reference fill), camera cam_true_ref, learned (if trained), light, noise, layout, background, language; finished robot_init / oracle / cam_true rows are kept
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/eval_oft_wrapper.py run --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/rows.jsonl --baseline /content/drive/MyDrive/restoring_organs/plus/rows_oft.jsonl --canon_ckpt /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/canon_net_oracle.pt 2>&1 | tee -a /content/logs/eval_oft_wrapper.log

In [ ]:
# camera diagnosis (standalone: works right after a restart, needs only Drive): true shift, calibration error, coverage, per-arm success
from google.colab import drive; drive.mount('/content/drive')
import json, collections
P = '/content/drive/MyDrive/restoring_organs/plus'
rows = [json.loads(l) for l in open(f'{P}/wrapper_oft/rows.jsonl') if l.strip()]
base = {json.loads(l)['task_id']: json.loads(l)['success'] for l in open(f'{P}/rows_oft.jsonl') if l.strip()}
by = collections.defaultdict(dict)
for r in rows:
    if r['category'] == 'camera':
        by[r['task_id']][r['arm']] = r
nan2 = [float('nan')] * 2
for t, a in sorted(by.items()):
    f = a.get('full') or a.get('full2') or {}
    s = f.get('cam_true_shift_cm_deg') or f.get('cam_true_err_cm_deg') or nan2
    e = f.get('cam_est_err_cm_deg') or nan2
    cov = a.get('cam_true', {}).get('reproj_coverage', float('nan'))
    ok = lambda arm: int(a.get(arm, {}).get('success', 0)) if arm in a else '-'
    print(t, (f.get('name') or '')[:26], '| shift %.0fcm %.0fdeg' % tuple(s[:2]), '| est err %.0fcm %.0fdeg' % tuple(e[:2]),
          '| cov %.2f' % cov, '| base', int(base.get(t, 0)), 'oracle', ok('oracle'), 'true', ok('cam_true'),
          'true_ref', ok('cam_true_ref'), 'full', int(f.get('success', 0)) if f else '-')

In [ ]:
# camera significance: 30 MORE camera tasks outside every subset, paired raw vs full (~2.5 h on A100, resumable)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --arms raw,full --categories camera --per_cat_scale 0 --extra_from_suite 30 2>&1 | tee -a /content/logs/wrap_run.log

In [ ]:
# pooled paired test for the deployable camera fix (paste this)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/camera_pooled.py --baseline /content/drive/MyDrive/restoring_organs/plus/rows_oft.jsonl --wrapper /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/rows.jsonl --ext /content/drive/MyDrive/restoring_organs/plus/wraps/wrap_openvla_oft.jsonl 2>&1 | tee -a /content/logs/camera_pooled.log

In [ ]:
# self-identifiability, policy-free: local sensitivity per scene (~10 min), then per-task index (~45 min, resumable)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/identifiability.py local --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --out /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/identifiability_local.json 2>&1 | tee -a /content/logs/identifiability.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/identifiability.py tasks --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --out /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/identifiability_tasks.jsonl --wrapper /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/rows.jsonl --baseline /content/drive/MyDrive/restoring_organs/plus/rows_oft.jsonl 2>&1 | tee -a /content/logs/identifiability.log

In [ ]:
# no simulator shortcuts, step 2: OFT with LEARNED sensing (segmenter + Depth Anything scaled on the robot), camera / robot_init / light / noise (~5 h on A100, resumable)
# OFT pins its own transformers fork (4.40): do not upgrade it; Depth Anything V1 loads there (V2 falls back automatically)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --arms full_learned --categories camera,robot_init,light,sensor_noise --seg_ckpt /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/seg_net.pt 2>&1 | tee -a /content/logs/wrap_run.log

In [ ]:
%%bash
# measurement plus learning, step 1: shards with LEARNED sensing (segmenter + table plane), so the canonicalizer learns to repair the deployable sensing's artefacts (~1 h, 4 workers, resumable)
D=/content/drive/MyDrive/restoring_organs/plus
mkdir -p $D/wrapper_oft/canon_data_v3_plane
for i in 0 1 2 3; do
  MUJOCO_GL=egl PYTHONPATH=/content/LIBERO-plus MPLBACKEND=Agg /content/venv_oft/bin/python /content/scripts/collect_canon.py --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --out_dir $D/wrapper_oft/canon_data_v3_plane --seg_ckpt /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/seg_net.pt --depth plane --worker $i --workers 4 > /content/collect_plane_$i.log 2>&1 &
  sleep 60
done
wait
tail -n 3 /content/collect_plane_*.log
ls $D/wrapper_oft/canon_data_v3_plane | wc -l

In [ ]:
# measurement plus learning, step 2: train on the learned-sensing shards, oracle target (~30 min)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/train_canon.py --data_dir /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/canon_data_v3_plane --out /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/canon_net_plane.pt --target oracle --input reproj 2>&1 | tee -a /content/logs/train_canon.log

In [ ]:
# measurement plus learning, step 3: the learned arm with LEARNED sensing (segmenter + plane depth + canonicalizer), camera tasks (~1 h)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/eval_oft_wrapper.py run --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/rows.jsonl --baseline /content/drive/MyDrive/restoring_organs/plus/rows_oft.jsonl --canon_ckpt /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/canon_net_plane.pt --seg_ckpt /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/seg_net.pt --depth plane --skip_arms full,cam_true_ref,oracle,cam_true,state 2>&1 | tee -a /content/logs/eval_oft_wrapper.log

In [ ]:
# no simulator shortcuts, step 3: OFT with the learned segmenter and NO depth network (training-setup table plane), camera tasks (~1.5 h on A100, resumable)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --arms full_plane --categories camera --seg_ckpt /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/seg_net.pt 2>&1 | tee -a /content/logs/wrap_run.log

In [ ]:
# learned sensing root cause, step 1 (DIAGNOSTIC, ~1 h on A100, resumable): the six tasks simulator sensing solves and learned sensing loses, every arm in the SAME harness. full = simulator sensing control; *_truepose = same sensing with the simulator's camera pose (upper bounds, never reported as deployable)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --categories camera --seg_ckpt /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/seg_net.pt --tasks 768,800,803,956,958,1060 --redo --arms full,full_plane,full_plane_truepose,full_hybrid,full_hybrid_truepose 2>&1 | tee -a /content/logs/wrap_run.log

In [ ]:
# learned sensing root cause, step 2: the table (paste this)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/diag_learned.py --rows /content/drive/MyDrive/restoring_organs/plus/wraps/wrap_openvla_oft.jsonl --tasks 768,800,803,956,958,1060 2>&1 | tee -a /content/logs/diag_learned.log

In [ ]:
# learned sensing root cause, step 3 (only after step 2 is read): hybrid depth on all 30 camera tasks (~1.5 h on A100, resumable)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --arms full_hybrid --categories camera --seg_ckpt /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/seg_net.pt 2>&1 | tee -a /content/logs/wrap_run.log

In [ ]:
# graded sweep, step 1: policy-free recoverability scores, scores_v2 (own-block reference + coverage / shading / denoiser components; 3 base tasks x 24 conditions + 21 reference instances, ~45 min, no policy). The v1 scores.jsonl normalised every family by the camera block and is kept only for the record
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/sweep_run.py score --n_tasks 3 --out_dir /content/drive/MyDrive/restoring_organs/plus/sweep --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json 2>&1 | tee -a /content/logs/sweep_run.log

In [ ]:
# graded sweep, step 2: OpenVLA-OFT episodes (3 base tasks x 24 conditions x raw/full, ~4 h on A100, resumable)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/sweep_run.py episodes --family oft --n_tasks 3 --out_dir /content/drive/MyDrive/restoring_organs/plus/sweep --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json 2>&1 | tee -a /content/logs/sweep_run.log

In [ ]:
# four suites, libero_spatial: OFT checkpoint for this suite, 20 robot_init + 20 camera tasks, paired raw / full (~3 h on A100, resumable). The subset and the nominal camera for the suite are built on first use
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && LIBERO_SUITE=libero_spatial MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero_spatial.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_libero_spatial --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_libero_spatial.json --arms raw,full --categories robot_init,camera --per_cat_scale 0.67 2>&1 | tee -a /content/logs/wrap_run.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/arms_report.py --rows /content/drive/MyDrive/restoring_organs/plus/wraps_libero_spatial/wrap_openvla_oft.jsonl --arms raw,full 2>&1 | tee -a /content/logs/arms_report.log

In [ ]:
# four suites, libero_object: OFT checkpoint for this suite, 20 robot_init + 20 camera tasks, paired raw / full (~3.5 h on A100, resumable). The subset and the nominal camera for the suite are built on first use
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && LIBERO_SUITE=libero_object MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero_object.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_libero_object --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_libero_object.json --arms raw,full --categories robot_init,camera --per_cat_scale 0.67 2>&1 | tee -a /content/logs/wrap_run.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/arms_report.py --rows /content/drive/MyDrive/restoring_organs/plus/wraps_libero_object/wrap_openvla_oft.jsonl --arms raw,full 2>&1 | tee -a /content/logs/arms_report.log

In [ ]:
# four suites, libero_goal: OFT checkpoint for this suite, 20 robot_init + 20 camera tasks, paired raw / full (~3.5 h on A100, resumable). The subset and the nominal camera for the suite are built on first use
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && LIBERO_SUITE=libero_goal MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero_goal.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_libero_goal --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_libero_goal.json --arms raw,full --categories robot_init,camera --per_cat_scale 0.67 2>&1 | tee -a /content/logs/wrap_run.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/arms_report.py --rows /content/drive/MyDrive/restoring_organs/plus/wraps_libero_goal/wrap_openvla_oft.jsonl --arms raw,full 2>&1 | tee -a /content/logs/arms_report.log

In [ ]:
# robot_init base rate (reviewer: is the joint result just a reset?), ~1.5 h on A100 for all three suites, resumable. nominal_start = the arm PLACED at the training start pose (no servo, no correction), then the raw policy, on the SAME 20 robot_init tasks and seeds per suite whose raw / full rows exist. Paste the three reports
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && LIBERO_SUITE=libero_spatial MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero_spatial.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_libero_spatial --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_libero_spatial.json --arms nominal_start --categories robot_init --per_cat_scale 0.67 2>&1 | tee -a /content/logs/wrap_run.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/arms_report.py --rows /content/drive/MyDrive/restoring_organs/plus/wraps_libero_spatial/wrap_openvla_oft.jsonl --arms raw,full,nominal_start 2>&1 | tee -a /content/logs/arms_report.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && LIBERO_SUITE=libero_object MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero_object.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_libero_object --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_libero_object.json --arms nominal_start --categories robot_init --per_cat_scale 0.67 2>&1 | tee -a /content/logs/wrap_run.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/arms_report.py --rows /content/drive/MyDrive/restoring_organs/plus/wraps_libero_object/wrap_openvla_oft.jsonl --arms raw,full,nominal_start 2>&1 | tee -a /content/logs/arms_report.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && LIBERO_SUITE=libero_goal MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero_goal.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_libero_goal --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_libero_goal.json --arms nominal_start --categories robot_init --per_cat_scale 0.67 2>&1 | tee -a /content/logs/wrap_run.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/arms_report.py --rows /content/drive/MyDrive/restoring_organs/plus/wraps_libero_goal/wrap_openvla_oft.jsonl --arms raw,full,nominal_start 2>&1 | tee -a /content/logs/arms_report.log

In [ ]:
# camera extension, libero_spatial: 40 MORE camera tasks from the full suite, paired raw / full (~2 h on A100, resumable). These checkpoints survive most camera shifts, so the camera claim needs more failures to recover; the report pools them with the first 20
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && LIBERO_SUITE=libero_spatial MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero_spatial.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_libero_spatial --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_libero_spatial.json --arms raw,full --categories camera --per_cat_scale 0 --extra_from_suite 40 2>&1 | tee -a /content/logs/wrap_run.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/arms_report.py --rows /content/drive/MyDrive/restoring_organs/plus/wraps_libero_spatial/wrap_openvla_oft.jsonl --arms raw,full 2>&1 | tee -a /content/logs/arms_report.log

In [ ]:
# camera extension, libero_object: 40 MORE camera tasks from the full suite, paired raw / full (~2 h on A100, resumable). These checkpoints survive most camera shifts, so the camera claim needs more failures to recover; the report pools them with the first 20
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && LIBERO_SUITE=libero_object MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero_object.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_libero_object --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_libero_object.json --arms raw,full --categories camera --per_cat_scale 0 --extra_from_suite 40 2>&1 | tee -a /content/logs/wrap_run.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/arms_report.py --rows /content/drive/MyDrive/restoring_organs/plus/wraps_libero_object/wrap_openvla_oft.jsonl --arms raw,full 2>&1 | tee -a /content/logs/arms_report.log

In [ ]:
# camera extension, libero_goal: 40 MORE camera tasks from the full suite, paired raw / full (~2 h on A100, resumable). These checkpoints survive most camera shifts, so the camera claim needs more failures to recover; the report pools them with the first 20
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && LIBERO_SUITE=libero_goal MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero_goal.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_libero_goal --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_libero_goal.json --arms raw,full --categories camera --per_cat_scale 0 --extra_from_suite 40 2>&1 | tee -a /content/logs/wrap_run.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/arms_report.py --rows /content/drive/MyDrive/restoring_organs/plus/wraps_libero_goal/wrap_openvla_oft.jsonl --arms raw,full 2>&1 | tee -a /content/logs/arms_report.log

In [ ]:
# tolerance probe, OFT (~25 min, resumable, NO rollouts): how wrong may the camera pose be before the policy's opening action leaves what it does under a perfect calibration? True pose plus a known residual (0.5 to 16 cm, 0.5 to 8 deg per camera axis) on 10 base tasks x 2 orbit conditions; prints the dose-response table and where the robot's own calibration error sits on it (paste the report). Own folder
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/tolerance_probe.py run --family oft --n_tasks 10 --out_dir /content/drive/MyDrive/restoring_organs/plus/tolerance --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json 2>&1 | tee -a /content/logs/tolerance_probe.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/tolerance_probe.py report --out_dir /content/drive/MyDrive/restoring_organs/plus/tolerance 2>&1 | tee -a /content/logs/tolerance_probe.log

In [ ]:
# ladder 1, libero_10 (~35 min, resumable): the two UPPER RUNGS of the attribution ladder, only on camera tasks that fail under both raw and full. full_truepose = the same correction with the simulator's true camera pose (was it the measurement?), full_oracle = a perfect render from the training camera (was it the lost view? what still fails here is the policy). Writes to its OWN folder
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_ladder_libero_10 --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --arms full_truepose,full_oracle --categories camera --per_cat_scale 0 --extra_from_suite 30 --only_failed_in /content/drive/MyDrive/restoring_organs/plus/wraps/wrap_openvla_oft.jsonl 2>&1 | tee -a /content/logs/wrap_run.log

In [ ]:
# ladder 2, libero_object (~35 min, resumable): upper rungs on the object camera tasks that fail under both raw and full
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && LIBERO_SUITE=libero_object MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero_object.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_ladder_libero_object --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_libero_object.json --arms full_truepose,full_oracle --categories camera --per_cat_scale 0.67 --extra_from_suite 40 --only_failed_in /content/drive/MyDrive/restoring_organs/plus/wraps_libero_object/wrap_openvla_oft.jsonl 2>&1 | tee -a /content/logs/wrap_run.log

In [ ]:
# ladder 3, libero_goal (~30 min, resumable): upper rungs on the goal camera tasks that fail under both raw and full
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && LIBERO_SUITE=libero_goal MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero_goal.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_ladder_libero_goal --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_libero_goal.json --arms full_truepose,full_oracle --categories camera --per_cat_scale 0.67 --extra_from_suite 40 --only_failed_in /content/drive/MyDrive/restoring_organs/plus/wraps_libero_goal/wrap_openvla_oft.jsonl 2>&1 | tee -a /content/logs/wrap_run.log

In [ ]:
# ladder 4, libero_spatial (~15 min, resumable): upper rungs on the spatial camera tasks that fail under both raw and full
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && LIBERO_SUITE=libero_spatial MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero_spatial.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_ladder_libero_spatial --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_libero_spatial.json --arms full_truepose,full_oracle --categories camera --per_cat_scale 0.67 --extra_from_suite 40 --only_failed_in /content/drive/MyDrive/restoring_organs/plus/wraps_libero_spatial/wrap_openvla_oft.jsonl 2>&1 | tee -a /content/logs/wrap_run.log

In [ ]:
# ladder: report (paste all of it). Attributes every raw camera failure to the first rung that repairs it
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/attribution_ladder.py --set libero_10_first30 /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/rows.jsonl /content/drive/MyDrive/restoring_organs/plus/rows_oft.jsonl --set libero_10_extra30 /content/drive/MyDrive/restoring_organs/plus/wraps/wrap_openvla_oft.jsonl /content/drive/MyDrive/restoring_organs/plus/wraps_ladder_libero_10/wrap_openvla_oft.jsonl --set object /content/drive/MyDrive/restoring_organs/plus/wraps_libero_object/wrap_openvla_oft.jsonl /content/drive/MyDrive/restoring_organs/plus/wraps_ladder_libero_object/wrap_openvla_oft.jsonl --set goal /content/drive/MyDrive/restoring_organs/plus/wraps_libero_goal/wrap_openvla_oft.jsonl /content/drive/MyDrive/restoring_organs/plus/wraps_ladder_libero_goal/wrap_openvla_oft.jsonl --set spatial /content/drive/MyDrive/restoring_organs/plus/wraps_libero_spatial/wrap_openvla_oft.jsonl /content/drive/MyDrive/restoring_organs/plus/wraps_ladder_libero_spatial/wrap_openvla_oft.jsonl 2>&1 | tee -a /content/logs/attribution_ladder.log

In [ ]:
# camera power 1, libero_10 (~2.5 h, resumable): three NEW arms on the 30 extra camera tasks whose raw / full / tta rows exist: full_settle (hand the arm over at rest, where the episode began), full_verify (full_settle + measure again with 7 poses when the robot's own self-check rejects the calibration), full_tta (the wrapper with the policy averaged over jitters of the CORRECTED frame)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --arms raw,full,tta,full_settle,full_verify,full_tta --categories camera --per_cat_scale 0 --extra_from_suite 30 2>&1 | tee -a /content/logs/wrap_run.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/arms_report.py --rows /content/drive/MyDrive/restoring_organs/plus/wraps/wrap_openvla_oft.jsonl --arms raw,full,tta,full_settle,full_verify,full_tta 2>&1 | tee -a /content/logs/arms_report.log

In [ ]:
# camera power 2, libero_object (~6 h, resumable): tta and the three new arms on all 60 object camera tasks (raw / full exist), so the wrapper is compared with test-time augmentation on a second suite
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && LIBERO_SUITE=libero_object MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero_object.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_libero_object --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_libero_object.json --arms raw,full,tta,full_settle,full_verify,full_tta --categories camera --per_cat_scale 0.67 --extra_from_suite 40 2>&1 | tee -a /content/logs/wrap_run.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/arms_report.py --rows /content/drive/MyDrive/restoring_organs/plus/wraps_libero_object/wrap_openvla_oft.jsonl --arms raw,full,tta,full_settle,full_verify,full_tta 2>&1 | tee -a /content/logs/arms_report.log

In [ ]:
# camera power 3, libero_goal (~5 h, resumable): 40 more goal camera tasks, all arms
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && LIBERO_SUITE=libero_goal MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero_goal.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_libero_goal --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_libero_goal.json --arms raw,full,tta,full_verify --categories camera --per_cat_scale 0.67 --extra_from_suite 40 2>&1 | tee -a /content/logs/wrap_run.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/arms_report.py --rows /content/drive/MyDrive/restoring_organs/plus/wraps_libero_goal/wrap_openvla_oft.jsonl --arms raw,full,tta,full_verify 2>&1 | tee -a /content/logs/arms_report.log

In [ ]:
# baselines, camera (OFT, libero_10): test-time augmentation, generic corrections, a depth-free homography with the measured pose, and the wrapper without the reference fill, on the 30 extra camera tasks whose raw / full rows already exist in this harness (~7 h on A100, resumable)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --arms raw,full,tta,generic,full_homog,full_nofill --categories camera --per_cat_scale 0 --extra_from_suite 30 2>&1 | tee -a /content/logs/wrap_run.log

In [ ]:
# baselines, generic corrections where OFT is already robust (light / sensor_noise / background, 30 tasks, raw vs generic, ~3 h): does an unmeasured correction hurt a policy that was coping?
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --arms raw,generic --categories light,sensor_noise,background 2>&1 | tee -a /content/logs/wrap_run.log

In [ ]:
# baselines: report (paste all of it)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/arms_report.py --rows /content/drive/MyDrive/restoring_organs/plus/wraps/wrap_openvla_oft.jsonl --arms raw,full,tta,generic,full_homog,full_nofill 2>&1 | tee -a /content/logs/arms_report.log

In [ ]:
# self-motion consistency, OFT probe (NO rollouts: ~1 min per unit, 72 units, ~1.5 h on A100, resumable). At 7 known end-effector positions the frozen policy is asked for its action on the raw and on the corrected frame; rays that stop meeting at one sub-goal mean the policy is not perceiving correctly
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/smc_probe.py run --family oft --n_tasks 3 --out_dir /content/drive/MyDrive/restoring_organs/plus/sweep --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json 2>&1 | tee -a /content/logs/smc_probe.log

In [ ]:
# self-motion consistency: report (joins the probe with the raw / full outcomes already on Drive; paste all of it)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/smc_probe.py report --out_dir /content/drive/MyDrive/restoring_organs/plus/sweep 2>&1 | tee -a /content/logs/smc_probe.log

In [ ]:
# scale-up, step 1: policy-free recoverability scores for all 10 base tasks (7 new, ~2 h, no policy, resumable)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/sweep_run.py score --n_tasks 10 --out_dir /content/drive/MyDrive/restoring_organs/plus/sweep --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json 2>&1 | tee -a /content/logs/sweep_run.log

In [ ]:
# scale-up, step 2: OFT on all 10 base tasks with the gated arm (7 new tasks x 24 conditions x raw/full + gated on all 10, ~11 h on A100, resumable). OFT is the DEVELOPMENT family: its gate numbers are descriptive, the test is OpenVLA
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/sweep_run.py episodes --family oft --n_tasks 10 --arms raw,full,gated --out_dir /content/drive/MyDrive/restoring_organs/plus/sweep --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json 2>&1 | tee -a /content/logs/sweep_run.log

In [ ]:
# do-no-harm gate on OFT: report (paste all of it)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/gate_report.py --rows /content/drive/MyDrive/restoring_organs/plus/sweep/episodes_openvla_oft.jsonl 2>&1 | tee -a /content/logs/gate_report.log

In [ ]:
# graded sweep, step 3: does the score predict recovery? (reads every family's episodes on Drive; paste this)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/sweep_analysis.py --out_dir /content/drive/MyDrive/restoring_organs/plus/sweep 2>&1 | tee -a /content/logs/sweep_analysis.log

In [ ]:
# claims 1 and 2: invertibility report (paste this)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/invertibility_report.py --family openvla_oft:/content/drive/MyDrive/restoring_organs/plus/rows_oft.jsonl:/content/drive/MyDrive/restoring_organs/plus/wrapper_oft/rows.jsonl --out /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/invertibility_report.json 2>&1 | tee -a /content/logs/invertibility_report.log

In [ ]:
# nominal base rate 1, robot_init (~2.5 h on A100, resumable). nominal_start = the arm PLACED at the training start (no servo, nothing measured), then the raw policy: the unperturbed task on the SAME tasks and seeds. libero_10: raw, full AND nominal_start in this harness, because the Table 2 libero_10 row came from the older eval_oft_wrapper harness and single rollouts are harness-sensitive. spatial / object / goal: nominal_start only, paired with the raw / full rows of the four-suites cells (same harness, seed = task id). Own folders plus/wraps_nominal_<suite>
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_nominal_libero_10 --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --arms raw,full,nominal_start --categories robot_init 2>&1 | tee -a /content/logs/wrap_run.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && LIBERO_SUITE=libero_spatial MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero_spatial.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_nominal_libero_spatial --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_libero_spatial.json --arms nominal_start --categories robot_init --per_cat_scale 0.67 2>&1 | tee -a /content/logs/wrap_run.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && LIBERO_SUITE=libero_object MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero_object.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_nominal_libero_object --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_libero_object.json --arms nominal_start --categories robot_init --per_cat_scale 0.67 2>&1 | tee -a /content/logs/wrap_run.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && LIBERO_SUITE=libero_goal MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero_goal.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_nominal_libero_goal --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_libero_goal.json --arms nominal_start --categories robot_init --per_cat_scale 0.67 2>&1 | tee -a /content/logs/wrap_run.log

In [ ]:
# nominal base rate 2, ladder sets (~1.5 h on A100, resumable). nominal_view = every frame is the simulator's render from the TRAINING camera, nothing measured, the arm untouched: the unperturbed task at the same layout, start state and seed. Set A (first 30 libero_10 camera tasks) in full; set B and the other suites only on the ladder units (fail under both raw and full), with EXACTLY the ladder cells' task selection. Same own folders as cell 1
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_nominal_libero_10 --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --arms nominal_view --categories camera 2>&1 | tee -a /content/logs/wrap_run.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_nominal_libero_10 --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --arms nominal_view --categories camera --per_cat_scale 0 --extra_from_suite 30 --only_failed_in /content/drive/MyDrive/restoring_organs/plus/wraps/wrap_openvla_oft.jsonl 2>&1 | tee -a /content/logs/wrap_run.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && LIBERO_SUITE=libero_object MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero_object.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_nominal_libero_object --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_libero_object.json --arms nominal_view --categories camera --per_cat_scale 0.67 --extra_from_suite 40 --only_failed_in /content/drive/MyDrive/restoring_organs/plus/wraps_libero_object/wrap_openvla_oft.jsonl 2>&1 | tee -a /content/logs/wrap_run.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && LIBERO_SUITE=libero_goal MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero_goal.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_nominal_libero_goal --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_libero_goal.json --arms nominal_view --categories camera --per_cat_scale 0.67 --extra_from_suite 40 --only_failed_in /content/drive/MyDrive/restoring_organs/plus/wraps_libero_goal/wrap_openvla_oft.jsonl 2>&1 | tee -a /content/logs/wrap_run.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && LIBERO_SUITE=libero_spatial MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero_spatial.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_nominal_libero_spatial --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_libero_spatial.json --arms nominal_view --categories camera --per_cat_scale 0.67 --extra_from_suite 40 --only_failed_in /content/drive/MyDrive/restoring_organs/plus/wraps_libero_spatial/wrap_openvla_oft.jsonl 2>&1 | tee -a /content/logs/wrap_run.log

In [ ]:
# nominal base rate 3: report (paste all of it). robot_init: nominal_start against raw and full, same tasks and seeds (McNemar vs nominal_start)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_oft/bin/python /content/scripts/arms_report.py --rows /content/drive/MyDrive/restoring_organs/plus/wraps_nominal_libero_10/wrap_openvla_oft.jsonl --arms raw,full,nominal_start --ref nominal_start 2>&1 | tee -a /content/logs/arms_report.log
!cat /content/drive/MyDrive/restoring_organs/plus/wraps_libero_spatial/wrap_openvla_oft.jsonl /content/drive/MyDrive/restoring_organs/plus/wraps_nominal_libero_spatial/wrap_openvla_oft.jsonl > /content/nominal_spatial.jsonl
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; echo libero_spatial; /content/venv_oft/bin/python /content/scripts/arms_report.py --rows /content/nominal_spatial.jsonl --arms raw,full,nominal_start --ref nominal_start 2>&1 | tee -a /content/logs/arms_report.log
!cat /content/drive/MyDrive/restoring_organs/plus/wraps_libero_object/wrap_openvla_oft.jsonl /content/drive/MyDrive/restoring_organs/plus/wraps_nominal_libero_object/wrap_openvla_oft.jsonl > /content/nominal_object.jsonl
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; echo libero_object; /content/venv_oft/bin/python /content/scripts/arms_report.py --rows /content/nominal_object.jsonl --arms raw,full,nominal_start --ref nominal_start 2>&1 | tee -a /content/logs/arms_report.log
!cat /content/drive/MyDrive/restoring_organs/plus/wraps_libero_goal/wrap_openvla_oft.jsonl /content/drive/MyDrive/restoring_organs/plus/wraps_nominal_libero_goal/wrap_openvla_oft.jsonl > /content/nominal_goal.jsonl
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; echo libero_goal; /content/venv_oft/bin/python /content/scripts/arms_report.py --rows /content/nominal_goal.jsonl --arms raw,full,nominal_start --ref nominal_start 2>&1 | tee -a /content/logs/arms_report.log

In [ ]:
# nominal base rate 4: the ladder sets, unperturbed view against the perfect-render rung (seconds; paste all of it)
import json, os
D = '/content/drive/MyDrive/restoring_organs/plus'
import glob; rows = lambda p: [json.loads(l) for q in [p] + sorted(glob.glob(p[:-6] + '.*.jsonl')) if os.path.exists(q) for l in open(q) if l.strip()]
for s in ('libero_10', 'libero_object', 'libero_goal', 'libero_spatial'):
    nv = {r['task_id']: bool(r['success']) for r in rows(f'{D}/wraps_nominal_{s}/wrap_openvla_oft.jsonl') if r['arm'] == 'nominal_view'}
    orc = {r['task_id']: bool(r['success']) for r in rows(f'{D}/wraps_ladder_{s}/wrap_openvla_oft.jsonl') if r['arm'] == 'full_oracle'}
    if s == 'libero_10':   # set A's perfect-render rung is in the older harness (cross-harness pairing, said in the paper if used)
        orc.update({r['task_id']: bool(r['success']) for r in rows(f'{D}/wrapper_oft/rows.jsonl') if r.get('arm') == 'oracle' and r['category'] == 'camera'})
    b = sorted(set(nv) & set(orc))
    print(s, '| nominal_view', sum(nv.values()), 'of', len(nv), '| paired with the perfect render:', len(b), 'units: both', sum(nv[t] and orc[t] for t in b),
          'unperturbed only', sum(nv[t] and not orc[t] for t in b), 'render only', sum(orc[t] and not nv[t] for t in b), 'neither', sum(not (nv[t] or orc[t]) for t in b))

In [ ]:
# 3D renders 1, smoke (~3 min, no policy): one job at 256 px into /content only. It must print 'anchor check' >= 0.90; paste the line and STOP on an error
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/render_paper_views.py --smoke --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --out_dir /content/render_smoke 2>&1 | tee -a /content/logs/render_paper_views.log

In [ ]:
# 3D renders 2 (~15 min, no policy, 512 px): god's-eye scene with the training camera and the four Fig. 1 cameras as frusta, the joint-offset start against the training start, the 7 fingerprint probe poses. Own folder plus/paper_renders; paste the three 'wrote' lines
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/render_paper_views.py --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --out_dir /content/drive/MyDrive/restoring_organs/plus/paper_renders 2>&1 | tee -a /content/logs/render_paper_views.log
!ls -la /content/drive/MyDrive/restoring_organs/plus/paper_renders

In [ ]:
# metric depth 1, smoke (~10 min): full_metric on 2 set A tasks, own folder. It must print 'learned sensing: segmenter ... metric depth ...' and two episode lines; paste them and STOP on an error
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_metric_smoke --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --arms full_metric --categories camera --seg_ckpt /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/seg_net.pt --tasks 699,707 2>&1 | tee -a /content/logs/wrap_run.log

In [ ]:
# metric depth 2 (~1.5 h on A100, resumable): full_metric (learned segmenter + Depth Anything V2 metric, used as is) on all 30 set A camera tasks. Same harness and seeds as full_plane / full_hybrid in plus/wraps; own folder plus/wraps_metric
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_metric --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --arms full_metric --categories camera --seg_ckpt /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/seg_net.pt 2>&1 | tee -a /content/logs/wrap_run.log

In [ ]:
# metric depth 3: report (seconds; paste all of it). Calibration error against the simulator's pose and success, per learned-sensing arm on the same 30 tasks
import json, os, numpy as np
D = '/content/drive/MyDrive/restoring_organs/plus'
import glob; rows = lambda p: [json.loads(l) for q in [p] + sorted(glob.glob(p[:-6] + '.*.jsonl')) if os.path.exists(q) for l in open(q) if l.strip()]
R = rows(f'{D}/wraps_metric/wrap_openvla_oft.jsonl') + [r for r in rows(f'{D}/wraps/wrap_openvla_oft.jsonl') if r['arm'] in ('full_plane', 'full_hybrid')]
ids = {r['task_id'] for r in R if r['arm'] == 'full_metric' and r['category'] == 'camera'}
for arm in ('full_plane', 'full_hybrid', 'full_metric'):
    a = {r['task_id']: r for r in R if r['arm'] == arm and r['task_id'] in ids}
    e = np.array([a[t]['cam_est_err_cm_deg'][:2] for t in a if a[t].get('cam_est_err_cm_deg')], float).reshape(-1, 2)
    print(arm, 'n', len(a), 'success', sum(bool(r['success']) for r in a.values()), '| calib err median cm %.2f deg %.2f' % tuple(np.median(e, 0)) if len(e) else '', '| within 1 cm', int((e[:, 0] <= 1.0).sum()) if len(e) else 0)
m = {r['task_id']: bool(r['success']) for r in R if r['arm'] == 'full_metric'}; h = {r['task_id']: bool(r['success']) for r in R if r['arm'] == 'full_hybrid'}
b = sorted(set(m) & set(h)); print('metric vs hybrid, paired:', len(b), 'tasks, metric only', sum(m[t] and not h[t] for t in b), 'hybrid only', sum(h[t] and not m[t] for t in b))

In [ ]:
# learned refine 1, smoke (~10 min): full_hybrid_refine2 (learned segmenter, table-plane depth, the pose refined EasyHeC-style from two starts on the LEARNED masks) on 2 set A tasks, own smoke folder. Paste the two episode lines and STOP on an error
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_refine2_smoke --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --arms full_hybrid_refine2 --categories camera --seg_ckpt /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/seg_net.pt --tasks 699,707 2>&1 | tee -a /content/logs/wrap_run.log

In [ ]:
# learned refine 2 (~2.5 h on A100, resumable): raw and full_hybrid_refine2 on all 30 set A camera tasks, same harness and seeds as full_hybrid in plus/wraps. Own folder plus/wraps_refine2
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_refine2 --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --arms raw,full_hybrid_refine2 --categories camera --seg_ckpt /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/seg_net.pt 2>&1 | tee -a /content/logs/wrap_run.log

In [ ]:
# learned refine 3: report (seconds; paste all of it). Success and calibration error per arm on the same 30 tasks, paired against raw and against full_hybrid
import json, os, numpy as np
from math import comb
D = '/content/drive/MyDrive/restoring_organs/plus'
import glob; rows = lambda p: [json.loads(l) for q in [p] + sorted(glob.glob(p[:-6] + '.*.jsonl')) if os.path.exists(q) for l in open(q) if l.strip()]
R = rows(f'{D}/wraps_refine2/wrap_openvla_oft.jsonl') + [r for r in rows(f'{D}/wraps/wrap_openvla_oft.jsonl') if r['arm'] in ('full_plane', 'full_hybrid')]
ids = {r['task_id'] for r in R if r['arm'] == 'full_hybrid_refine2'}
by = {}
for r in R:
    if r['task_id'] in ids and r.get('category') == 'camera' and r.get('rep', 0) == 0: by.setdefault(r['task_id'], {}).setdefault(r['arm'], r)
mc = lambda b, c: min(1.0, 2 * sum(comb(b + c, i) for i in range(min(b, c) + 1)) / 2 ** (b + c)) if b + c else 1.0
for arm in ('raw', 'full_hybrid', 'full_hybrid_refine2'):
    a = [by[t][arm] for t in by if arm in by[t]]
    e = np.array([r['cam_est_err_cm_deg'][0] for r in a if r.get('cam_est_err_cm_deg')])
    print(arm, 'n', len(a), 'success', sum(bool(r['success']) for r in a), ('| cal err median %.2f cm, within 1 cm %d, within 2 cm %d' % (np.median(e), (e <= 1).sum(), (e <= 2).sum())) if len(e) else '')
for ref in ('raw', 'full_hybrid'):
    t = [i for i in by if ref in by[i] and 'full_hybrid_refine2' in by[i]]
    b = sum(by[i]['full_hybrid_refine2']['success'] and not by[i][ref]['success'] for i in t); c = sum(by[i][ref]['success'] and not by[i]['full_hybrid_refine2']['success'] for i in t)
    print(f'full_hybrid_refine2 vs {ref}: n {len(t)}, wins {b}, losses {c}, exact McNemar p {mc(b, c):.4f}')

In [ ]:
# learned refine set B 1 (~1.5 h on A100, resumable): full_hybrid_refine2 on the 30 set B camera tasks (raw and full exist in plus/wraps, same seeds). Own folder plus/wraps_refine2_setB
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_refine2_setB --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --arms full_hybrid_refine2 --categories camera --per_cat_scale 0 --extra_from_suite 30 --seg_ckpt /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/seg_net.pt 2>&1 | tee -a /content/logs/wrap_run.log

In [ ]:
# learned refine set B 2: report (seconds; paste all of it)
import json, os, numpy as np
from math import comb
D = '/content/drive/MyDrive/restoring_organs/plus'
import glob; rows = lambda p: [json.loads(l) for q in [p] + sorted(glob.glob(p[:-6] + '.*.jsonl')) if os.path.exists(q) for l in open(q) if l.strip()]
new = {r['task_id']: r for r in rows(f'{D}/wraps_refine2_setB/wrap_openvla_oft.jsonl') if r['arm'] == 'full_hybrid_refine2'}
old = {}
for r in rows(f'{D}/wraps/wrap_openvla_oft.jsonl'):
    if r['task_id'] in new and r.get('rep', 0) == 0: old.setdefault(r['task_id'], {}).setdefault(r['arm'], bool(r['success']))
mc = lambda b, c: min(1.0, 2 * sum(comb(b + c, i) for i in range(min(b, c) + 1)) / 2 ** (b + c)) if b + c else 1.0
e = np.array([r['cam_est_err_cm_deg'][0] for r in new.values() if r.get('cam_est_err_cm_deg')])
print('refine2 n', len(new), 'success', sum(bool(r['success']) for r in new.values()), '| cal err median %.2f cm, within 2 cm %d' % (np.median(e), (e <= 2).sum()))
for ref in ('raw', 'full'):
    t = [i for i in new if ref in old.get(i, {})]
    b = sum(new[i]['success'] and not old[i][ref] for i in t); c = sum(old[i][ref] and not new[i]['success'] for i in t)
    print(f'refine2 vs {ref}: n {len(t)} {ref} {sum(old[i][ref] for i in t)} -> refine2 {sum(bool(new[i]["success"]) for i in t)} | wins {b} losses {c} | exact McNemar p {mc(b, c):.4f}')

In [ ]:
# pose source 1, smoke (~10 min): full_refine and full_fiducial on 2 set B tasks into their own smoke folder. Each episode line must print; paste them and STOP on an error
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_posesource_smoke --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --arms full_refine,full_fiducial --categories camera --per_cat_scale 0 --extra_from_suite 30 --tasks 827,855 2>&1 | tee -a /content/logs/wrap_run.log

In [ ]:
# pose source 2 (~3.5 h on A100, resumable): the same reprojection driven by (ii) EasyHeC-style dense refinement from the silhouette pose and (iii) a fiducial (oracle-corner) PnP pose, on all 30 set B camera tasks, same harness and seeds as the raw / full rows in plus/wraps. Own folder plus/wraps_posesource
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/wrap_run.py --family oft --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out_dir /content/drive/MyDrive/restoring_organs/plus/wraps_posesource --nominals /content/drive/MyDrive/restoring_organs/plus/wrapper_oft/nominals_v3.json --arms full_refine,full_fiducial --categories camera --per_cat_scale 0 --extra_from_suite 30 2>&1 | tee -a /content/logs/wrap_run.log

In [ ]:
# pose source 3: report (seconds; paste all of it). Per pose source on the same set B tasks: success, calibration error, paired against the silhouette pose (full); also writes the compact rows file posesource_rows.jsonl
import json, os, numpy as np
from math import comb
D = '/content/drive/MyDrive/restoring_organs/plus'
import glob; rows = lambda p: [json.loads(l) for q in [p] + sorted(glob.glob(p[:-6] + '.*.jsonl')) if os.path.exists(q) for l in open(q) if l.strip()]
new = rows(f'{D}/wraps_posesource/wrap_openvla_oft.jsonl')
ids = {r['task_id'] for r in new if r['category'] == 'camera'}
old = [r for r in rows(f'{D}/wraps/wrap_openvla_oft.jsonl') if r['task_id'] in ids and r['category'] == 'camera' and r.get('rep', 0) == 0 and r['arm'] in ('raw', 'full')]
lad = [r for r in rows(f'{D}/wraps_ladder_libero_10/wrap_openvla_oft.jsonl') if r['task_id'] in ids and r['arm'] == 'full_truepose']
A = {}
for r in old + lad + new:
    A.setdefault(r['arm'], {})[r['task_id']] = r
with open(f'{D}/wraps_posesource/posesource_rows.jsonl', 'w') as f:
    for arm, a in A.items():
        for t, r in sorted(a.items()):
            f.write(json.dumps(dict(task_id=t, arm=arm, cam_est_err_cm_deg=r.get('cam_est_err_cm_deg'), success=bool(r['success']), pose_applied=r.get('cam_pose_applied', r.get('cam_applied')), fid_markers_visible=r.get('fid_markers_visible'))) + chr(10))
mc = lambda b, c: 1.0 if b + c == 0 else min(1.0, 2 * sum(comb(b + c, k) for k in range(min(b, c) + 1)) / 2 ** (b + c))
raw = {t: bool(r['success']) for t, r in A.get('raw', {}).items()}
full = {t: bool(r['success']) for t, r in A.get('full', {}).items()}
for arm in ('raw', 'full', 'full_refine', 'full_fiducial', 'full_truepose'):
    a = A.get(arm, {})
    if not a:
        continue
    e = np.array([r['cam_est_err_cm_deg'][:2] for r in a.values() if r.get('cam_est_err_cm_deg')], float).reshape(-1, 2)
    s = {t: bool(r['success']) for t, r in a.items()}
    rf = [t for t in s if t in raw and not raw[t]]
    b = sorted(set(s) & set(full))
    print(f'{arm:<14} n {len(a):2d} success {sum(s.values()):2d} | repaired {sum(s[t] for t in rf)} of {len(rf)} raw failures'
          + (' | calib err median %.2f cm %.2f deg, within 2 cm %d of %d' % (*np.median(e, 0), int((e[:, 0] <= 2).sum()), len(e)) if len(e) else '')
          + (f' | vs full: only this {sum(s[t] and not full[t] for t in b)}, only full {sum(full[t] and not s[t] for t in b)}, p {mc(sum(s[t] and not full[t] for t in b), sum(full[t] and not s[t] for t in b)):.3g}' if arm not in ('raw', 'full') else ''))
fv = [r.get('fid_markers_visible') for r in A.get('full_fiducial', {}).values()]
print('fiducial: markers visible per task', sorted(fv))